In [1]:
!pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask
import matplotlib.pyplot as plt
import time
import gc

# ===================================================================
# CẤU HÌNH (CONFIG)
# ===================================================================
DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')

# Output paths
CHECKPOINT_PATH = "/kaggle/working/gp4_deeplab_best.pth"
OUTPUT_DIR = "/kaggle/working/training_logs_charts/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
LEARNING_RATE = 5e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
NUM_EPOCHS = 320
EARLY_STOPPING_PATIENCE = 15 # Dừng nếu không cải thiện sau 15 epoch
NUM_WORKERS = 2
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 512
PIN_MEMORY = True

ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
NUM_CLASSES = len(ALLOWED_CLASSES) + 1
CLASS_NAMES = ['background'] + sorted(list(ALLOWED_CLASSES))

print(f"--- CONFIGURATION ---\nDevice: {DEVICE} | Model: DeepLabV3+ | Patience: {EARLY_STOPPING_PATIENCE}")

# ===================================================================
# DATASET & MODEL
# ===================================================================
class CocoDataset(Dataset):
    def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
        self.img_dir = img_dir
        self.transform = transform
        self.coco = COCO(annotation_file)
        self.img_ids = list(sorted(self.coco.imgs.keys()))
        
        all_cats = self.coco.loadCats(self.coco.getCatIds())
        if allowed_classes:
            cats_filtered = [c for c in all_cats if c['name'] in allowed_classes]
        else:
            cats_filtered = all_cats
        cats_filtered.sort(key=lambda x: x['name'])
        
        self.cat_id_to_class = {cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)}
        print(f"Dataset loaded: {len(self.img_ids)} images.")

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        
        image = np.array(Image.open(img_path).convert("RGB"))
        h, w = image.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        for ann in anns:
            class_idx = self.cat_id_to_class.get(ann['category_id'], 0)
            if class_idx > 0:
                if 'segmentation' in ann:
                    if isinstance(ann['segmentation'], list):
                        for seg in ann['segmentation']:
                            poly = np.array(seg).reshape(-1, 2).astype(np.int32)
                            cv2.fillPoly(mask, [poly], class_idx)
                    else:
                        rle = coco_mask.decode(ann['segmentation'])
                        mask[rle > 0] = class_idx

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        return image, mask.long()

def create_deeplabv3_model(num_classes):
    model = deeplabv3_resnet50(pretrained=True)
    model.classifier = DeepLabHead(2048, num_classes)
    return model

# ===================================================================
# LOSS FUNCTIONS (Dice + Focal + CE)
# ===================================================================
class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
    
    def dice_loss(self, pred, target, smooth=1.0):
        pred = F.softmax(pred, dim=1)
        target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 3, 1, 2).float()
        intersection = (pred[:, 1:] * target_one_hot[:, 1:]).sum(dim=(2, 3))
        union = pred[:, 1:].sum(dim=(2, 3)) + target_one_hot[:, 1:].sum(dim=(2, 3))
        return 1.0 - ((2.0 * intersection + smooth) / (union + smooth)).mean()

    def focal_loss(self, pred, target, alpha=0.25, gamma=2.0):
        ce_loss = F.cross_entropy(pred, target, reduction='none')
        pt = torch.exp(-ce_loss)
        return (alpha * (1 - pt) ** gamma * ce_loss).mean()

    def forward(self, pred, target):
        return 0.3 * self.ce(pred, target) + 0.5 * self.dice_loss(pred, target) + 0.2 * self.focal_loss(pred, target)

# ===================================================================
# CLASS EARLY STOPPING (MỚI THÊM)
# ===================================================================
class EarlyStopping:
    def __init__(self, patience=7, delta=0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss + self.delta:
            self.counter += 1
            print(f"   ⚠️ EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# ===================================================================
# TRAINING HELPER & PLOTTING
# ===================================================================
def calculate_iou(pred, target, num_classes):
    pred = pred.view(-1)
    target = target.view(-1)
    ious = []
    for cls_id in range(num_classes):
        pred_cls = (pred == cls_id)
        target_cls = (target == cls_id)
        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()
        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append((intersection / union).item())
    return ious

def plot_training_results(history, output_dir):
    epochs = range(1, len(history['train_loss']) + 1)

    # 1. Biểu đồ Loss
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'chart_loss.png'))
    plt.close()

    # 2. Biểu đồ mIoU
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history['val_miou'], 'g-', label='Validation mIoU')
    plt.title('Validation Mean IoU over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('mIoU')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'chart_miou.png'))
    plt.close()
    
    print(f"✅ Đã lưu biểu đồ tại: {output_dir}")

# ===================================================================
# MAIN TRAINING LOOP
# ===================================================================
def main():
    # 1. Setup Data
    train_transform = A.Compose([
        A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.Rotate(limit=15, p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    val_transform = A.Compose([
        A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

    train_ds = CocoDataset(TRAIN_IMG_DIR, TRAIN_JSON, train_transform, ALLOWED_CLASSES)
    val_ds = CocoDataset(VALID_IMG_DIR, VALID_JSON, val_transform, ALLOWED_CLASSES)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # 2. Setup Model & Training
    model = create_deeplabv3_model(NUM_CLASSES).to(DEVICE)
    loss_fn = CombinedLoss().to(DEVICE)
    optimizer = optim.Adam([
        {'params': model.backbone.parameters(), 'lr': LEARNING_RATE * 0.1},
        {'params': model.classifier.parameters(), 'lr': LEARNING_RATE}
    ], lr=LEARNING_RATE)
    
    scaler = torch.cuda.amp.GradScaler()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=5)
    
    # Kích hoạt Early Stopping
    early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, delta=0.0001)

    # History tracking
    history = {'train_loss': [], 'val_loss': [], 'val_miou': []}
    best_miou = 0.0

    print(f"\n🚀 START TRAINING FOR {NUM_EPOCHS} EPOCHS...")
    
    for epoch in range(NUM_EPOCHS):
        start_time = time.time()
        
        # --- TRAIN ---
        model.train()
        train_loss = 0
        for images, masks in tqdm(train_loader, desc=f"Ep {epoch+1}/{NUM_EPOCHS} [Train]", leave=False):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            
            with torch.cuda.amp.autocast():
                output = model(images)['out']
                loss = loss_fn(output, masks)
            
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)

        # --- VALIDATE ---
        model.eval()
        val_loss = 0
        total_ious = []
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Ep {epoch+1}/{NUM_EPOCHS} [Valid]", leave=False):
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                output = model(images)['out']
                val_loss += loss_fn(output, masks).item()
                
                preds = torch.argmax(output, dim=1)
                total_ious.append(np.nanmean(calculate_iou(preds, masks, NUM_CLASSES)))
        
        avg_val_loss = val_loss / len(val_loader)
        avg_miou = np.nanmean(total_ious)
        
        # --- LOGGING ---
        duration = time.time() - start_time
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_miou'].append(avg_miou)
        
        print(f"Epoch {epoch+1:03d} | Time: {duration:.1f}s | "
              f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | mIoU: {avg_miou:.4f}")
        
        scheduler.step(avg_val_loss)

        # Save Best Model
        if avg_miou > best_miou:
            best_miou = avg_miou
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            print(f"   >>> New Best mIoU! Saved model.")
        
        # Check Early Stopping
        early_stopping(avg_val_loss)
        if early_stopping.early_stop:
            print(f"\n⏹️ Early Stopping triggered at Epoch {epoch+1}. Validation loss không giảm trong {EARLY_STOPPING_PATIENCE} epochs.")
            break

    # 3. Finish
    print("\n✅ Training Completed.")
    print(f"🏆 Best Validation mIoU: {best_miou:.4f}")
    
    # 4. Generate Charts for Report
    plot_training_results(history, OUTPUT_DIR)

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"Error: {e}")

--- CONFIGURATION ---
Device: cuda | Model: DeepLabV3+ | Patience: 15
loading annotations into memory...
Done (t=0.19s)
creating index...
index created!
Dataset loaded: 1372 images.
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Dataset loaded: 294 images.


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth
100%|██████████| 161M/161M [00:00<00:00, 226MB/s]
/tmp/ipykernel_21/895283471.py:234: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScale


🚀 START TRAINING FOR 320 EPOCHS...


Ep 1/320 [Train]:   0%|          | 0/171 [00:00<?, ?it/s]/tmp/ipykernel_21/895283471.py:255: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 001 | Time: 65.6s | Train Loss: 0.6123 | Val Loss: 0.4598 | mIoU: 0.5393
   >>> New Best mIoU! Saved model.


...

Epoch 057 | Time: 74.8s | Train Loss: 0.0817 | Val Loss: 0.1459 | mIoU: 0.7747
   ⚠️ EarlyStopping counter: 11 out of 15


Epoch 058 | Time: 73.9s | Train Loss: 0.0806 | Val Loss: 0.1457 | mIoU: 0.7757
   >>> New Best mIoU! Saved model.
   ⚠️ EarlyStopping counter: 12 out of 15


Epoch 059 | Time: 73.7s | Train Loss: 0.0813 | Val Loss: 0.1464 | mIoU: 0.7742
   ⚠️ EarlyStopping counter: 13 out of 15


Epoch 060 | Time: 73.2s | Train Loss: 0.0808 | Val Loss: 0.1458 | mIoU: 0.7746
   ⚠️ EarlyStopping counter: 14 out of 15


Epoch 061 | Time: 72.9s | Train Loss: 0.0809 | Val Loss: 0.1456 | mIoU: 0.7749
   ⚠️ EarlyStopping counter: 15 out of 15

⏹️ Early Stopping triggered at Epoch 61. Validation loss không giảm trong 15 epochs.

✅ Training Completed.
🏆 Best Validation mIoU: 0.7757
✅ Đã lưu biểu đồ tại: /kaggle/working/training_logs_charts/


In [3]:
# # ===================================================================
# # GIẢI PHÁP 10: EFFICIENTNET-B1 + 320x640 + LOVASZ LOSS (MAX MIOU)
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import segmentation_models_pytorch as smp
# import os
# import cv2
# import numpy as np
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import warnings

# warnings.filterwarnings("ignore")

# # ===================================================================
# # 1. CẤU HÌNH (NÂNG CẤP ĐỘ PHÂN GIẢI)
# # ===================================================================
# print("--- CẤU HÌNH: EFFICIENTNET-B1 HIGH RES ---")

# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# CHECKPOINT_PATH = "/kaggle/working/best_effb1_highres.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/predictions_gp10/"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 8           # Giảm Batch Size xuống vì tăng độ phân giải ảnh
# NUM_EPOCHS = 150
# PATIENCE = 20            # Kiên nhẫn hơn chút
# LEARNING_RATE = 1e-4     
# IMAGE_HEIGHT = 320       # <--- TĂNG ĐỘ PHÂN GIẢI (Key cho mIoU cao)
# IMAGE_WIDTH = 640        # <--- TĂNG ĐỘ PHÂN GIẢI
# NUM_WORKERS = 2

# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  
# COLOR_MAP = {0: [0, 0, 0], 1: [255, 0, 255], 2: [0, 0, 255], 3: [0, 255, 0]}

# # ===================================================================
# # 2. DATASET
# # ===================================================================
# class CocoDataset(Dataset):
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         self.img_dir = img_dir
#         self.transform = transform
#         self.coco = COCO(annotation_file)
#         self.img_ids = list(sorted(self.coco.imgs.keys()))
#         all_cats = self.coco.loadCats(self.coco.getCatIds())
#         cats_filtered = [c for c in all_cats if c['name'] in allowed_classes] if allowed_classes else all_cats
#         cats_filtered.sort(key=lambda x: x['name'])
#         self.cat_id_to_class = {cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)}

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
#             image = cv2.imread(img_path)
#             image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#             h, w = image.shape[:2]
#             mask = np.zeros((h, w), dtype=np.uint8)
#             anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
#             for ann in anns:
#                 class_idx = self.cat_id_to_class.get(ann['category_id'], 0)
#                 if class_idx > 0:
#                     if 'segmentation' in ann:
#                         if isinstance(ann['segmentation'], list):
#                             for seg in ann['segmentation']:
#                                 poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                                 cv2.fillPoly(mask, [poly], int(class_idx))
#                         elif isinstance(ann['segmentation'], dict):
#                             mask[coco_mask.decode(ann['segmentation']) > 0] = int(class_idx)
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image, mask = augmented['image'], augmented['mask']
#             return image, mask.long()
#         except:
#             return torch.zeros(3, IMAGE_HEIGHT, IMAGE_WIDTH), torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # ===================================================================
# # 3. LOVASZ LOSS (VŨ KHÍ BÍ MẬT CHO MIOU)
# # ===================================================================
# def create_model():
#     model = smp.DeepLabV3Plus(
#         encoder_name="efficientnet-b1", 
#         encoder_weights="imagenet",
#         in_channels=3,
#         classes=NUM_CLASSES,
#     )
#     return model

# class SotaLoss(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # Lovasz Loss: Tối ưu trực tiếp Jaccard Index (mIoU)
#         self.lovasz = smp.losses.LovaszLoss(mode='multiclass', from_logits=True)
#         # Focal Loss: Giải quyết mất cân bằng mẫu
#         self.focal = smp.losses.FocalLoss(mode='multiclass', gamma=2.0)
        
#     def forward(self, pred, target):
#         # Tỷ lệ 0.4 Focal + 0.6 Lovasz (Ưu tiên Lovasz để đẩy mIoU)
#         return 0.4 * self.focal(pred, target) + 0.6 * self.lovasz(pred, target)

# # ===================================================================
# # 4. TRAINING UTILS
# # ===================================================================
# class EarlyStopping:
#     def __init__(self, patience=15, delta=0.0001, path='checkpoint.pth'):
#         self.patience = patience
#         self.delta = delta
#         self.path = path
#         self.best_score = None
#         self.counter = 0
#         self.early_stop = False

#     def __call__(self, val_score, model):
#         if self.best_score is None:
#             self.best_score = val_score
#             torch.save(model.state_dict(), self.path)
#         elif val_score < self.best_score + self.delta:
#             self.counter += 1
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = val_score
#             torch.save(model.state_dict(), self.path)
#             self.counter = 0

# def calculate_iou(pred, target, num_classes):
#     pred = torch.argmax(pred, dim=1).view(-1)
#     target = target.view(-1)
#     ious = []
#     for cls_id in range(1, num_classes):
#         p, t = (pred == cls_id), (target == cls_id)
#         inter, union = (p & t).sum().float(), (p | t).sum().float()
#         if union == 0: ious.append(float('nan'))
#         else: ious.append((inter / union).item())
#     return np.nanmean(ious)

# def train_epoch(model, loader, optimizer, loss_fn, scaler):
#     model.train()
#     total_loss = 0
#     loop = tqdm(loader, desc="Train", leave=False)
#     for img, mask in loop:
#         img, mask = img.to(DEVICE), mask.to(DEVICE)
#         optimizer.zero_grad()
#         with torch.cuda.amp.autocast():
#             loss = loss_fn(model(img), mask)
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         total_loss += loss.item()
#         loop.set_postfix(loss=loss.item())
#     return total_loss / len(loader)

# def validate(model, loader, loss_fn):
#     model.eval()
#     total_loss, ious = 0, []
#     with torch.no_grad():
#         for img, mask in tqdm(loader, desc="Val", leave=False):
#             img, mask = img.to(DEVICE), mask.to(DEVICE)
#             out = model(img)
#             total_loss += loss_fn(out, mask).item()
#             ious.append(calculate_iou(out, mask, NUM_CLASSES))
#     return total_loss / len(loader), np.nanmean(ious)

# # ===================================================================
# # 5. MAIN (GIỮ AUGMENTATION NHƯNG GIẢM CƯỜNG ĐỘ 1 CHÚT)
# # ===================================================================
# def run_training():
#     print(f"🖼️ Training Resolution: {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
    
#     # Augmentation: Giữ Shadow/Fog để chống rỗ, nhưng giảm xác suất (p)
#     # để model dễ học hơn -> Tăng mIoU
#     train_transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.HorizontalFlip(p=0.5),
        
#         # Giảm p xuống 0.2 để không làm khó model quá mức
#         A.RandomShadow(num_shadows_lower=1, num_shadows_upper=2, shadow_dimension=4, p=0.2), 
#         A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.2, p=0.15),
#         A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        
#         A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=10, p=0.5),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     val_transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])

#     train_ds = CocoDataset(TRAIN_IMG_DIR, TRAIN_JSON, transform=train_transform, allowed_classes=ALLOWED_CLASSES)
#     val_ds = CocoDataset(VALID_IMG_DIR, VALID_JSON, transform=val_transform, allowed_classes=ALLOWED_CLASSES)
    
#     # Drop_last=True quan trọng cho BatchNorm khi batch size nhỏ
#     train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
#     val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

#     print("🏗️ Creating Model (Eff-B1 + Lovasz Loss)...")
#     model = create_model().to(DEVICE)
#     loss_fn = SotaLoss().to(DEVICE)
    
#     # Dùng AdamW với weight decay chuẩn
#     optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
#     scaler = torch.cuda.amp.GradScaler()
#     # OneCycleLR thường giúp đạt điểm cao hơn CosineAnnealing
#     scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS, pct_start=0.3)
    
#     early_stopping = EarlyStopping(patience=PATIENCE, path=CHECKPOINT_PATH)

#     print("\n🚀 START TRAINING")
#     for epoch in range(NUM_EPOCHS):
#         loss = train_epoch(model, train_loader, optimizer, loss_fn, scaler)
#         val_loss, val_iou = validate(model, val_loader, loss_fn)
#         scheduler.step() # Step mỗi batch với OneCycleLR (nhưng ở đây mình step cuối epoch cũng tạm ổn)
        
#         early_stopping(val_iou, model)
        
#         print(f"Ep {epoch+1:03d} | Loss: {loss:.4f} | Val Loss: {val_loss:.4f} | mIoU: {val_iou:.4f}")
        
#         if early_stopping.early_stop:
#             print(f"⏹️ Early Stopping at epoch {epoch+1}")
#             break
            
#     print(f"🏆 Best mIoU: {early_stopping.best_score:.4f}")

# # ===================================================================
# # 6. INFERENCE (CLEANING + RESIZE)
# # ===================================================================
# def post_process_mask(mask_np):
#     processed_mask = np.zeros_like(mask_np)
#     # Kernel nhỏ gọn hơn cho độ phân giải cao
#     kernel = np.ones((5,5), np.uint8) 

#     for cls in range(1, NUM_CLASSES):
#         binary_mask = (mask_np == cls).astype(np.uint8)
#         # Closing để lấp lỗ
#         binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
        
#         # Lọc vùng nhiễu (tăng ngưỡng lên vì ảnh to hơn)
#         num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
#         for i in range(1, num_labels):
#             if stats[i, cv2.CC_STAT_AREA] >= 150: 
#                 processed_mask[labels == i] = cls
                
#     return processed_mask

# def run_inference():
#     print("\n🔍 INFERENCE TEST (HIGH RES)")
#     if not os.path.exists(CHECKPOINT_PATH): return

#     model = create_model()
#     model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#     model.to(DEVICE).eval()
    
#     transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH), # 320x640
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2()
#     ])
    
#     colors = np.array(list(COLOR_MAP.values()), dtype=np.uint8)
#     files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('jpg', 'png'))][:20]
    
#     for f in tqdm(files):
#         path = os.path.join(TEST_REAL_DIR, f)
#         img_bgr = cv2.imread(path)
#         if img_bgr is None: continue
        
#         orig_size = img_bgr.shape[:2]
#         img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
#         tensor = transform(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
        
#         t0 = cv2.getTickCount()
#         with torch.no_grad():
#             output = model(tensor)
#             pred = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)
#         time_sec = (cv2.getTickCount() - t0) / cv2.getTickFrequency()
        
#         pred = cv2.resize(pred, (orig_size[1], orig_size[0]), interpolation=cv2.INTER_NEAREST)
#         pred_clean = post_process_mask(pred)
        
#         color_mask = np.zeros_like(img_bgr)
#         for cls_id in range(1, NUM_CLASSES):
#             color_mask[pred_clean == cls_id] = colors[cls_id]
            
#         overlay = cv2.addWeighted(img_bgr, 0.7, color_mask, 0.3, 0)
#         cv2.putText(overlay, f"EffB1-HD: {time_sec*1000:.1f}ms", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
#         cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"gp10_{f}"), overlay)
        
#     print(f"✅ Xong! Check folder: {PREDICTION_OUTPUT_DIR}")

# if __name__ == "__main__":
#     run_training()
#     run_inference()

In [4]:
# # ===================================================================
# # GIẢI PHÁP 11 (ALL-IN): U-NET++ EFFICIENTNET-B1 (HIGH RES & SHARP)
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import segmentation_models_pytorch as smp
# import os
# import cv2
# import numpy as np
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import warnings

# warnings.filterwarnings("ignore")

# # ===================================================================
# # 1. CẤU HÌNH (HIGH RES + U-NET++)
# # ===================================================================
# print("--- CẤU HÌNH: U-NET++ EFFICIENTNET-B1 (SHARP EDGES) ---")

# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# CHECKPOINT_PATH = "/kaggle/working/best_unetplusplus_b1.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/predictions_gp11/"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# # U-Net++ decoder hơi nặng hơn DeepLab một chút, giảm batch nếu cần
# BATCH_SIZE = 8           
# NUM_EPOCHS = 320         # Treo máy 1 tiếng thì cứ để nhiều epoch
# PATIENCE = 32            
# LEARNING_RATE = 1e-4     
# IMAGE_HEIGHT = 320       # Giữ độ phân giải cao để bắt nét
# IMAGE_WIDTH = 640
# NUM_WORKERS = 2

# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  
# COLOR_MAP = {0: [0, 0, 0], 1: [255, 0, 255], 2: [0, 0, 255], 3: [0, 255, 0]}

# # ===================================================================
# # 2. DATASET (GIỮ NGUYÊN)
# # ===================================================================
# class CocoDataset(Dataset):
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         self.img_dir = img_dir
#         self.transform = transform
#         self.coco = COCO(annotation_file)
#         self.img_ids = list(sorted(self.coco.imgs.keys()))
#         all_cats = self.coco.loadCats(self.coco.getCatIds())
#         cats_filtered = [c for c in all_cats if c['name'] in allowed_classes] if allowed_classes else all_cats
#         cats_filtered.sort(key=lambda x: x['name'])
#         self.cat_id_to_class = {cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)}

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
#             image = cv2.imread(img_path)
#             image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#             h, w = image.shape[:2]
#             mask = np.zeros((h, w), dtype=np.uint8)
#             anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
#             for ann in anns:
#                 class_idx = self.cat_id_to_class.get(ann['category_id'], 0)
#                 if class_idx > 0:
#                     if 'segmentation' in ann:
#                         if isinstance(ann['segmentation'], list):
#                             for seg in ann['segmentation']:
#                                 poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                                 cv2.fillPoly(mask, [poly], int(class_idx))
#                         elif isinstance(ann['segmentation'], dict):
#                             mask[coco_mask.decode(ann['segmentation']) > 0] = int(class_idx)
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image, mask = augmented['image'], augmented['mask']
#             return image, mask.long()
#         except:
#             return torch.zeros(3, IMAGE_HEIGHT, IMAGE_WIDTH), torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # ===================================================================
# # 3. MODEL: U-NET++ (THAY ĐỔI LỚN)
# # ===================================================================
# def create_model():
#     # UnetPlusPlus: "Vua" của segmentation đường nét mảnh
#     model = smp.UnetPlusPlus(
#         encoder_name="efficientnet-b1", 
#         encoder_weights="imagenet",
#         in_channels=3,
#         classes=NUM_CLASSES,
#         activation=None
#     )
#     return model

# # Combo Loss: Dice (Hình dạng) + Focal (Điểm khó) -> Ổn định nhất cho U-Net
# class DiceFocalLoss(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.dice = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
#         self.focal = smp.losses.FocalLoss(mode='multiclass', gamma=2.0)
        
#     def forward(self, pred, target):
#         return 0.5 * self.dice(pred, target) + 0.5 * self.focal(pred, target)

# # ===================================================================
# # 4. UTILS
# # ===================================================================
# class EarlyStopping:
#     def __init__(self, patience=15, delta=0.0001, path='checkpoint.pth'):
#         self.patience = patience
#         self.delta = delta
#         self.path = path
#         self.best_score = None
#         self.counter = 0
#         self.early_stop = False

#     def __call__(self, val_score, model):
#         if self.best_score is None:
#             self.best_score = val_score
#             torch.save(model.state_dict(), self.path)
#         elif val_score < self.best_score + self.delta:
#             self.counter += 1
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = val_score
#             torch.save(model.state_dict(), self.path)
#             self.counter = 0

# def calculate_iou(pred, target, num_classes):
#     pred = torch.argmax(pred, dim=1).view(-1)
#     target = target.view(-1)
#     ious = []
#     for cls_id in range(1, num_classes):
#         p, t = (pred == cls_id), (target == cls_id)
#         inter, union = (p & t).sum().float(), (p | t).sum().float()
#         if union == 0: ious.append(float('nan'))
#         else: ious.append((inter / union).item())
#     return np.nanmean(ious)

# def train_epoch(model, loader, optimizer, loss_fn, scaler):
#     model.train()
#     total_loss = 0
#     loop = tqdm(loader, desc="Train", leave=False)
#     for img, mask in loop:
#         img, mask = img.to(DEVICE), mask.to(DEVICE)
#         optimizer.zero_grad()
#         with torch.cuda.amp.autocast():
#             loss = loss_fn(model(img), mask)
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         total_loss += loss.item()
#         loop.set_postfix(loss=loss.item())
#     return total_loss / len(loader)

# def validate(model, loader, loss_fn):
#     model.eval()
#     total_loss, ious = 0, []
#     with torch.no_grad():
#         for img, mask in tqdm(loader, desc="Val", leave=False):
#             img, mask = img.to(DEVICE), mask.to(DEVICE)
#             out = model(img)
#             total_loss += loss_fn(out, mask).item()
#             ious.append(calculate_iou(out, mask, NUM_CLASSES))
#     return total_loss / len(loader), np.nanmean(ious)

# # ===================================================================
# # 5. MAIN TRAINING
# # ===================================================================
# def run_training():
#     print(f"🖼️ Resolution: {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
    
#     # Augmentation: Giữ nguyên bộ trị "bóng râm" từ GP9
#     train_transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.HorizontalFlip(p=0.5),
        
#         # Shadow & Fog Augmentation (Fix lỗi rỗ)
#         A.RandomShadow(num_shadows_lower=1, num_shadows_upper=2, shadow_dimension=4, p=0.2), 
#         A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.2, p=0.15),
        
#         A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
#         A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=10, p=0.5),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     val_transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])

#     train_ds = CocoDataset(TRAIN_IMG_DIR, TRAIN_JSON, transform=train_transform, allowed_classes=ALLOWED_CLASSES)
#     val_ds = CocoDataset(VALID_IMG_DIR, VALID_JSON, transform=val_transform, allowed_classes=ALLOWED_CLASSES)
    
#     train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
#     val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

#     print("🏗️ Creating Model: U-NET++ EfficientNet-B1...")
#     model = create_model().to(DEVICE)
#     loss_fn = DiceFocalLoss().to(DEVICE)
#     optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
#     scaler = torch.cuda.amp.GradScaler()
    
#     # OneCycleLR: Học cực nhanh trong thời gian ngắn (phù hợp train 1 tiếng)
#     scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS, pct_start=0.3)
#     early_stopping = EarlyStopping(patience=PATIENCE, path=CHECKPOINT_PATH)

#     print("\n🚀 START TRAINING (AFK MODE)")
#     for epoch in range(NUM_EPOCHS):
#         loss = train_epoch(model, train_loader, optimizer, loss_fn, scaler)
#         val_loss, val_iou = validate(model, val_loader, loss_fn)
#         scheduler.step()
        
#         early_stopping(val_iou, model)
        
#         print(f"Ep {epoch+1:03d} | Loss: {loss:.4f} | Val Loss: {val_loss:.4f} | mIoU: {val_iou:.4f}")
        
#         if early_stopping.early_stop:
#             print(f"⏹️ Early Stopping at epoch {epoch+1}")
#             break
            
#     print(f"🏆 Best mIoU: {early_stopping.best_score:.4f}")

# # ===================================================================
# # 6. INFERENCE (U-NET++ CLEANING)
# # ===================================================================
# def post_process_mask(mask_np):
#     processed_mask = np.zeros_like(mask_np)
#     kernel = np.ones((5,5), np.uint8) 

#     for cls in range(1, NUM_CLASSES):
#         binary_mask = (mask_np == cls).astype(np.uint8)
#         # U-Net++ đã rất nét, chỉ cần closing nhẹ để liền mạch
#         binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
        
#         num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
#         for i in range(1, num_labels):
#             if stats[i, cv2.CC_STAT_AREA] >= 150: 
#                 processed_mask[labels == i] = cls
                
#     return processed_mask

# def run_inference():
#     print("\n🔍 INFERENCE TEST (U-NET++)")
#     if not os.path.exists(CHECKPOINT_PATH): return

#     model = create_model()
#     model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#     model.to(DEVICE).eval()
    
#     transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2()
#     ])
    
#     colors = np.array(list(COLOR_MAP.values()), dtype=np.uint8)
#     files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('jpg', 'png'))][:20]
    
#     for f in tqdm(files):
#         path = os.path.join(TEST_REAL_DIR, f)
#         img_bgr = cv2.imread(path)
#         if img_bgr is None: continue
        
#         orig_size = img_bgr.shape[:2]
#         img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
#         tensor = transform(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
        
#         t0 = cv2.getTickCount()
#         with torch.no_grad():
#             output = model(tensor)
#             pred = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)
#         time_sec = (cv2.getTickCount() - t0) / cv2.getTickFrequency()
        
#         pred = cv2.resize(pred, (orig_size[1], orig_size[0]), interpolation=cv2.INTER_NEAREST)
#         pred_clean = post_process_mask(pred)
        
#         color_mask = np.zeros_like(img_bgr)
#         for cls_id in range(1, NUM_CLASSES):
#             color_mask[pred_clean == cls_id] = colors[cls_id]
            
#         overlay = cv2.addWeighted(img_bgr, 0.7, color_mask, 0.3, 0)
#         cv2.putText(overlay, f"UNet++: {time_sec*1000:.1f}ms", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
#         cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"gp11_{f}"), overlay)
        
#     print(f"✅ Đã xong! Check folder: {PREDICTION_OUTPUT_DIR}")

# if __name__ == "__main__":
#     run_training()
#     run_inference()

In [5]:
# # ===================================================================
# # GIẢI PHÁP 12: FPN + EFFICIENTNET-B3 + CLAHE (SẮC NÉT & TƯƠNG PHẢN)
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import segmentation_models_pytorch as smp
# import os
# import cv2
# import numpy as np
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import warnings

# warnings.filterwarnings("ignore")

# # ===================================================================
# # 1. CẤU HÌNH (CÂN BẰNG GIỮA SPEED VÀ ACC)
# # ===================================================================
# print("--- CẤU HÌNH: FPN + EFFICIENTNET-B3 + CLAHE ---")

# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# CHECKPOINT_PATH = "/kaggle/working/best_fpn_b3_clahe.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/predictions_gp12/"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# # B3 nặng hơn, ta dùng batch 8 để an toàn
# BATCH_SIZE = 8           
# NUM_EPOCHS = 320
# PATIENCE = 32            
# LEARNING_RATE = 1e-4     
# # Độ phân giải tầm trung (lớn hơn 256 nhưng nhỏ hơn 320) để B3 chạy mượt
# IMAGE_HEIGHT = 288       
# IMAGE_WIDTH = 576        
# NUM_WORKERS = 2

# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  
# COLOR_MAP = {0: [0, 0, 0], 1: [255, 0, 255], 2: [0, 0, 255], 3: [0, 255, 0]}

# # ===================================================================
# # 2. DATASET
# # ===================================================================
# class CocoDataset(Dataset):
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         self.img_dir = img_dir
#         self.transform = transform
#         self.coco = COCO(annotation_file)
#         self.img_ids = list(sorted(self.coco.imgs.keys()))
#         all_cats = self.coco.loadCats(self.coco.getCatIds())
#         cats_filtered = [c for c in all_cats if c['name'] in allowed_classes] if allowed_classes else all_cats
#         cats_filtered.sort(key=lambda x: x['name'])
#         self.cat_id_to_class = {cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)}

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
#             image = cv2.imread(img_path)
#             image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#             h, w = image.shape[:2]
#             mask = np.zeros((h, w), dtype=np.uint8)
#             anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
#             for ann in anns:
#                 class_idx = self.cat_id_to_class.get(ann['category_id'], 0)
#                 if class_idx > 0:
#                     if 'segmentation' in ann:
#                         if isinstance(ann['segmentation'], list):
#                             for seg in ann['segmentation']:
#                                 poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                                 cv2.fillPoly(mask, [poly], int(class_idx))
#                         elif isinstance(ann['segmentation'], dict):
#                             mask[coco_mask.decode(ann['segmentation']) > 0] = int(class_idx)
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image, mask = augmented['image'], augmented['mask']
#             return image, mask.long()
#         except:
#             return torch.zeros(3, IMAGE_HEIGHT, IMAGE_WIDTH), torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # ===================================================================
# # 3. MODEL: FPN (THAY ĐỔI KIẾN TRÚC)
# # ===================================================================
# def create_model():
#     # FPN: Feature Pyramid Network -> Giỏi bắt chi tiết đa tỉ lệ
#     # Backbone: EfficientNet-B3 -> Rất mạnh mẽ
#     model = smp.FPN(
#         encoder_name="efficientnet-b3", 
#         encoder_weights="imagenet",
#         in_channels=3,
#         classes=NUM_CLASSES,
#     )
#     return model

# class ComboLoss(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # Dice Loss + Focal Loss: Combo an toàn nhất
#         self.dice = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
#         self.focal = smp.losses.FocalLoss(mode='multiclass', gamma=2.0)
        
#     def forward(self, pred, target):
#         return 0.5 * self.dice(pred, target) + 0.5 * self.focal(pred, target)

# # ===================================================================
# # 4. UTILS
# # ===================================================================
# class EarlyStopping:
#     def __init__(self, patience=15, delta=0.0001, path='checkpoint.pth'):
#         self.patience = patience
#         self.delta = delta
#         self.path = path
#         self.best_score = None
#         self.counter = 0
#         self.early_stop = False

#     def __call__(self, val_score, model):
#         if self.best_score is None:
#             self.best_score = val_score
#             torch.save(model.state_dict(), self.path)
#         elif val_score < self.best_score + self.delta:
#             self.counter += 1
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = val_score
#             torch.save(model.state_dict(), self.path)
#             self.counter = 0

# def calculate_iou(pred, target, num_classes):
#     pred = torch.argmax(pred, dim=1).view(-1)
#     target = target.view(-1)
#     ious = []
#     for cls_id in range(1, num_classes):
#         p, t = (pred == cls_id), (target == cls_id)
#         inter, union = (p & t).sum().float(), (p | t).sum().float()
#         if union == 0: ious.append(float('nan'))
#         else: ious.append((inter / union).item())
#     return np.nanmean(ious)

# def train_epoch(model, loader, optimizer, loss_fn, scaler):
#     model.train()
#     total_loss = 0
#     loop = tqdm(loader, desc="Train", leave=False)
#     for img, mask in loop:
#         img, mask = img.to(DEVICE), mask.to(DEVICE)
#         optimizer.zero_grad()
#         with torch.cuda.amp.autocast():
#             loss = loss_fn(model(img), mask)
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         total_loss += loss.item()
#         loop.set_postfix(loss=loss.item())
#     return total_loss / len(loader)

# def validate(model, loader, loss_fn):
#     model.eval()
#     total_loss, ious = 0, []
#     with torch.no_grad():
#         for img, mask in tqdm(loader, desc="Val", leave=False):
#             img, mask = img.to(DEVICE), mask.to(DEVICE)
#             out = model(img)
#             total_loss += loss_fn(out, mask).item()
#             ious.append(calculate_iou(out, mask, NUM_CLASSES))
#     return total_loss / len(loader), np.nanmean(ious)

# # ===================================================================
# # 5. MAIN TRAINING (CLAHE - CHÌA KHÓA VÀNG)
# # ===================================================================
# def run_training():
#     print(f"🖼️ Resolution: {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
    
#     # --- AUGMENTATION VỚI CLAHE ---
#     # CLAHE giúp cân bằng sáng, làm rõ chi tiết trong bóng râm
#     # Nó hoạt động giống như việc bạn đeo kính râm phân cực vậy.
#     train_transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0), # <--- LUÔN LUÔN BẬT
#         A.HorizontalFlip(p=0.5),
#         A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=10, p=0.5),
#         A.RandomBrightnessContrast(p=0.2),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     # Validation cũng dùng CLAHE để model nhìn thấy những gì nó đã học
#     val_transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0), # <--- QUAN TRỌNG
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])

#     train_ds = CocoDataset(TRAIN_IMG_DIR, TRAIN_JSON, transform=train_transform, allowed_classes=ALLOWED_CLASSES)
#     val_ds = CocoDataset(VALID_IMG_DIR, VALID_JSON, transform=val_transform, allowed_classes=ALLOWED_CLASSES)
    
#     train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
#     val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

#     print("🏗️ Creating Model: FPN + EfficientNet-B3...")
#     model = create_model().to(DEVICE)
#     loss_fn = ComboLoss().to(DEVICE)
#     optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
#     scaler = torch.cuda.amp.GradScaler()
#     scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
#     early_stopping = EarlyStopping(patience=PATIENCE, path=CHECKPOINT_PATH)

#     print("\n🚀 START TRAINING (GP12)")
#     for epoch in range(NUM_EPOCHS):
#         loss = train_epoch(model, train_loader, optimizer, loss_fn, scaler)
#         val_loss, val_iou = validate(model, val_loader, loss_fn)
        
#         # Scheduler dựa trên mIoU để giảm LR khi bão hòa
#         scheduler.step(val_iou)
#         early_stopping(val_iou, model)
        
#         print(f"Ep {epoch+1:03d} | Loss: {loss:.4f} | Val Loss: {val_loss:.4f} | mIoU: {val_iou:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        
#         if early_stopping.early_stop:
#             print(f"⏹️ Early Stopping at epoch {epoch+1}")
#             break
            
#     print(f"🏆 Best mIoU: {early_stopping.best_score:.4f}")

# # ===================================================================
# # 6. INFERENCE (WITH CLAHE)
# # ===================================================================
# def run_inference():
#     print("\n🔍 INFERENCE TEST (FPN B3 + CLAHE)")
#     if not os.path.exists(CHECKPOINT_PATH): return

#     model = create_model()
#     model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#     model.to(DEVICE).eval()
    
#     # Phải có CLAHE trong Inference để khớp với lúc Train
#     transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0), 
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2()
#     ])
    
#     colors = np.array(list(COLOR_MAP.values()), dtype=np.uint8)
#     files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('jpg', 'png'))][:20]
    
#     for f in tqdm(files):
#         path = os.path.join(TEST_REAL_DIR, f)
#         img_bgr = cv2.imread(path)
#         if img_bgr is None: continue
        
#         orig_size = img_bgr.shape[:2]
#         img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
#         tensor = transform(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
        
#         t0 = cv2.getTickCount()
#         with torch.no_grad():
#             output = model(tensor)
#             pred = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)
#         time_sec = (cv2.getTickCount() - t0) / cv2.getTickFrequency()
        
#         pred = cv2.resize(pred, (orig_size[1], orig_size[0]), interpolation=cv2.INTER_NEAREST)
        
#         # Cleaning nhẹ
#         kernel = np.ones((5,5), np.uint8)
#         for cls in range(1, NUM_CLASSES):
#             binary_mask = (pred == cls).astype(np.uint8)
#             binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
#             pred[binary_mask == 0] = 0 # Xóa nhiễu nếu cần, hoặc để nguyên
            
#         color_mask = np.zeros_like(img_bgr)
#         for cls_id in range(1, NUM_CLASSES):
#             color_mask[pred == cls_id] = colors[cls_id]
            
#         overlay = cv2.addWeighted(img_bgr, 0.7, color_mask, 0.3, 0)
#         cv2.putText(overlay, f"FPN-B3: {time_sec*1000:.1f}ms", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
#         cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"gp12_{f}"), overlay)
        
#     print(f"✅ Đã xong! Check folder: {PREDICTION_OUTPUT_DIR}")

# if __name__ == "__main__":
#     run_training()
#     run_inference()

In [6]:
# # ===================================================================
# # GIẢI PHÁP 4 (LITE): DEEPLABV3+ (MobileNetV3) + EARLY STOPPING
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# import torchvision
# from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large
# from torchvision.models.segmentation.deeplabv3 import DeepLabHead
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import cv2
# import numpy as np
# import json
# from PIL import Image
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import gc
# import time
# import traceback

# # ===================================================================
# # PHẦN 1: CẤU HÌNH
# # ===================================================================
# print("--- PHẦN 1: Cấu hình ---")
# # --- Đường dẫn ---
# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# # --- Checkpoint & Outputs ---
# CHECKPOINT_PATH = "/kaggle/working/gp4_mobilenet_best.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/gp4_predictions/"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# # --- Hyperparameters ---
# LEARNING_RATE = 1e-4    
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 16         
# NUM_EPOCHS = 500        # (ĐÃ TĂNG) Tăng lên 200, EarlyStopping sẽ lo việc dừng
# NUM_WORKERS = 2         
# IMAGE_HEIGHT = 256
# IMAGE_WIDTH = 512
# PIN_MEMORY = True
# PATIENCE = 30           # (MỚI) Số epoch chấp nhận không cải thiện trước khi dừng

# # --- Cấu hình Class ---
# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  
# COLOR_MAP = {0: [0, 0, 0], 1: [255, 0, 255], 2: [0, 0, 255], 3: [0, 255, 0]}

# # --- Cấu hình System ---
# torch.backends.cudnn.benchmark = True
# print(f"Device: {DEVICE} | Model: MobileNetV3-Large | Batch: {BATCH_SIZE} | Patience: {PATIENCE}")

# # ===================================================================
# # PHẦN 2: DATASET & EARLY STOPPING CLASS
# # ===================================================================
# class CocoDataset(Dataset):
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         self.img_dir = img_dir
#         self.transform = transform
#         try:
#             self.coco = COCO(annotation_file)
#             self.img_ids = list(sorted(self.coco.imgs.keys()))
#             all_cats = self.coco.loadCats(self.coco.getCatIds())
#             if allowed_classes:
#                 cats_filtered = [c for c in all_cats if c['name'] in allowed_classes]
#             else:
#                 cats_filtered = all_cats
#             cats_filtered.sort(key=lambda x: x['name'])
#             self.cat_id_to_class = {cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)}
#             print(f"Dataset loaded: {len(self.img_ids)} images")
#         except Exception as e:
#             print(f"Error loading dataset: {e}")
#             raise

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
#             image = np.array(Image.open(img_path).convert("RGB"))
#             h, w = image.shape[:2]
#             mask = np.zeros((h, w), dtype=np.uint8)
#             anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
#             for ann in anns:
#                 class_idx = self.cat_id_to_class.get(ann['category_id'], 0)
#                 if class_idx > 0:
#                     if 'segmentation' in ann:
#                         if isinstance(ann['segmentation'], list):
#                             for seg in ann['segmentation']:
#                                 poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                                 cv2.fillPoly(mask, [poly], int(class_idx))
#                         elif isinstance(ann['segmentation'], dict):
#                             mask[coco_mask.decode(ann['segmentation']) > 0] = int(class_idx)
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image = augmented['image']
#                 mask = augmented['mask']
#             return image, mask.long()
#         except Exception as e:
#             print(f"Error loading sample {idx}: {e}")
#             return torch.zeros(3, IMAGE_HEIGHT, IMAGE_WIDTH), torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # (MỚI) CLASS EARLY STOPPING
# class EarlyStopping:
#     def __init__(self, patience=10, delta=0):
#         self.patience = patience
#         self.delta = delta
#         self.counter = 0
#         self.best_loss = None
#         self.early_stop = False

#     def __call__(self, val_loss):
#         if self.best_loss is None:
#             self.best_loss = val_loss
#         elif val_loss > self.best_loss + self.delta:
#             self.counter += 1
#             print(f"⚠️ EarlyStopping counter: {self.counter} out of {self.patience}")
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_loss = val_loss
#             self.counter = 0

# # ===================================================================
# # PHẦN 3: MODEL (MobileNetV3-Large)
# # ===================================================================
# def create_deeplabv3_model(num_classes, pretrained=True):
#     print(f"Khởi tạo DeepLabV3+ MobileNetV3 (Pretrained={pretrained})...")
#     model = deeplabv3_mobilenet_v3_large(pretrained=pretrained, progress=True)
#     model.classifier = DeepLabHead(960, num_classes)
#     if hasattr(model, 'aux_classifier') and model.aux_classifier is not None:
#          model.aux_classifier = None
#     return model

# # ===================================================================
# # PHẦN 4: LOSS & UTILS
# # ===================================================================
# class CombinedLoss(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.ce = nn.CrossEntropyLoss()
    
#     def focal_loss(self, pred, target, alpha=0.25, gamma=2.0):
#         ce_loss = F.cross_entropy(pred, target, reduction='none')
#         pt = torch.exp(-ce_loss)
#         return (alpha * (1 - pt) ** gamma * ce_loss).mean()

#     def dice_loss(self, pred, target, smooth=1.0):
#         pred = F.softmax(pred, dim=1)
#         target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 3, 1, 2).float()
#         intersection = (pred * target_one_hot).sum(dim=(2, 3))
#         union = pred.sum(dim=(2, 3)) + target_one_hot.sum(dim=(2, 3))
#         return 1.0 - ((2.0 * intersection + smooth) / (union + smooth)).mean()

#     def forward(self, pred, target):
#         return 0.3 * self.ce(pred, target) + 0.5 * self.dice_loss(pred, target) + 0.2 * self.focal_loss(pred, target)

# def calculate_iou(pred, target, num_classes):
#     ious = []
#     pred = pred.view(-1)
#     target = target.view(-1)
#     for cls_id in range(num_classes):
#         p = (pred == cls_id)
#         t = (target == cls_id)
#         inter = (p & t).sum().float()
#         union = (p | t).sum().float()
#         if union == 0: ious.append(float('nan'))
#         else: ious.append((inter / union).item())
#     return ious

# # ===================================================================
# # PHẦN 5: TRAINING
# # ===================================================================
# def train_one_epoch(model, loader, optimizer, loss_fn, scaler, device):
#     model.train()
#     running_loss = 0.0
#     loop = tqdm(loader, desc="Train", leave=False)
    
#     for images, masks in loop:
#         images, masks = images.to(device), masks.to(device)
#         with torch.cuda.amp.autocast():
#             outputs = model(images)['out']
#             loss = loss_fn(outputs, masks)
#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         running_loss += loss.item()
#         loop.set_postfix(loss=loss.item())
#     return running_loss / len(loader)

# def validate(model, loader, loss_fn, device, num_classes):
#     model.eval()
#     all_ious = [[] for _ in range(num_classes)]
#     running_loss = 0.0
    
#     with torch.no_grad():
#         for images, masks in tqdm(loader, desc="Valid", leave=False):
#             images, masks = images.to(device), masks.to(device)
#             outputs = model(images)['out']
            
#             # Tính loss validation để dùng cho EarlyStopping
#             loss = loss_fn(outputs, masks)
#             running_loss += loss.item()

#             preds = torch.argmax(outputs, dim=1)
#             batch_ious = calculate_iou(preds, masks, num_classes)
#             for i, iou in enumerate(batch_ious):
#                 if not np.isnan(iou): all_ious[i].append(iou)
                
#     mean_ious = [np.nanmean(ious) if ious else 0.0 for ious in all_ious]
#     val_loss = running_loss / len(loader)
#     val_iou_score = np.nanmean([m for m in mean_ious if m > 0])
    
#     return val_loss, val_iou_score

# def run_training():
#     # Transforms
#     train_transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.HorizontalFlip(p=0.5),
#         A.RandomBrightnessContrast(p=0.2),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
#     val_transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])

#     # Data
#     train_ds = CocoDataset(TRAIN_IMG_DIR, TRAIN_JSON, transform=train_transform, allowed_classes=ALLOWED_CLASSES)
#     val_ds = CocoDataset(VALID_IMG_DIR, VALID_JSON, transform=val_transform, allowed_classes=ALLOWED_CLASSES)
    
#     train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
#     val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

#     # Model Setup
#     model = create_deeplabv3_model(NUM_CLASSES).to(DEVICE)
#     loss_fn = CombinedLoss().to(DEVICE)
#     optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
#     scaler = torch.cuda.amp.GradScaler()
#     scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    
#     # (MỚI) Khởi tạo EarlyStopping
#     early_stopping = EarlyStopping(patience=PATIENCE, delta=0.0001)
    
#     best_iou = 0.0
#     print("\n🚀 Bắt đầu Training...")
    
#     for epoch in range(NUM_EPOCHS):
#         loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, DEVICE)
#         val_loss, val_iou = validate(model, val_loader, loss_fn, DEVICE, NUM_CLASSES)
        
#         print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {loss:.4f} | Val Loss: {val_loss:.4f} | mIoU: {val_iou:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        
#         scheduler.step(val_iou)
        
#         # Lưu model tốt nhất (dựa trên mIoU)
#         if val_iou > best_iou:
#             best_iou = val_iou
#             torch.save(model.state_dict(), CHECKPOINT_PATH)
#             print(f"💾 Model Saved! (Best mIoU: {best_iou:.4f})")
            
#         # Kiểm tra Early Stopping (dựa trên Val Loss)
#         early_stopping(val_loss)
#         if early_stopping.early_stop:
#             print(f"\n⏹️ Dừng sớm (Early Stopping) tại epoch {epoch+1} vì Val Loss không giảm nữa!")
#             break
            
#     print(f"🏁 Hoàn tất! Best mIoU: {best_iou:.4f}")

# # ===================================================================
# # PHẦN 6: INFERENCE
# # ===================================================================
# def run_inference():
#     print("\n🔍 Đang chạy Inference...")
#     model = create_deeplabv3_model(NUM_CLASSES, pretrained=False)
#     if os.path.exists(CHECKPOINT_PATH):
#         model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#     else:
#         print("Không tìm thấy checkpoint!")
#         return
        
#     model.to(DEVICE).eval()
    
#     transform = A.Compose([
#         A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     colors = np.array(list(COLOR_MAP.values()), dtype=np.uint8)
#     files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('jpg', 'png'))]
    
#     for f in tqdm(files):
#         path = os.path.join(TEST_REAL_DIR, f)
#         img = cv2.imread(path)
#         if img is None: continue
#         orig_size = img.shape[:2]
        
#         img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#         tensor = transform(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
        
#         with torch.no_grad():
#             output = model(tensor)['out']
#             pred = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)
            
#         pred = cv2.resize(pred, (orig_size[1], orig_size[0]), interpolation=cv2.INTER_NEAREST)
#         color_mask = np.zeros_like(img)
#         for cls_id in range(1, NUM_CLASSES):
#             color_mask[pred == cls_id] = colors[cls_id]
            
#         overlay = cv2.addWeighted(img, 0.6, color_mask, 0.4, 0)
#         cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"res_{f}"), overlay)

# if __name__ == "__main__":
#     run_training()
#     run_inference()

**Giải Pháp 1**

In [7]:
# # ===================================================================
# # GIẢI PHÁP 1
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import cv2
# import numpy as np
# import json
# from PIL import Image
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import matplotlib.pyplot as plt
# import gc
# import psutil
# import time
# import traceback

# # ===================================================================
# # PHẦN 0: HÀM MONITORING & DEBUG (Giữ nguyên)
# # ===================================================================

# def get_gpu_memory_info():
#     """Lấy thông tin GPU memory"""
#     if torch.cuda.is_available():
#         allocated = torch.cuda.memory_allocated() / 1024**3  # GB
#         reserved = torch.cuda.memory_reserved() / 1024**3    # GB
#         max_allocated = torch.cuda.max_memory_allocated() / 1024**3  # GB
#         total = torch.cuda.get_device_properties(0).total_memory / 1024**3  # GB
#         return {
#             'allocated': allocated,
#             'reserved': reserved,
#             'max_allocated': max_allocated,
#             'total': total,
#             'free': total - allocated
#         }
#     return None

# def get_cpu_memory_info():
#     """Lấy thông tin CPU memory"""
#     mem = psutil.virtual_memory()
#     return {
#         'total': mem.total / 1024**3,  # GB
#         'available': mem.available / 1024**3,  # GB
#         'used': mem.used / 1024**3,  # GB
#         'percent': mem.percent
#     }

# def print_memory_status(stage=""):
#     """In chi tiết memory status"""
#     print(f"\n{'='*70}")
#     print(f"🔍 MEMORY STATUS - {stage}")
#     print(f"{'='*70}")
    
#     # CPU Memory
#     cpu_mem = get_cpu_memory_info()
#     print(f"💻 CPU Memory:")
#     print(f"    Total: {cpu_mem['total']:.2f} GB")
#     print(f"    Used: {cpu_mem['used']:.2f} GB ({cpu_mem['percent']:.1f}%)")
#     print(f"    Available: {cpu_mem['available']:.2f} GB")
    
#     # GPU Memory
#     if torch.cuda.is_available():
#         for i in range(torch.cuda.device_count()):
#             gpu_mem = get_gpu_memory_info()
#             print(f"\n🎮 GPU {i} Memory:")
#             print(f"    Total: {gpu_mem['total']:.2f} GB")
#             print(f"    Allocated: {gpu_mem['allocated']:.2f} GB")
#             print(f"    Reserved: {gpu_mem['reserved']:.2f} GB")
#             print(f"    Free: {gpu_mem['free']:.2f} GB")
#             print(f"    Max Allocated: {gpu_mem['max_allocated']:.2f} GB")
#     print(f"{'='*70}\n")

# # ===================================================================
# # PHẦN 1: CẤU HÌNH
# # ===================================================================
# print("--- PHẦN 1: Bắt đầu cấu hình ---")

# # --- Đường dẫn ---
# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# # --- Checkpoint & Outputs ---
# CHECKPOINT_PATH = "/kaggle/working/sol2_unet_filtered_best.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/sol2_predictions/"
# LOG_FILE = "/kaggle/working/training_log.txt"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# # --- Hyperparameters ---
# LEARNING_RATE = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 8      # ✅ SỬA ĐỔI: Giảm BATCH_SIZE để gỡ lỗi
# NUM_EPOCHS = 100
# NUM_WORKERS = 0     # ✅ SỬA ĐỔI: Set = 0 để gỡ lỗi. Nếu chạy thành công, hãy set lại = 2
# IMAGE_HEIGHT = 256
# IMAGE_WIDTH = 512
# PIN_MEMORY = True

# # ✅ SỬA ĐỔI: Định nghĩa class bạn muốn train
# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  # 3 class + 1 background = 4

# # ✅ SỬA ĐỔI: Tạo class name tự động theo thứ tự alphabet để khớp với Dataset
# CLASS_NAMES = ['background'] + sorted(list(ALLOWED_CLASSES))
# # Kết quả CLASS_NAMES sẽ là:
# # ['background', 'le_duong', 'phan_cach_nguoc_chieu', 'phan_lan']

# # ✅ SỬA ĐỔI: Color map phải khớp với CLASS_NAMES ở trên
# COLOR_MAP = {
#     0: [0, 0, 0],       # background
#     1: [255, 0, 255],   # le_duong
#     2: [0, 0, 255],     # phan_cach_nguoc_chieu
#     3: [0, 255, 0],     # phan_lan
# }

# # --- CRITICAL: Set memory management ---
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.deterministic = False

# print(f"Device: {DEVICE}")
# print(f"Batch Size: {BATCH_SIZE}")
# print(f"Image Size: {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
# print(f"Num Workers: {NUM_WORKERS}")
# print(f"Total Classes (đã lọc): {NUM_CLASSES}")
# print(f"Class names (đã lọc): {CLASS_NAMES}")

# # print_memory_status("AFTER CONFIG")

# # ===================================================================
# # PHẦN 2: LỚP DATASET (ĐÃ SỬA ĐỔI)
# # ===================================================================
# class CocoDataset(Dataset):
#     """Dataset class với error handling và lọc class"""
    
#     # ✅ SỬA ĐỔI: Thêm tham số allowed_classes
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         print(f"\n📂 Loading dataset from: {annotation_file}")
#         self.img_dir = img_dir
#         self.transform = transform
#         self.allowed_classes_set = allowed_classes if allowed_classes else None
        
#         try:
#             self.coco = COCO(annotation_file)
#             self.img_ids = list(sorted(self.coco.imgs.keys()))
            
#             # Lấy TẤT CẢ categories từ file
#             all_cats_dict = self.coco.loadCats(self.coco.getCatIds())
            
#             # ✅ SỬA ĐỔI: Lọc categories
#             if self.allowed_classes_set:
#                 print(f"Filtering for: {self.allowed_classes_set}")
#                 cats_filtered = [
#                     cat for cat in all_cats_dict 
#                     if cat['name'] in self.allowed_classes_set
#                 ]
#             else:
#                 print("No filter, loading all classes.")
#                 cats_filtered = all_cats_dict

#             # Sort các class đã lọc theo tên (để đảm bảo thứ tự)
#             cats_filtered.sort(key=lambda x: x['name'])
            
#             # ✅ SỬA ĐỔI: Tạo mapping chỉ từ các class đã lọc
#             # class 0 là background
#             self.cat_id_to_class = {
#                 cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)
#             }
#             self.class_names = ['background'] + [cat['name'] for cat in cats_filtered]
#             self.num_classes = len(self.class_names) # Sẽ là 4 (3 + 1)
            
#             print(f"✅ Successfully loaded {len(self.img_ids)} images")
#             print(f"✅ Found and filtered to {len(cats_filtered)} classes: {self.class_names[1:]}")
#             print(f"Total classes (inc. background): {self.num_classes}")
            
#         except Exception as e:
#             print(f"❌ ERROR loading dataset: {e}")
#             traceback.print_exc()
#             raise

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
            
#             # Load image
#             image = np.array(Image.open(img_path).convert("RGB"))
#             h, w = image.shape[:2]
            
#             # Create empty mask (full background)
#             mask = np.zeros((h, w), dtype=np.uint8)
            
#             # Load annotations
#             ann_ids = self.coco.getAnnIds(imgIds=img_id)
#             anns = self.coco.loadAnns(ann_ids)
            
#             # Fill mask
#             for ann in anns:
#                 cat_id = ann['category_id']
                
#                 # ✅ SỬA ĐỔI: Dùng .get(cat_id, 0)
#                 # Nếu cat_id nằm trong self.cat_id_to_class -> gán class_idx (1, 2, hoặc 3)
#                 # Nếu không (ví dụ: class 'objects') -> gán 0 (background)
#                 class_idx = self.cat_id_to_class.get(cat_id, 0) 
                
#                 if class_idx > 0: # Chỉ vẽ nếu nó là class ta muốn
#                     if 'segmentation' in ann and isinstance(ann['segmentation'], list):
#                         for seg in ann['segmentation']:
#                             poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                             cv2.fillPoly(mask, [poly], color=int(class_idx))
#                     elif 'segmentation' in ann and isinstance(ann['segmentation'], dict):
#                         rle = ann['segmentation']
#                         binary_mask = coco_mask.decode(rle)
#                         mask[binary_mask > 0] = int(class_idx)
            
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image = augmented['image']
#                 mask = augmented['mask']
                
#             return image, mask.long()
            
#         except Exception as e:
#             print(f"❌ ERROR loading sample {idx} (Image: {img_info.get('file_name', 'N/A')}): {e}")
#             traceback.print_exc()
#             # Trả về data rỗng để loader có thể bỏ qua
#             return torch.randn(3, IMAGE_HEIGHT, IMAGE_WIDTH), \
#                    torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # ===================================================================
# # PHẦN 3: MODEL (Giữ nguyên)
# # ===================================================================
# class DoubleConv(nn.Module):
#     def __init__(self, in_c, out_c):
#         super().__init__()
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False), 
#             nn.BatchNorm2d(out_c), 
#             nn.ReLU(inplace=True), 
#             nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False), 
#             nn.BatchNorm2d(out_c), 
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x): 
#         return self.conv(x)

# class UNET(nn.Module):
#     def __init__(self, in_c=3, out_c=4, features=[64, 128, 256, 512]):
#         super().__init__()
#         self.ups, self.downs, self.pool = nn.ModuleList(), nn.ModuleList(), nn.MaxPool2d(2, 2)
        
#         for f in features: 
#             self.downs.append(DoubleConv(in_c, f))
#             in_c = f
            
#         for f in reversed(features): 
#             self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
#             self.ups.append(DoubleConv(f*2, f))
            
#         self.bottleneck = DoubleConv(features[-1], features[-1]*2)
#         self.final_conv = nn.Conv2d(features[0], out_c, 1)
        
#     def forward(self, x):
#         skips = []
#         for down in self.downs: 
#             x = down(x)
#             skips.append(x)
#             x = self.pool(x)
            
#         x = self.bottleneck(x)
#         skips = skips[::-1]
        
#         for i in range(0, len(self.ups), 2):
#             x = self.ups[i](x)
#             skip = skips[i//2]
#             if x.shape != skip.shape: 
#                 x = TF.resize(x, size=skip.shape[2:])
#             x = self.ups[i+1](torch.cat((skip, x), 1))
            
#         return self.final_conv(x)

# # ===================================================================
# # PHẦN 4: TRAINING FUNCTIONS (Giữ nguyên)
# # ===================================================================

# class EarlyStopping:
#     def __init__(self, patience=10, delta=0.0001, path='checkpoint.pth', trace_func=print):
#         self.patience = patience
#         self.delta = delta
#         self.path = path
#         self.trace_func = trace_func
#         self.counter = 0
#         self.best_score = None
#         self.early_stop = False
#         self.best_loss = float('inf')

#     def __call__(self, val_loss, model):
#         score = -val_loss
#         if self.best_score is None:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#         elif score < self.best_score + self.delta:
#             self.counter += 1
#             self.trace_func(f'EarlyStopping counter: {self.counter}/{self.patience}')
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#             self.counter = 0

#     def save_checkpoint(self, val_loss, model):
#         self.trace_func(f'✅ Validation loss giảm ({self.best_loss:.4f} → {val_loss:.4f}). Đang lưu model...')
#         if isinstance(model, nn.DataParallel):
#             torch.save(model.module.state_dict(), self.path)
#         else:
#             torch.save(model.state_dict(), self.path)
#         self.best_loss = val_loss

# def calculate_iou(pred, target, num_classes):
#     ious = []
#     pred = pred.view(-1)
#     target = target.view(-1)
    
#     for cls_id in range(num_classes):
#         pred_cls = (pred == cls_id)
#         target_cls = (target == cls_id)
        
#         intersection = (pred_cls & target_cls).sum().float()
#         union = (pred_cls | target_cls).sum().float()
        
#         if union == 0:
#             ious.append(float('nan'))
#         else:
#             ious.append((intersection / union).item())
#     return ious

# def train_one_epoch(model, loader, optimizer, loss_fn, scaler, device, epoch):
#     model.train()
#     running_loss = 0.0
#     loop = tqdm(loader, desc=f"Training Epoch {epoch}")
    
#     try:
#         for batch_idx, (images, masks) in enumerate(loop):
#             # if batch_idx % 20 == 0:
#             #     gpu_mem = get_gpu_memory_info()
#             #     if gpu_mem:
#             #         loop.set_postfix(
#             #             loss=running_loss/(batch_idx+1) if batch_idx > 0 else 0,
#             #             gpu_alloc=f"{gpu_mem['allocated']:.2f}GB",
#             #             gpu_free=f"{gpu_mem['free']:.2f}GB"
#             #         )
            
#             images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False) # Sửa nhỏ
#             masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False) # Sửa nhỏ
            
#             with torch.amp.autocast('cuda'):
#                 outputs = model(images)
#                 loss = loss_fn(outputs, masks)
                
#             optimizer.zero_grad(set_to_none=True) 
#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()
            
#             running_loss += loss.item()
#             loop.set_postfix(loss=loss.item())
            
#             del images, masks, outputs, loss
            
#             if batch_idx % 10 == 0:
#                 torch.cuda.empty_cache()
        
#         return running_loss / len(loader)
        
#     except Exception as e:
#         print(f"\n❌ ERROR in training loop at batch {batch_idx}:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status(f"ERROR at batch {batch_idx}")
#         raise

# def validate(model, loader, loss_fn, device, num_classes):
#     model.eval()
#     running_loss = 0.0
#     all_ious = [[] for _ in range(num_classes)]
    
#     try:
#         with torch.no_grad():
#             for batch_idx, (images, masks) in enumerate(tqdm(loader, desc="Validating")):
#                 images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False) # Sửa nhỏ
#                 masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False) # Sửa nhỏ
                
#                 outputs = model(images)
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
#                 running_loss += loss.item()
                
#                 preds = torch.argmax(outputs, dim=1)
#                 batch_ious = calculate_iou(preds, masks, num_classes)
                
#                 for i, iou in enumerate(batch_ious):
#                     if not np.isnan(iou):
#                         all_ious[i].append(iou)
                
#                 del images, masks, outputs, preds
                        
#         mean_ious = [np.nanmean(ious) if ious else 0.0 for ious in all_ious]
#         mean_iou_total = np.nanmean([iou for iou in mean_ious if not np.isnan(iou)])
#         val_loss = running_loss / len(loader)
        
#         print(f"\nValidation Loss: {val_loss:.4f}")
#         print(f"Mean IoU (mIoU): {mean_iou_total:.4f}")
        
#         # ✅ SỬA ĐỔI: Dùng CLASS_NAMES (global)
#         for i, class_name in enumerate(CLASS_NAMES): 
#             print(f"  - {class_name}: {mean_ious[i]:.4f}")
            
#         return val_loss, mean_iou_total
        
#     except Exception as e:
#         print(f"\n❌ ERROR in validation loop:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR in validation")
#         raise

# # ===================================================================
# # PHẦN 5: MAIN TRAINING (Sửa đổi nhỏ)
# # ===================================================================
# def run_training():
#     print("\n" + "="*70)
#     print("🚀 BẮT ĐẦU HUẤN LUYỆN - CHẾ ĐỘ LỌC CLASS")
#     print("="*70)
    
#     try:
#         # --- 1. Transforms ---
#         print("\n📋 Khởi tạo transforms...")
#         train_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#             A.HorizontalFlip(p=0.5),
#             A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
#             A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
        
#         val_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
#         print("✅ Transforms created")
        
#         # --- 2. Datasets ---
#         print("\n📦 Loading datasets...")
#         # print_memory_status("BEFORE LOADING DATASETS")
        
#         # ✅ SỬA ĐỔI: Truyền allowed_classes vào
#         train_dataset = CocoDataset(
#             TRAIN_IMG_DIR, TRAIN_JSON, 
#             transform=train_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # print_memory_status("AFTER LOADING TRAIN DATASET")
        
#         val_dataset = CocoDataset(
#             VALID_IMG_DIR, VALID_JSON, 
#             transform=val_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # # print_memory_status("AFTER LOADING VAL DATASET")

#         # ✅ SỬA ĐỔI: Kiểm tra xem dataset có trả về đúng số class không
#         if train_dataset.num_classes != NUM_CLASSES or val_dataset.num_classes != NUM_CLASSES:
#              print(f"❌ LỖI NGHIÊM TRỌNG: Cấu hình NUM_CLASSES ({NUM_CLASSES}) không khớp với dataset ({train_dataset.num_classes})")
#              raise ValueError("Class count mismatch")

#         # --- 3. DataLoaders ---
#         print("\n🔄 Creating DataLoaders...")
#         train_loader = DataLoader(
#             train_dataset, batch_size=BATCH_SIZE, shuffle=True,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         val_loader = DataLoader(
#             val_dataset, batch_size=BATCH_SIZE, shuffle=False,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         print("✅ DataLoaders created")
#         print_memory_status("AFTER CREATING DATALOADERS")
        
#         # --- 4. Model ---
#         print("\n🏗️ Creating model...")
#         # ✅ SỬA ĐỔI: Model được tạo với NUM_CLASSES ( = 4 )
#         model = UNET(in_c=3, out_c=NUM_CLASSES).to(DEVICE)
        
#         if torch.cuda.device_count() > 1:
#             print(f"✅ Using {torch.cuda.device_count()} GPUs with DataParallel")
#             model = nn.DataParallel(model)
#         else:
#             print("✅ Using single GPU")
            
#         # print_memory_status("AFTER CREATING MODEL")
        
#         # --- 5. Loss, Optimizer, Scheduler ---
#         print("\n⚙️ Setting up training components...")
#         loss_fn = nn.CrossEntropyLoss().to(DEVICE)
#         optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
#         scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#             optimizer, mode='min', factor=0.1, patience=5, verbose=True
#         )
#         scaler = torch.amp.GradScaler('cuda')
#         early_stopping = EarlyStopping(patience=10, delta=0.0001, path=CHECKPOINT_PATH)
#         print("✅ Training components ready")
        
#         # --- 6. Test forward pass ---
#         print("\n🧪 Testing forward pass with dummy batch...")
#         try:
#             dummy_input = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH).to(DEVICE)
#             with torch.no_grad():
#                 dummy_output = model(dummy_input)
#             # ✅ SỬA ĐỔI: Kiểm tra output shape
#             print(f"✅ Forward pass successful! Output shape: {dummy_output.shape}")
#             if dummy_output.shape != (2, NUM_CLASSES, IMAGE_HEIGHT, IMAGE_WIDTH):
#                  print(f"❌ LỖI SHAPE: Output shape {dummy_output.shape} không khớp mong đợi")
#                  raise ValueError("Output shape mismatch")
#             del dummy_input, dummy_output
#             torch.cuda.empty_cache()
#             # print_memory_status("AFTER DUMMY FORWARD PASS")
#         except Exception as e:
#             print(f"❌ Forward pass failed: {e}")
#             traceback.print_exc()
#             return
        
#         # --- 7. Training Loop ---
#         print("\n" + "="*70)
#         print("🎯 STARTING TRAINING LOOP")
#         print("="*70)
        
#         best_iou = -1.0
        
#         for epoch in range(NUM_EPOCHS):
#             print(f"\n{'='*70}")
#             print(f"📅 Epoch {epoch+1}/{NUM_EPOCHS}")
#             print(f"{'='*70}")
#             # print_memory_status(f"START of Epoch {epoch+1}")
            
#             epoch_start = time.time()
            
#             # Training
#             train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, DEVICE, epoch+1)
#             # print_memory_status(f"AFTER TRAINING Epoch {epoch+1}")
            
#             # Validation
#             # ✅ SỬA ĐỔI: Truyền NUM_CLASSES vào validate
#             val_loss, val_iou = validate(model, val_loader, loss_fn, DEVICE, NUM_CLASSES)
#             # print_memory_status(f"AFTER VALIDATION Epoch {epoch+1}")
            
#             epoch_time = time.time() - epoch_start
            
#             print(f"\n📊 Tóm tắt Epoch {epoch+1}:")
#             print(f"  ⏱️  Time: {epoch_time:.2f}s")
#             print(f"  📉 Train Loss: {train_loss:.4f}")
#             print(f"  📉 Val Loss: {val_loss:.4f}")
#             print(f"  📈 Val mIoU: {val_iou:.4f}")
            
#             scheduler.step(val_loss)
            
#             if val_iou > best_iou:
#                 best_iou = val_iou
#                 print(f"✨ New best mIoU: {best_iou:.4f}")
                
#             early_stopping(val_loss, model)
#             if early_stopping.early_stop:
#                 print("\n⏹️ Early stopping triggered")
#                 break
                
#             # Clean up
#             torch.cuda.empty_cache()
#             gc.collect()
                 
#         print("\n" + "="*70)
#         print("🎉 TRAINING COMPLETED")
#         print("="*70)
#         print(f"✅ Best model saved at: {CHECKPOINT_PATH}")
#         print(f"🏆 Best mIoU: {best_iou:.4f}")
#         print_memory_status("FINAL")
        
#     except Exception as e:
#         print(f"\n{'='*70}")
#         print(f"❌ CRITICAL ERROR IN TRAINING")
#         print(f"{'='*70}")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR STATE")
#         raise

# # ===================================================================
# # PHẦN 6: INFERENCE (Sửa đổi nhỏ)
# # ===================================================================

# def preprocess_image_test(image_path, height, width):
#     image = cv2.imread(image_path)
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     original_size = image.shape[:2]
    
#     transform = A.Compose([
#         A.Resize(height, width, interpolation=cv2.INTER_LINEAR),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     transformed = transform(image=image_rgb)
#     tensor = transformed['image'].unsqueeze(0).to(DEVICE)
#     return tensor, image, original_size

# def mask_to_color_img(mask):
#     h, w = mask.shape
#     color_mask = np.zeros((h, w, 3), dtype=np.uint8)
#     for class_id, color in COLOR_MAP.items(): # ✅ SỬA ĐỔI: Dùng COLOR_MAP global
#         color_mask[mask == class_id] = color
#     return color_mask

# def overlay_mask_on_image(image, mask, alpha=0.5):
#     return cv2.addWeighted(image, 1 - alpha, mask, alpha, 0)

# def run_inference():
#     print(f"\n{'='*70}")
#     print("🔍 STARTING INFERENCE")
#     print(f"{'='*70}")
    
#     try:
#         print("\n🏗️ Loading model...")
#         # ✅ SỬA ĐỔI: Khởi tạo model với NUM_CLASSES ( = 4 )
#         model = UNET(in_c=3, out_c=NUM_CLASSES)
        
#         try:
#             model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#         except RuntimeError:
#             print("⚠️ Model was trained with DataParallel, removing 'module.' prefix...")
#             from collections import OrderedDict
#             state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
#             new_state_dict = OrderedDict()
#             for k, v in state_dict.items():
#                 name = k.replace('module.', '')
#                 new_state_dict[name] = v
#             model.load_state_dict(new_state_dict)

#         model.to(DEVICE)
#         model.eval()
#         print("✅ Model loaded successfully")

#         image_files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
#         print(f"\n📁 Found {len(image_files)} test images")
        
#         if len(image_files) == 0:
#             print("⚠️ No test images found!")
#             return
        
#         for img_file in tqdm(image_files, desc="Processing"):
#             img_path = os.path.join(TEST_REAL_DIR, img_file)
            
#             tensor, original_bgr_img, original_size = preprocess_image_test(
#                 img_path, IMAGE_HEIGHT, IMAGE_WIDTH
#             )
            
#             with torch.no_grad():
#                 output = model(tensor)
#                 if isinstance(output, dict):
#                     output = output['out']
                
#                 prediction_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
                
#             prediction_resized = cv2.resize(
#                 prediction_mask.astype(np.uint8),
#                 (original_size[1], original_size[0]),
#                 interpolation=cv2.INTER_NEAREST
#             )
            
#             color_mask = mask_to_color_img(prediction_resized)
#             overlay_img = overlay_mask_on_image(original_bgr_img, color_mask, alpha=0.6)
            
#             base_name = os.path.splitext(img_file)[0]
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_mask.png"), color_mask)
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_overlay.png"), overlay_img)
            
#         print(f"\n✅ Inference completed! Results saved to: {PREDICTION_OUTPUT_DIR}")
        
#     except Exception as e:
#         print(f"\n❌ ERROR in inference:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         raise

# # ===================================================================
# # MAIN
# # ===================================================================
# if __name__ == "__main__":
#     print("\n" + "="*70)
#     print("🚀 STARTING PROGRAM - DEBUG & MONITORING ENABLED")
#     print("="*70)
    
#     try:
#         if DEVICE == "cuda":
#             torch.cuda.empty_cache()
#             gc.collect()
            
#         # print_memory_status("INITIAL")
        
#         # Training
#         run_training()
        
#         # Inference
#         if os.path.exists(CHECKPOINT_PATH):
#             run_inference()
#         else:
#             print(f"⚠️ Checkpoint not found at {CHECKPOINT_PATH}")
            
#     except KeyboardInterrupt:
#         print("\n⚠️ Training interrupted by user")
#         # print_memory_status("INTERRUPTED")
#     except Exception as e:
#         print(f"\n❌ FATAL ERROR:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("FATAL ERROR")

**Giải Pháp 2**

In [8]:
# # ===================================================================
# # GIẢI PHÁP 2: U-NET GỐC + LOSS XỊN (DICE + FOCAL)
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import cv2
# import numpy as np
# import json
# from PIL import Image
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import matplotlib.pyplot as plt
# import gc
# import psutil
# import time
# import traceback

# # ===================================================================
# # PHẦN 0: HÀM MONITORING & DEBUG (Giữ nguyên)
# # ===================================================================

# def get_gpu_memory_info():
#     """Lấy thông tin GPU memory"""
#     if torch.cuda.is_available():
#         allocated = torch.cuda.memory_allocated() / 1024**3  # GB
#         reserved = torch.cuda.memory_reserved() / 1024**3   # GB
#         max_allocated = torch.cuda.max_memory_allocated() / 1024**3  # GB
#         total = torch.cuda.get_device_properties(0).total_memory / 1024**3  # GB
#         return {
#             'allocated': allocated,
#             'reserved': reserved,
#             'max_allocated': max_allocated,
#             'total': total,
#             'free': total - allocated
#         }
#     return None

# def get_cpu_memory_info():
#     """Lấy thông tin CPU memory"""
#     mem = psutil.virtual_memory()
#     return {
#         'total': mem.total / 1024**3,  # GB
#         'available': mem.available / 1024**3,  # GB
#         'used': mem.used / 1024**3,  # GB
#         'percent': mem.percent
#     }

# def print_memory_status(stage=""):
#     """In chi tiết memory status"""
#     print(f"\n{'='*70}")
#     print(f"🔍 MEMORY STATUS - {stage}")
#     print(f"{'='*70}")
    
#     # CPU Memory
#     cpu_mem = get_cpu_memory_info()
#     print(f"💻 CPU Memory:")
#     print(f"    Total: {cpu_mem['total']:.2f} GB")
#     print(f"    Used: {cpu_mem['used']:.2f} GB ({cpu_mem['percent']:.1f}%)")
#     print(f"    Available: {cpu_mem['available']:.2f} GB")
    
#     # GPU Memory
#     if torch.cuda.is_available():
#         for i in range(torch.cuda.device_count()):
#             gpu_mem = get_gpu_memory_info()
#             print(f"\n🎮 GPU {i} Memory:")
#             print(f"    Total: {gpu_mem['total']:.2f} GB")
#             print(f"    Allocated: {gpu_mem['allocated']:.2f} GB")
#             print(f"    Reserved: {gpu_mem['reserved']:.2f} GB")
#             print(f"    Free: {gpu_mem['free']:.2f} GB")
#             print(f"    Max Allocated: {gpu_mem['max_allocated']:.2f} GB")
#     print(f"{'='*70}\n")

# # ===================================================================
# # PHẦN 1: CẤU HÌNH
# # ===================================================================
# print("--- PHẦN 1: Bắt đầu cấu hình ---")

# # --- Đường dẫn ---
# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# # --- Checkpoint & Outputs (THAY ĐỔI) ---
# CHECKPOINT_PATH = "/kaggle/working/gp2_unet_dicefocal_best.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/gp2_predictions/"
# LOG_FILE = "/kaggle/working/training_log_gp2.txt"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# # --- Hyperparameters ---
# LEARNING_RATE = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 8     # Giảm BATCH_SIZE để gỡ lỗi
# NUM_EPOCHS = 100
# NUM_WORKERS = 0    # Set = 0 để gỡ lỗi.
# IMAGE_HEIGHT = 256
# IMAGE_WIDTH = 512
# PIN_MEMORY = True

# # --- Cấu hình Class (Giữ nguyên) ---
# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  # 3 class + 1 background = 4
# CLASS_NAMES = ['background'] + sorted(list(ALLOWED_CLASSES))
# # ['background', 'le_duong', 'phan_cach_nguoc_chieu', 'phan_lan']

# COLOR_MAP = {
#     0: [0, 0, 0],         # background
#     1: [255, 0, 255],     # le_duong
#     2: [0, 0, 255],       # phan_cach_nguoc_chieu
#     3: [0, 255, 0],       # phan_lan
# }

# # --- Cấu hình Memory (Giữ nguyên) ---
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.deterministic = False

# print(f"Device: {DEVICE}")
# print(f"Batch Size: {BATCH_SIZE}")
# print(f"Image Size: {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
# print(f"Num Workers: {NUM_WORKERS}")
# print(f"Total Classes (đã lọc): {NUM_CLASSES}")
# print(f"Class names (đã lọc): {CLASS_NAMES}")

# # print_memory_status("AFTER CONFIG")

# # ===================================================================
# # PHẦN 2: LỚP DATASET (Giữ nguyên code vá lỗi)
# # ===================================================================
# class CocoDataset(Dataset):
#     """Dataset class với error handling và lọc class"""
    
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         print(f"\n📂 Loading dataset from: {annotation_file}")
#         self.img_dir = img_dir
#         self.transform = transform
#         self.allowed_classes_set = allowed_classes if allowed_classes else None
        
#         try:
#             self.coco = COCO(annotation_file)
#             self.img_ids = list(sorted(self.coco.imgs.keys()))
            
#             all_cats_dict = self.coco.loadCats(self.coco.getCatIds())
            
#             if self.allowed_classes_set:
#                 print(f"Filtering for: {self.allowed_classes_set}")
#                 cats_filtered = [
#                     cat for cat in all_cats_dict 
#                     if cat['name'] in self.allowed_classes_set
#                 ]
#             else:
#                 print("No filter, loading all classes.")
#                 cats_filtered = all_cats_dict

#             cats_filtered.sort(key=lambda x: x['name'])
            
#             self.cat_id_to_class = {
#                 cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)
#             }
#             self.class_names = ['background'] + [cat['name'] for cat in cats_filtered]
#             self.num_classes = len(self.class_names)
            
#             print(f"✅ Successfully loaded {len(self.img_ids)} images")
#             print(f"✅ Found and filtered to {len(cats_filtered)} classes: {self.class_names[1:]}")
#             print(f"Total classes (inc. background): {self.num_classes}")
            
#         except Exception as e:
#             print(f"❌ ERROR loading dataset: {e}")
#             traceback.print_exc()
#             raise

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
            
#             image = np.array(Image.open(img_path).convert("RGB"))
#             h, w = image.shape[:2]
            
#             mask = np.zeros((h, w), dtype=np.uint8)
            
#             ann_ids = self.coco.getAnnIds(imgIds=img_id)
#             anns = self.coco.loadAnns(ann_ids)
            
#             for ann in anns:
#                 cat_id = ann['category_id']
#                 class_idx = self.cat_id_to_class.get(cat_id, 0) 
                
#                 if class_idx > 0:
#                     if 'segmentation' in ann and isinstance(ann['segmentation'], list):
#                         for seg in ann['segmentation']:
#                             poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                             cv2.fillPoly(mask, [poly], color=int(class_idx))
#                     elif 'segmentation' in ann and isinstance(ann['segmentation'], dict):
#                         rle = ann['segmentation']
#                         binary_mask = coco_mask.decode(rle)
#                         mask[binary_mask > 0] = int(class_idx)
            
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image = augmented['image']
#                 mask = augmented['mask']
                
#             return image, mask.long()
            
#         except Exception as e:
#             print(f"❌ ERROR loading sample {idx} (Image: {img_info.get('file_name', 'N/A')}): {e}")
#             traceback.print_exc()
#             return torch.randn(3, IMAGE_HEIGHT, IMAGE_WIDTH), \
#                    torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # ===================================================================
# # PHẦN 3: MODEL (Giữ nguyên U-Net Gốc)
# # ===================================================================
# class DoubleConv(nn.Module):
#     def __init__(self, in_c, out_c):
#         super().__init__()
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False), 
#             nn.BatchNorm2d(out_c), 
#             nn.ReLU(inplace=True), 
#             nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False), 
#             nn.BatchNorm2d(out_c), 
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x): 
#         return self.conv(x)

# class UNET(nn.Module):
#     def __init__(self, in_c=3, out_c=4, features=[64, 128, 256, 512]):
#         super().__init__()
#         self.ups, self.downs, self.pool = nn.ModuleList(), nn.ModuleList(), nn.MaxPool2d(2, 2)
        
#         for f in features: 
#             self.downs.append(DoubleConv(in_c, f))
#             in_c = f
            
#         for f in reversed(features): 
#             self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
#             self.ups.append(DoubleConv(f*2, f))
            
#         self.bottleneck = DoubleConv(features[-1], features[-1]*2)
#         self.final_conv = nn.Conv2d(features[0], out_c, 1)
        
#     def forward(self, x):
#         skips = []
#         for down in self.downs: 
#             x = down(x)
#             skips.append(x)
#             x = self.pool(x)
            
#         x = self.bottleneck(x)
#         skips = skips[::-1]
        
#         for i in range(0, len(self.ups), 2):
#             x = self.ups[i](x)
#             skip = skips[i//2]
#             if x.shape != skip.shape: 
#                 x = TF.resize(x, size=skip.shape[2:])
#             x = self.ups[i+1](torch.cat((skip, x), 1))
            
#         return self.final_conv(x)

# # ===================================================================
# # PHẦN 3_BIS: LOSS FUNCTIONS (THAY ĐỔI)
# # ===================================================================
# print("\nKhởi tạo các class Loss Function (Dice, Focal)...")

# class DiceLoss(nn.Module):
#     """Dice Loss - Tốt cho vấn đề imbalanced classes và ranh giới"""
#     def __init__(self, smooth=1.0):
#         super().__init__()
#         self.smooth = smooth
        
#     def forward(self, pred, target):
#         pred = F.softmax(pred, dim=1)
#         target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 3, 1, 2).float()
        
#         # Bắt đầu từ class 1 (bỏ class 0 - background)
#         intersection = (pred[:, 1:] * target_one_hot[:, 1:]).sum(dim=(2, 3))
#         union = pred[:, 1:].sum(dim=(2, 3)) + target_one_hot[:, 1:].sum(dim=(2, 3))
        
#         dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
#         return 1.0 - dice.mean()

# class FocalLoss(nn.Module):
#     """Focal Loss - Tập trung vào các pixel khó phân loại (như ranh giới)"""
#     def __init__(self, alpha=0.25, gamma=2.0):
#         super().__init__()
#         self.alpha = alpha
#         self.gamma = gamma
        
#     def forward(self, pred, target):
#         ce_loss = F.cross_entropy(pred, target, reduction='none')
#         pt = torch.exp(-ce_loss) # probabilities của class đúng
#         focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
#         return focal_loss.mean()

# class CombinedLoss(nn.Module):
#     """Kết hợp Dice + Focal Loss"""
#     def __init__(self, dice_weight=0.6, focal_weight=0.4):
#         super().__init__()
#         self.dice = DiceLoss()
#         self.focal = FocalLoss()
#         self.dice_weight = dice_weight
#         self.focal_weight = focal_weight
#         print(f"Khởi tạo CombinedLoss (Dice: {dice_weight*100}%, Focal: {focal_weight*100}%)")

#     def forward(self, pred, target):
#         return self.dice_weight * self.dice(pred, target) + self.focal_weight * self.focal(pred, target)

# # ===================================================================
# # PHẦN 4: TRAINING FUNCTIONS (Giữ nguyên)
# # ===================================================================

# class EarlyStopping:
#     def __init__(self, patience=10, delta=0.0001, path='checkpoint.pth', trace_func=print):
#         self.patience = patience
#         self.delta = delta
#         self.path = path
#         self.trace_func = trace_func
#         self.counter = 0
#         self.best_score = None
#         self.early_stop = False
#         self.best_loss = float('inf')

#     def __call__(self, val_loss, model):
#         score = -val_loss
#         if self.best_score is None:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#         elif score < self.best_score + self.delta:
#             self.counter += 1
#             self.trace_func(f'EarlyStopping counter: {self.counter}/{self.patience}')
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#             self.counter = 0

#     def save_checkpoint(self, val_loss, model):
#         self.trace_func(f'✅ Validation loss giảm ({self.best_loss:.4f} → {val_loss:.4f}). Đang lưu model...')
#         if isinstance(model, nn.DataParallel):
#             torch.save(model.module.state_dict(), self.path)
#         else:
#             torch.save(model.state_dict(), self.path)
#         self.best_loss = val_loss

# def calculate_iou(pred, target, num_classes):
#     ious = []
#     pred = pred.view(-1)
#     target = target.view(-1)
    
#     for cls_id in range(num_classes):
#         pred_cls = (pred == cls_id)
#         target_cls = (target == cls_id)
        
#         intersection = (pred_cls & target_cls).sum().float()
#         union = (pred_cls | target_cls).sum().float()
        
#         if union == 0:
#             ious.append(float('nan'))
#         else:
#             ious.append((intersection / union).item())
#     return ious

# def train_one_epoch(model, loader, optimizer, loss_fn, scaler, device, epoch):
#     model.train()
#     running_loss = 0.0
#     loop = tqdm(loader, desc=f"Training Epoch {epoch}")
    
#     try:
#         for batch_idx, (images, masks) in enumerate(loop):
#             # if batch_idx % 20 == 0:
#             #     gpu_mem = get_gpu_memory_info()
#             #     if gpu_mem:
#             #         loop.set_postfix(
#             #             loss=running_loss/(batch_idx+1) if batch_idx > 0 else 0,
#             #             gpu_alloc=f"{gpu_mem['allocated']:.2f}GB",
#             #             gpu_free=f"{gpu_mem['free']:.2f}GB"
#             #         )
            
#             images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
#             masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
            
#             with torch.amp.autocast('cuda'):
#                 outputs = model(images)
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
                
#             optimizer.zero_grad(set_to_none=True) 
#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()
            
#             running_loss += loss.item()
#             loop.set_postfix(loss=loss.item())
            
#             del images, masks, outputs, loss
            
#             if batch_idx % 10 == 0:
#                 torch.cuda.empty_cache()
        
#         return running_loss / len(loader)
        
#     except Exception as e:
#         print(f"\n❌ ERROR in training loop at batch {batch_idx}:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status(f"ERROR at batch {batch_idx}")
#         raise

# def validate(model, loader, loss_fn, device, num_classes):
#     model.eval()
#     running_loss = 0.0
#     all_ious = [[] for _ in range(num_classes)]
    
#     try:
#         with torch.no_grad():
#             for batch_idx, (images, masks) in enumerate(tqdm(loader, desc="Validating")):
#                 images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
#                 masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
                
#                 outputs = model(images)
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
#                 running_loss += loss.item()
                
#                 preds = torch.argmax(outputs, dim=1)
#                 batch_ious = calculate_iou(preds, masks, num_classes)
                
#                 for i, iou in enumerate(batch_ious):
#                     if not np.isnan(iou):
#                         all_ious[i].append(iou)
                
#                 del images, masks, outputs, preds
                        
#         mean_ious = [np.nanmean(ious) if ious else 0.0 for ious in all_ious]
#         mean_iou_total = np.nanmean([iou for iou in mean_ious if not np.isnan(iou)])
#         val_loss = running_loss / len(loader)
        
#         print(f"\nValidation Loss: {val_loss:.4f}")
#         print(f"Mean IoU (mIoU): {mean_iou_total:.4f}")
        
#         for i, class_name in enumerate(CLASS_NAMES): 
#             print(f"  - {class_name}: {mean_ious[i]:.4f}")
            
#         return val_loss, mean_iou_total
        
#     except Exception as e:
#         print(f"\n❌ ERROR in validation loop:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR in validation")
#         raise

# # ===================================================================
# # PHẦN 5: MAIN TRAINING (THAY ĐỔI: LOSS)
# # ===================================================================
# def run_training():
#     print("\n" + "="*70)
#     print("🚀 BẮT ĐẦU HUẤN LUYỆN - GIẢI PHÁP 2: U-NET + LOSS XỊN")
#     print("="*70)
    
#     try:
#         # --- 1. Transforms ---
#         print("\n📋 Khởi tạo transforms...")
#         train_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#             A.HorizontalFlip(p=0.5),
#             A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
#             A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
        
#         val_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
#         print("✅ Transforms created")
        
#         # --- 2. Datasets ---
#         print("\n📦 Loading datasets...")
#         # print_memory_status("BEFORE LOADING DATASETS")
        
#         train_dataset = CocoDataset(
#             TRAIN_IMG_DIR, TRAIN_JSON, 
#             transform=train_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # print_memory_status("AFTER LOADING TRAIN DATASET")
        
#         val_dataset = CocoDataset(
#             VALID_IMG_DIR, VALID_JSON, 
#             transform=val_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # # print_memory_status("AFTER LOADING VAL DATASET")

#         if train_dataset.num_classes != NUM_CLASSES or val_dataset.num_classes != NUM_CLASSES:
#              print(f"❌ LỖI NGHIÊM TRỌNG: Cấu hình NUM_CLASSES ({NUM_CLASSES}) không khớp với dataset ({train_dataset.num_classes})")
#              raise ValueError("Class count mismatch")

#         # --- 3. DataLoaders ---
#         print("\n🔄 Creating DataLoaders...")
#         train_loader = DataLoader(
#             train_dataset, batch_size=BATCH_SIZE, shuffle=True,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         val_loader = DataLoader(
#             val_dataset, batch_size=BATCH_SIZE, shuffle=False,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         print("✅ DataLoaders created")
#         print_memory_status("AFTER CREATING DATALOADERS")
        
#         # --- 4. Model ---
#         print("\n🏗️ Creating model...")
#         model = UNET(in_c=3, out_c=NUM_CLASSES).to(DEVICE)
        
#         if torch.cuda.device_count() > 1:
#             print(f"✅ Using {torch.cuda.device_count()} GPUs with DataParallel")
#             model = nn.DataParallel(model)
#         else:
#             print("✅ Using single GPU")
            
#         # print_memory_status("AFTER CREATING MODEL")
        
#         # --- 5. Loss, Optimizer, Scheduler (THAY ĐỔI) ---
#         print("\n⚙️ Setting up training components...")
#         loss_fn = CombinedLoss(dice_weight=0.6, focal_weight=0.4).to(DEVICE) # <-- ĐÃ THAY
#         optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        
#         scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#             optimizer, mode='min', factor=0.1, patience=5, verbose=True
#         )
#         scaler = torch.amp.GradScaler('cuda')
#         early_stopping = EarlyStopping(patience=10, delta=0.0001, path=CHECKPOINT_PATH)
#         print("✅ Training components ready")
        
#         # --- 6. Test forward pass ---
#         print("\n🧪 Testing forward pass with dummy batch...")
#         try:
#             dummy_input = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH).to(DEVICE)
#             with torch.no_grad():
#                 dummy_output = model(dummy_input)
#                 if isinstance(dummy_output, dict):
#                     dummy_output = dummy_output['out']
            
#             print(f"✅ Forward pass successful! Output shape: {dummy_output.shape}")
#             if dummy_output.shape != (2, NUM_CLASSES, IMAGE_HEIGHT, IMAGE_WIDTH):
#                  print(f"❌ LỖI SHAPE: Output shape {dummy_output.shape} không khớp mong đợi")
#                  raise ValueError("Output shape mismatch")
#             del dummy_input, dummy_output
#             torch.cuda.empty_cache()
#             # print_memory_status("AFTER DUMMY FORWARD PASS")
#         except Exception as e:
#             print(f"❌ Forward pass failed: {e}")
#             traceback.print_exc()
#             return
        
#         # --- 7. Training Loop ---
#         print("\n" + "="*70)
#         print("🎯 STARTING TRAINING LOOP")
#         print("="*70)
        
#         best_iou = -1.0
        
#         for epoch in range(NUM_EPOCHS):
#             print(f"\n{'='*70}")
#             print(f"📅 Epoch {epoch+1}/{NUM_EPOCHS}")
#             print(f"{'='*70}")
#             # print_memory_status(f"START of Epoch {epoch+1}")
            
#             epoch_start = time.time()
            
#             train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, DEVICE, epoch+1)
#             # print_memory_status(f"AFTER TRAINING Epoch {epoch+1}")
            
#             val_loss, val_iou = validate(model, val_loader, loss_fn, DEVICE, NUM_CLASSES)
#             # print_memory_status(f"AFTER VALIDATION Epoch {epoch+1}")
            
#             epoch_time = time.time() - epoch_start
            
#             print(f"\n📊 Tóm tắt Epoch {epoch+1}:")
#             print(f"   ⏱️  Time: {epoch_time:.2f}s")
#             print(f"   📉 Train Loss: {train_loss:.4f}")
#             print(f"   📉 Val Loss: {val_loss:.4f}")
#             print(f"   📈 Val mIoU: {val_iou:.4f}")
            
#             scheduler.step(val_loss)
            
#             if val_iou > best_iou:
#                 best_iou = val_iou
#                 print(f"✨ New best mIoU: {best_iou:.4f}")
                
#             early_stopping(val_loss, model)
#             if early_stopping.early_stop:
#                 print("\n⏹️ Early stopping triggered")
#                 break
                
#             torch.cuda.empty_cache()
#             gc.collect()
                    
#         print("\n" + "="*70)
#         print("🎉 TRAINING COMPLETED")
#         print("="*70)
#         print(f"✅ Best model saved at: {CHECKPOINT_PATH}")
#         print(f"🏆 Best mIoU: {best_iou:.4f}")
#         print_memory_status("FINAL")
        
#     except Exception as e:
#         traceback.print_exc()
#         print_memory_status("ERROR STATE")
#         raise

# # ===================================================================
# # PHẦN 6: INFERENCE (Giữ nguyên)
# # ===================================================================

# def preprocess_image_test(image_path, height, width):
#     image = cv2.imread(image_path)
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     original_size = image.shape[:2]
    
#     transform = A.Compose([
#         A.Resize(height, width, interpolation=cv2.INTER_LINEAR),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     transformed = transform(image=image_rgb)
#     tensor = transformed['image'].unsqueeze(0).to(DEVICE)
#     return tensor, image, original_size

# def mask_to_color_img(mask):
#     h, w = mask.shape
#     color_mask = np.zeros((h, w, 3), dtype=np.uint8)
#     for class_id, color in COLOR_MAP.items():
#         color_mask[mask == class_id] = color
#     return color_mask

# def overlay_mask_on_image(image, mask, alpha=0.5):
#     return cv2.addWeighted(image, 1 - alpha, mask, alpha, 0)

# def run_inference():
#     print(f"\n{'='*70}")
#     print("🔍 STARTING INFERENCE - GIẢI PHÁP 2")
#     print(f"{'='*70}")
    
#     try:
#         print("\n🏗️ Loading model...")
#         model = UNET(in_c=3, out_c=NUM_CLASSES)
        
#         try:
#             model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#         except RuntimeError:
#             print("⚠️ Model was trained with DataParallel, removing 'module.' prefix...")
#             from collections import OrderedDict
#             state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
#             new_state_dict = OrderedDict()
#             for k, v in state_dict.items():
#                 name = k.replace('module.', '')
#                 new_state_dict[name] = v
#             model.load_state_dict(new_state_dict)

#         model.to(DEVICE)
#         model.eval()
#         print("✅ Model loaded successfully")

#         image_files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
#         print(f"\n📁 Found {len(image_files)} test images")
        
#         if len(image_files) == 0:
#             print("⚠️ No test images found!")
#             return
        
#         for img_file in tqdm(image_files, desc="Processing"):
#             img_path = os.path.join(TEST_REAL_DIR, img_file)
            
#             tensor, original_bgr_img, original_size = preprocess_image_test(
#                 img_path, IMAGE_HEIGHT, IMAGE_WIDTH
#             )
            
#             with torch.no_grad():
#                 output = model(tensor)
#                 if isinstance(output, dict):
#                     output = output['out']
                
#                 prediction_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
                
#             prediction_resized = cv2.resize(
#                 prediction_mask.astype(np.uint8),
#                 (original_size[1], original_size[0]),
#                 interpolation=cv2.INTER_NEAREST
#             )
            
#             color_mask = mask_to_color_img(prediction_resized)
#             overlay_img = overlay_mask_on_image(original_bgr_img, color_mask, alpha=0.6)
            
#             base_name = os.path.splitext(img_file)[0]
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_mask.png"), color_mask)
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_overlay.png"), overlay_img)
            
#         print(f"\n✅ Inference completed! Results saved to: {PREDICTION_OUTPUT_DIR}")
        
#     except Exception as e:
#         print(f"\n❌ ERROR in inference:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         raise

# # ===================================================================
# # MAIN
# # ===================================================================
# if __name__ == "__main__":
#     print("\n" + "="*70)
#     print("🚀 STARTING PROGRAM - GIẢI PHÁP 2: U-NET + LOSS XỊN")
#     print("="*70)
    
#     try:
#         if DEVICE == "cuda":
#             torch.cuda.empty_cache()
#             gc.collect()
            
#         # print_memory_status("INITIAL")
        
#         # Training
#         run_training()
        
#         # Inference
#         if os.path.exists(CHECKPOINT_PATH):
#             run_inference()
#         else:
#             print(f"⚠️ Checkpoint not found at {CHECKPOINT_PATH}")
            
#     except KeyboardInterrupt:
#         print("\n⚠️ Training interrupted by user")
#         # print_memory_status("INTERRUPTED")
#     except Exception as e:
#         print(f"\n❌ FATAL ERROR:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         # print_memory_status("FATAL ERROR")

**Giải pháp 3**

In [9]:
# # ===================================================================
# # GIẢI PHÁP 3: ATTENTION U-NET + LOSS XỊN (DICE + FOCAL)
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import cv2
# import numpy as np
# import json
# from PIL import Image
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import matplotlib.pyplot as plt
# import gc
# import psutil
# import time
# import traceback

# # ===================================================================
# # PHẦN 0: HÀM MONITORING & DEBUG (Giữ nguyên)
# # ===================================================================

# def get_gpu_memory_info():
#     """Lấy thông tin GPU memory"""
#     if torch.cuda.is_available():
#         allocated = torch.cuda.memory_allocated() / 1024**3  # GB
#         reserved = torch.cuda.memory_reserved() / 1024**3   # GB
#         max_allocated = torch.cuda.max_memory_allocated() / 1024**3  # GB
#         total = torch.cuda.get_device_properties(0).total_memory / 1024**3  # GB
#         return {
#             'allocated': allocated,
#             'reserved': reserved,
#             'max_allocated': max_allocated,
#             'total': total,
#             'free': total - allocated
#         }
#     return None

# def get_cpu_memory_info():
#     """Lấy thông tin CPU memory"""
#     mem = psutil.virtual_memory()
#     return {
#         'total': mem.total / 1024**3,  # GB
#         'available': mem.available / 1024**3,  # GB
#         'used': mem.used / 1024**3,  # GB
#         'percent': mem.percent
#     }

# def print_memory_status(stage=""):
#     """In chi tiết memory status"""
#     print(f"\n{'='*70}")
#     print(f"🔍 MEMORY STATUS - {stage}")
#     print(f"{'='*70}")
    
#     # CPU Memory
#     cpu_mem = get_cpu_memory_info()
#     print(f"💻 CPU Memory:")
#     print(f"    Total: {cpu_mem['total']:.2f} GB")
#     print(f"    Used: {cpu_mem['used']:.2f} GB ({cpu_mem['percent']:.1f}%)")
#     print(f"    Available: {cpu_mem['available']:.2f} GB")
    
#     # GPU Memory
#     if torch.cuda.is_available():
#         for i in range(torch.cuda.device_count()):
#             gpu_mem = get_gpu_memory_info()
#             print(f"\n🎮 GPU {i} Memory:")
#             print(f"    Total: {gpu_mem['total']:.2f} GB")
#             print(f"    Allocated: {gpu_mem['allocated']:.2f} GB")
#             print(f"    Reserved: {gpu_mem['reserved']:.2f} GB")
#             print(f"    Free: {gpu_mem['free']:.2f} GB")
#             print(f"    Max Allocated: {gpu_mem['max_allocated']:.2f} GB")
#     print(f"{'='*70}\n")

# # ===================================================================
# # PHẦN 1: CẤU HÌNH
# # ===================================================================
# print("--- PHẦN 1: Bắt đầu cấu hình ---")

# # --- Đường dẫn ---
# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# # --- Checkpoint & Outputs (THAY ĐỔI) ---
# CHECKPOINT_PATH = "/kaggle/working/gp3_attnunet_dicefocal_best.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/gp3_predictions/"
# LOG_FILE = "/kaggle/working/training_log_gp3.txt"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# # --- Hyperparameters ---
# LEARNING_RATE = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 8     # Giảm BATCH_SIZE để gỡ lỗi
# NUM_EPOCHS = 100
# NUM_WORKERS = 0    # Set = 0 để gỡ lỗi.
# IMAGE_HEIGHT = 256
# IMAGE_WIDTH = 512
# PIN_MEMORY = True

# # --- Cấu hình Class (Giữ nguyên) ---
# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  # 3 class + 1 background = 4
# CLASS_NAMES = ['background'] + sorted(list(ALLOWED_CLASSES))
# # ['background', 'le_duong', 'phan_cach_nguoc_chieu', 'phan_lan']

# COLOR_MAP = {
#     0: [0, 0, 0],         # background
#     1: [255, 0, 255],     # le_duong
#     2: [0, 0, 255],       # phan_cach_nguoc_chieu
#     3: [0, 255, 0],       # phan_lan
# }

# # --- Cấu hình Memory (Giữ nguyên) ---
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.deterministic = False

# print(f"Device: {DEVICE}")
# print(f"Batch Size: {BATCH_SIZE}")
# print(f"Image Size: {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
# print(f"Num Workers: {NUM_WORKERS}")
# print(f"Total Classes (đã lọc): {NUM_CLASSES}")
# print(f"Class names (đã lọc): {CLASS_NAMES}")

# # print_memory_status("AFTER CONFIG")

# # ===================================================================
# # PHẦN 2: LỚP DATASET (Giữ nguyên code vá lỗi)
# # ===================================================================
# class CocoDataset(Dataset):
#     """Dataset class với error handling và lọc class"""
    
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         print(f"\n📂 Loading dataset from: {annotation_file}")
#         self.img_dir = img_dir
#         self.transform = transform
#         self.allowed_classes_set = allowed_classes if allowed_classes else None
        
#         try:
#             self.coco = COCO(annotation_file)
#             self.img_ids = list(sorted(self.coco.imgs.keys()))
            
#             all_cats_dict = self.coco.loadCats(self.coco.getCatIds())
            
#             if self.allowed_classes_set:
#                 print(f"Filtering for: {self.allowed_classes_set}")
#                 cats_filtered = [
#                     cat for cat in all_cats_dict 
#                     if cat['name'] in self.allowed_classes_set
#                 ]
#             else:
#                 print("No filter, loading all classes.")
#                 cats_filtered = all_cats_dict

#             cats_filtered.sort(key=lambda x: x['name'])
            
#             self.cat_id_to_class = {
#                 cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)
#             }
#             self.class_names = ['background'] + [cat['name'] for cat in cats_filtered]
#             self.num_classes = len(self.class_names)
            
#             print(f"✅ Successfully loaded {len(self.img_ids)} images")
#             print(f"✅ Found and filtered to {len(cats_filtered)} classes: {self.class_names[1:]}")
#             print(f"Total classes (inc. background): {self.num_classes}")
            
#         except Exception as e:
#             print(f"❌ ERROR loading dataset: {e}")
#             traceback.print_exc()
#             raise

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
            
#             image = np.array(Image.open(img_path).convert("RGB"))
#             h, w = image.shape[:2]
            
#             mask = np.zeros((h, w), dtype=np.uint8)
            
#             ann_ids = self.coco.getAnnIds(imgIds=img_id)
#             anns = self.coco.loadAnns(ann_ids)
            
#             for ann in anns:
#                 cat_id = ann['category_id']
#                 class_idx = self.cat_id_to_class.get(cat_id, 0) 
                
#                 if class_idx > 0:
#                     if 'segmentation' in ann and isinstance(ann['segmentation'], list):
#                         for seg in ann['segmentation']:
#                             poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                             cv2.fillPoly(mask, [poly], color=int(class_idx))
#                     elif 'segmentation' in ann and isinstance(ann['segmentation'], dict):
#                         rle = ann['segmentation']
#                         binary_mask = coco_mask.decode(rle)
#                         mask[binary_mask > 0] = int(class_idx)
            
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image = augmented['image']
#                 mask = augmented['mask']
                
#             return image, mask.long()
            
#         except Exception as e:
#             print(f"❌ ERROR loading sample {idx} (Image: {img_info.get('file_name', 'N/A')}): {e}")
#             traceback.print_exc()
#             return torch.randn(3, IMAGE_HEIGHT, IMAGE_WIDTH), \
#                    torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # ===================================================================
# # PHẦN 3: MODEL (THAY ĐỔI: ATTENTION U-NET)
# # ===================================================================
# print("\nKhởi tạo Model: ATTENTION U-NET (Giải pháp 3)")

# class DoubleConv(nn.Module):
#     """(Giữ nguyên như U-Net cũ)"""
#     def __init__(self, in_c, out_c):
#         super().__init__()
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False), 
#             nn.BatchNorm2d(out_c), 
#             nn.ReLU(inplace=True), 
#             nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False), 
#             nn.BatchNorm2d(out_c), 
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x): 
#         return self.conv(x)

# class AttentionBlock(nn.Module):
#     """Khối Attention Gate (MỚI)"""
#     def __init__(self, F_g, F_l, F_int):
#         super().__init__()
#         self.W_g = nn.Sequential(
#             nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(F_int)
#         )
#         self.W_x = nn.Sequential(
#             nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(F_int)
#         )
#         self.psi = nn.Sequential(
#             nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(1),
#             nn.Sigmoid()
#         )
#         self.relu = nn.ReLU(inplace=True)
        
#     def forward(self, g, x):
#         g1 = self.W_g(g)
#         x1 = self.W_x(x)
#         psi = self.relu(g1 + x1)
#         psi = self.psi(psi)
#         return x * psi # Nhân đầu ra (skip connection) với attention map

# class AttentionUNet(nn.Module):
#     """Kiến trúc U-Net với Attention Gates (MỚI)"""
#     def __init__(self, in_c=3, out_c=4, features=[64, 128, 256, 512]):
#         super().__init__()
#         self.downs = nn.ModuleList()
#         self.ups = nn.ModuleList()
#         self.attentions = nn.ModuleList() # (MỚI)
#         self.pool = nn.MaxPool2d(2, 2)
        
#         # Encoder (Giống U-Net)
#         for f in features:
#             self.downs.append(DoubleConv(in_c, f))
#             in_c = f
            
#         # Bottleneck (Giống U-Net)
#         self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        
#         # Decoder với Attention
#         for f in reversed(features):
#             self.ups.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
#             self.attentions.append(AttentionBlock(F_g=f, F_l=f, F_int=f // 2)) # (MỚI)
#             self.ups.append(DoubleConv(f * 2, f))
            
#         self.final_conv = nn.Conv2d(features[0], out_c, kernel_size=1)

#     def forward(self, x):
#         skip_connections = []
        
#         # Encoder
#         for down in self.downs:
#             x = down(x)
#             skip_connections.append(x)
#             x = self.pool(x)
            
#         x = self.bottleneck(x)
#         skip_connections = skip_connections[::-1]
        
#         # Decoder
#         for idx in range(0, len(self.ups), 2):
#             x = self.ups[idx](x)  # Upsample
#             skip = skip_connections[idx // 2]
            
#             # (MỚI) Cho skip connection đi qua Attention Gate
#             attention_idx = idx // 2
#             skip = self.attentions[attention_idx](g=x, x=skip)
            
#             # Resize (Giống U-Net)
#             if x.shape != skip.shape:
#                 x = TF.resize(x, size=skip.shape[2:])
                
#             # Concatenate và conv (Giống U-Net)
#             x = torch.cat([skip, x], dim=1)
#             x = self.ups[idx + 1](x)
            
#         return self.final_conv(x)

# # ===================================================================
# # PHẦN 3_BIS: LOSS FUNCTIONS (THAY ĐỔI: DICE + FOCAL)
# # ===================================================================
# print("\nKhởi tạo các class Loss Function (Dice, Focal)...")

# class DiceLoss(nn.Module):
#     """Dice Loss - Tốt cho vấn đề imbalanced classes và ranh giới"""
#     def __init__(self, smooth=1.0):
#         super().__init__()
#         self.smooth = smooth
        
#     def forward(self, pred, target):
#         pred = F.softmax(pred, dim=1)
#         target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 3, 1, 2).float()
        
#         # Bắt đầu từ class 1 (bỏ class 0 - background)
#         intersection = (pred[:, 1:] * target_one_hot[:, 1:]).sum(dim=(2, 3))
#         union = pred[:, 1:].sum(dim=(2, 3)) + target_one_hot[:, 1:].sum(dim=(2, 3))
        
#         dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
#         return 1.0 - dice.mean()

# class FocalLoss(nn.Module):
#     """Focal Loss - Tập trung vào các pixel khó phân loại (như ranh giới)"""
#     def __init__(self, alpha=0.25, gamma=2.0):
#         super().__init__()
#         self.alpha = alpha
#         self.gamma = gamma
        
#     def forward(self, pred, target):
#         ce_loss = F.cross_entropy(pred, target, reduction='none')
#         pt = torch.exp(-ce_loss) # probabilities của class đúng
#         focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
#         return focal_loss.mean()

# class CombinedLoss(nn.Module):
#     """Kết hợp Dice + Focal Loss"""
#     def __init__(self, dice_weight=0.6, focal_weight=0.4):
#         super().__init__()
#         self.dice = DiceLoss()
#         self.focal = FocalLoss()
#         self.dice_weight = dice_weight
#         self.focal_weight = focal_weight
#         print(f"Khởi tạo CombinedLoss (Dice: {dice_weight*100}%, Focal: {focal_weight*100}%)")

#     def forward(self, pred, target):
#         return self.dice_weight * self.dice(pred, target) + self.focal_weight * self.focal(pred, target)

# # ===================================================================
# # PHẦN 4: TRAINING FUNCTIONS (Giữ nguyên)
# # ===================================================================

# class EarlyStopping:
#     def __init__(self, patience=10, delta=0.0001, path='checkpoint.pth', trace_func=print):
#         self.patience = patience
#         self.delta = delta
#         self.path = path
#         self.trace_func = trace_func
#         self.counter = 0
#         self.best_score = None
#         self.early_stop = False
#         self.best_loss = float('inf')

#     def __call__(self, val_loss, model):
#         score = -val_loss
#         if self.best_score is None:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#         elif score < self.best_score + self.delta:
#             self.counter += 1
#             self.trace_func(f'EarlyStopping counter: {self.counter}/{self.patience}')
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#             self.counter = 0

#     def save_checkpoint(self, val_loss, model):
#         self.trace_func(f'✅ Validation loss giảm ({self.best_loss:.4f} → {val_loss:.4f}). Đang lưu model...')
#         if isinstance(model, nn.DataParallel):
#             torch.save(model.module.state_dict(), self.path)
#         else:
#             torch.save(model.state_dict(), self.path)
#         self.best_loss = val_loss

# def calculate_iou(pred, target, num_classes):
#     ious = []
#     pred = pred.view(-1)
#     target = target.view(-1)
    
#     for cls_id in range(num_classes):
#         pred_cls = (pred == cls_id)
#         target_cls = (target == cls_id)
        
#         intersection = (pred_cls & target_cls).sum().float()
#         union = (pred_cls | target_cls).sum().float()
        
#         if union == 0:
#             ious.append(float('nan'))
#         else:
#             ious.append((intersection / union).item())
#     return ious

# def train_one_epoch(model, loader, optimizer, loss_fn, scaler, device, epoch):
#     model.train()
#     running_loss = 0.0
#     loop = tqdm(loader, desc=f"Training Epoch {epoch}")
    
#     try:
#         for batch_idx, (images, masks) in enumerate(loop):
#             # if batch_idx % 20 == 0:
#             #     gpu_mem = get_gpu_memory_info()
#             #     if gpu_mem:
#             #         loop.set_postfix(
#             #             loss=running_loss/(batch_idx+1) if batch_idx > 0 else 0,
#             #             gpu_alloc=f"{gpu_mem['allocated']:.2f}GB",
#             #             gpu_free=f"{gpu_mem['free']:.2f}GB"
#             #         )
            
#             images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
#             masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
            
#             with torch.amp.autocast('cuda'):
#                 outputs = model(images)
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
                
#             optimizer.zero_grad(set_to_none=True) 
#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()
            
#             running_loss += loss.item()
#             loop.set_postfix(loss=loss.item())
            
#             del images, masks, outputs, loss
            
#             if batch_idx % 10 == 0:
#                 torch.cuda.empty_cache()
        
#         return running_loss / len(loader)
        
#     except Exception as e:
#         print(f"\n❌ ERROR in training loop at batch {batch_idx}:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status(f"ERROR at batch {batch_idx}")
#         raise

# def validate(model, loader, loss_fn, device, num_classes):
#     model.eval()
#     running_loss = 0.0
#     all_ious = [[] for _ in range(num_classes)]
    
#     try:
#         with torch.no_grad():
#             for batch_idx, (images, masks) in enumerate(tqdm(loader, desc="Validating")):
#                 images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
#                 masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
                
#                 outputs = model(images)
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
#                 running_loss += loss.item()
                
#                 preds = torch.argmax(outputs, dim=1)
#                 batch_ious = calculate_iou(preds, masks, num_classes)
                
#                 for i, iou in enumerate(batch_ious):
#                     if not np.isnan(iou):
#                         all_ious[i].append(iou)
                
#                 del images, masks, outputs, preds
                        
#         mean_ious = [np.nanmean(ious) if ious else 0.0 for ious in all_ious]
#         mean_iou_total = np.nanmean([iou for iou in mean_ious if not np.isnan(iou)])
#         val_loss = running_loss / len(loader)
        
#         print(f"\nValidation Loss: {val_loss:.4f}")
#         print(f"Mean IoU (mIoU): {mean_iou_total:.4f}")
        
#         for i, class_name in enumerate(CLASS_NAMES): 
#             print(f"  - {class_name}: {mean_ious[i]:.4f}")
            
#         return val_loss, mean_iou_total
        
#     except Exception as e:
#         print(f"\n❌ ERROR in validation loop:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR in validation")
#         raise

# # ===================================================================
# # PHẦN 5: MAIN TRAINING (THAY ĐỔI: MODEL, LOSS)
# # ===================================================================
# def run_training():
#     print("\n" + "="*70)
#     print("🚀 BẮT ĐẦU HUẤN LUYỆN - GIẢI PHÁP 3: ATTENTION U-NET")
#     print("="*70)
    
#     try:
#         # --- 1. Transforms ---
#         print("\n📋 Khởi tạo transforms...")
#         train_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#             A.HorizontalFlip(p=0.5),
#             A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
#             A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
        
#         val_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
#         print("✅ Transforms created")
        
#         # --- 2. Datasets ---
#         print("\n📦 Loading datasets...")
#         # print_memory_status("BEFORE LOADING DATASETS")
        
#         train_dataset = CocoDataset(
#             TRAIN_IMG_DIR, TRAIN_JSON, 
#             transform=train_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # print_memory_status("AFTER LOADING TRAIN DATASET")
        
#         val_dataset = CocoDataset(
#             VALID_IMG_DIR, VALID_JSON, 
#             transform=val_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # # print_memory_status("AFTER LOADING VAL DATASET")

#         if train_dataset.num_classes != NUM_CLASSES or val_dataset.num_classes != NUM_CLASSES:
#              print(f"❌ LỖI NGHIÊM TRỌNG: Cấu hình NUM_CLASSES ({NUM_CLASSES}) không khớp với dataset ({train_dataset.num_classes})")
#              raise ValueError("Class count mismatch")

#         # --- 3. DataLoaders ---
#         print("\n🔄 Creating DataLoaders...")
#         train_loader = DataLoader(
#             train_dataset, batch_size=BATCH_SIZE, shuffle=True,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         val_loader = DataLoader(
#             val_dataset, batch_size=BATCH_SIZE, shuffle=False,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         print("✅ DataLoaders created")
#         print_memory_status("AFTER CREATING DATALOADERS")
        
#         # --- 4. Model (THAY ĐỔI) ---
#         print("\n🏗️ Creating model...")
#         model = AttentionUNet(in_c=3, out_c=NUM_CLASSES).to(DEVICE) # <-- ĐÃ THAY
        
#         if torch.cuda.device_count() > 1:
#             print(f"✅ Using {torch.cuda.device_count()} GPUs with DataParallel")
#             model = nn.DataParallel(model)
#         else:
#             print("✅ Using single GPU")
            
#         # print_memory_status("AFTER CREATING MODEL")
        
#         # --- 5. Loss, Optimizer, Scheduler (THAY ĐỔI) ---
#         print("\n⚙️ Setting up training components...")
#         loss_fn = CombinedLoss(dice_weight=0.6, focal_weight=0.4).to(DEVICE) # <-- ĐÃ THAY
#         optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        
#         scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#             optimizer, mode='min', factor=0.1, patience=5, verbose=True
#         )
#         scaler = torch.amp.GradScaler('cuda')
#         early_stopping = EarlyStopping(patience=10, delta=0.0001, path=CHECKPOINT_PATH)
#         print("✅ Training components ready")
        
#         # --- 6. Test forward pass ---
#         print("\n🧪 Testing forward pass with dummy batch...")
#         try:
#             dummy_input = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH).to(DEVICE)
#             with torch.no_grad():
#                 dummy_output = model(dummy_input)
#                 if isinstance(dummy_output, dict):
#                     dummy_output = dummy_output['out']
            
#             print(f"✅ Forward pass successful! Output shape: {dummy_output.shape}")
#             if dummy_output.shape != (2, NUM_CLASSES, IMAGE_HEIGHT, IMAGE_WIDTH):
#                  print(f"❌ LỖI SHAPE: Output shape {dummy_output.shape} không khớp mong đợi")
#                  raise ValueError("Output shape mismatch")
#             del dummy_input, dummy_output
#             torch.cuda.empty_cache()
#             # print_memory_status("AFTER DUMMY FORWARD PASS")
#         except Exception as e:
#             print(f"❌ Forward pass failed: {e}")
#             traceback.print_exc()
#             return
        
#         # --- 7. Training Loop ---
#         print("\n" + "="*70)
#         print("🎯 STARTING TRAINING LOOP")
#         print("="*70)
        
#         best_iou = -1.0
        
#         for epoch in range(NUM_EPOCHS):
#             print(f"\n{'='*70}")
#             print(f"📅 Epoch {epoch+1}/{NUM_EPOCHS}")
#             print(f"{'='*70}")
#             # print_memory_status(f"START of Epoch {epoch+1}")
            
#             epoch_start = time.time()
            
#             train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, DEVICE, epoch+1)
#             # print_memory_status(f"AFTER TRAINING Epoch {epoch+1}")
            
#             val_loss, val_iou = validate(model, val_loader, loss_fn, DEVICE, NUM_CLASSES)
#             # print_memory_status(f"AFTER VALIDATION Epoch {epoch+1}")
            
#             epoch_time = time.time() - epoch_start
            
#             print(f"\n📊 Tóm tắt Epoch {epoch+1}:")
#             print(f"   ⏱️  Time: {epoch_time:.2f}s")
#             print(f"   📉 Train Loss: {train_loss:.4f}")
#             print(f"   📉 Val Loss: {val_loss:.4f}")
#             print(f"   📈 Val mIoU: {val_iou:.4f}")
            
#             scheduler.step(val_loss)
            
#             if val_iou > best_iou:
#                 best_iou = val_iou
#                 print(f"✨ New best mIoU: {best_iou:.4f}")
                
#             early_stopping(val_loss, model)
#             if early_stopping.early_stop:
#                 print("\n⏹️ Early stopping triggered")
#                 break
                
#             torch.cuda.empty_cache()
#             gc.collect()
                    
#         print("\n" + "="*70)
#         print("🎉 TRAINING COMPLETED")
#         print("="*70)
#         print(f"✅ Best model saved at: {CHECKPOINT_PATH}")
#         print(f"🏆 Best mIoU: {best_iou:.4f}")
#         print_memory_status("FINAL")
        
#     except Exception as e:
#         print(f"\n{'='*70}")
#         print(f"❌ CRITICAL ERROR IN TRAINING")
#         print(f"{'='*70}")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR STATE")
#         raise

# # ===================================================================
# # PHẦN 6: INFERENCE (THAY ĐỔI: MODEL)
# # ===================================================================

# def preprocess_image_test(image_path, height, width):
#     image = cv2.imread(image_path)
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     original_size = image.shape[:2]
    
#     transform = A.Compose([
#         A.Resize(height, width, interpolation=cv2.INTER_LINEAR),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     transformed = transform(image=image_rgb)
#     tensor = transformed['image'].unsqueeze(0).to(DEVICE)
#     return tensor, image, original_size

# def mask_to_color_img(mask):
#     h, w = mask.shape
#     color_mask = np.zeros((h, w, 3), dtype=np.uint8)
#     for class_id, color in COLOR_MAP.items():
#         color_mask[mask == class_id] = color
#     return color_mask

# def overlay_mask_on_image(image, mask, alpha=0.5):
#     return cv2.addWeighted(image, 1 - alpha, mask, alpha, 0)

# def run_inference():
#     print(f"\n{'='*70}")
#     print("🔍 STARTING INFERENCE - GIẢI PHÁP 3")
#     print(f"{'='*70}")
    
#     try:
#         print("\n🏗️ Loading model...")
#         # (THAY ĐỔI) Khởi tạo đúng model
#         model = AttentionUNet(in_c=3, out_c=NUM_CLASSES)
        
#         try:
#             model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#         except RuntimeError:
#             print("⚠️ Model was trained with DataParallel, removing 'module.' prefix...")
#             from collections import OrderedDict
#             state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
#             new_state_dict = OrderedDict()
#             for k, v in state_dict.items():
#                 name = k.replace('module.', '')
#                 new_state_dict[name] = v
#             model.load_state_dict(new_state_dict)

#         model.to(DEVICE)
#         model.eval()
#         print("✅ Model loaded successfully")

#         image_files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
#         print(f"\n📁 Found {len(image_files)} test images")
        
#         if len(image_files) == 0:
#             print("⚠️ No test images found!")
#             return
        
#         for img_file in tqdm(image_files, desc="Processing"):
#             img_path = os.path.join(TEST_REAL_DIR, img_file)
            
#             tensor, original_bgr_img, original_size = preprocess_image_test(
#                 img_path, IMAGE_HEIGHT, IMAGE_WIDTH
#             )
            
#             with torch.no_grad():
#                 output = model(tensor)
#                 if isinstance(output, dict):
#                     output = output['out']
                
#                 prediction_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
                
#             prediction_resized = cv2.resize(
#                 prediction_mask.astype(np.uint8),
#                 (original_size[1], original_size[0]),
#                 interpolation=cv2.INTER_NEAREST
#             )
            
#             color_mask = mask_to_color_img(prediction_resized)
#             overlay_img = overlay_mask_on_image(original_bgr_img, color_mask, alpha=0.6)
            
#             base_name = os.path.splitext(img_file)[0]
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_mask.png"), color_mask)
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_overlay.png"), overlay_img)
            
#         print(f"\n✅ Inference completed! Results saved to: {PREDICTION_OUTPUT_DIR}")
        
#     except Exception as e:
#         print(f"\n❌ ERROR in inference:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         raise

# # ===================================================================
# # MAIN
# # ===================================================================
# if __name__ == "__main__":
#     print("\n" + "="*70)
#     print("🚀 STARTING PROGRAM - GIẢI PHÁP 3: ATTENTION U-NET")
#     print("="*70)
    
#     try:
#         if DEVICE == "cuda":
#             torch.cuda.empty_cache()
#             gc.collect()
            
#         # print_memory_status("INITIAL")
        
#         # Training
#         run_training()
        
#         # Inference
#         if os.path.exists(CHECKPOINT_PATH):
#             run_inference()
#         else:
#             print(f"⚠️ Checkpoint not found at {CHECKPOINT_PATH}")
            
#     except KeyboardInterrupt:
#         print("\n⚠️ Training interrupted by user")
#         # print_memory_status("INTERRUPTED")
#     except Exception as e:
#         print(f"\n❌ FATAL ERROR:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         # print_memory_status("FATAL ERROR")

**Giải pháp 4**

In [10]:
# # ===================================================================
# # GIẢI PHÁP 4: DEEPLABV3+ (SOTA) + LOSS TỔNG LỰC
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF

# # (MỚI) Thêm 3 imports cho DeepLabV3+
# import torchvision
# from torchvision.models.segmentation import deeplabv3_resnet50
# from torchvision.models.segmentation.deeplabv3 import DeepLabHead

# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import cv2
# import numpy as np
# import json
# from PIL import Image
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import matplotlib.pyplot as plt
# import gc
# import psutil
# import time
# import traceback

# # ===================================================================
# # PHẦN 0: HÀM MONITORING & DEBUG (Giữ nguyên)
# # ===================================================================

# def get_gpu_memory_info():
#     """Lấy thông tin GPU memory"""
#     if torch.cuda.is_available():
#         allocated = torch.cuda.memory_allocated() / 1024**3  # GB
#         reserved = torch.cuda.memory_reserved() / 1024**3   # GB
#         max_allocated = torch.cuda.max_memory_allocated() / 1024**3  # GB
#         total = torch.cuda.get_device_properties(0).total_memory / 1024**3  # GB
#         return {
#             'allocated': allocated,
#             'reserved': reserved,
#             'max_allocated': max_allocated,
#             'total': total,
#             'free': total - allocated
#         }
#     return None

# def get_cpu_memory_info():
#     """Lấy thông tin CPU memory"""
#     mem = psutil.virtual_memory()
#     return {
#         'total': mem.total / 1024**3,  # GB
#         'available': mem.available / 1024**3,  # GB
#         'used': mem.used / 1024**3,  # GB
#         'percent': mem.percent
#     }

# def print_memory_status(stage=""):
#     """In chi tiết memory status"""
#     print(f"\n{'='*70}")
#     print(f"🔍 MEMORY STATUS - {stage}")
#     print(f"{'='*70}")
    
#     # CPU Memory
#     cpu_mem = get_cpu_memory_info()
#     print(f"💻 CPU Memory:")
#     print(f"    Total: {cpu_mem['total']:.2f} GB")
#     print(f"    Used: {cpu_mem['used']:.2f} GB ({cpu_mem['percent']:.1f}%)")
#     print(f"    Available: {cpu_mem['available']:.2f} GB")
    
#     # GPU Memory
#     if torch.cuda.is_available():
#         for i in range(torch.cuda.device_count()):
#             gpu_mem = get_gpu_memory_info()
#             print(f"\n🎮 GPU {i} Memory:")
#             print(f"    Total: {gpu_mem['total']:.2f} GB")
#             print(f"    Allocated: {gpu_mem['allocated']:.2f} GB")
#             print(f"    Reserved: {gpu_mem['reserved']:.2f} GB")
#             print(f"    Free: {gpu_mem['free']:.2f} GB")
#             print(f"    Max Allocated: {gpu_mem['max_allocated']:.2f} GB")
#     print(f"{'='*70}\n")

# # ===================================================================
# # PHẦN 1: CẤU HÌNH
# # ===================================================================
# print("--- PHẦN 1: Bắt đầu cấu hình ---")

# # --- Đường dẫn ---
# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# # --- Checkpoint & Outputs (THAY ĐỔI) ---
# CHECKPOINT_PATH = "/kaggle/working/gp4_deeplab_best.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/gp4_predictions/"
# LOG_FILE = "/kaggle/working/training_log_gp4.txt"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# # --- Hyperparameters (THAY ĐỔI) ---
# LEARNING_RATE = 5e-5  # (DeepLab nên dùng LR thấp hơn)
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 8     # Giảm BATCH_SIZE (DeepLab rất nặng)
# NUM_EPOCHS = 100
# NUM_WORKERS = 0    # (Giữ = 0 để tránh crash)
# IMAGE_HEIGHT = 256
# IMAGE_WIDTH = 512
# PIN_MEMORY = True

# # --- Cấu hình Class (Giữ nguyên) ---
# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  # 3 class + 1 background = 4
# CLASS_NAMES = ['background'] + sorted(list(ALLOWED_CLASSES))
# # ['background', 'le_duong', 'phan_cach_nguoc_chieu', 'phan_lan']

# COLOR_MAP = {
#     0: [0, 0, 0],         # background
#     1: [255, 0, 255],     # le_duong
#     2: [0, 0, 255],       # phan_cach_nguoc_chieu
#     3: [0, 255, 0],       # phan_lan
# }

# # --- Cấu hình Memory (Giữ nguyên) ---
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.deterministic = False

# print(f"Device: {DEVICE}")
# print(f"Batch Size: {BATCH_SIZE}")
# print(f"Image Size: {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
# print(f"Num Workers: {NUM_WORKERS}")
# print(f"Total Classes (đã lọc): {NUM_CLASSES}")
# print(f"Class names (đã lọc): {CLASS_NAMES}")
# print(f"LEARNING_RATE: {LEARNING_RATE}")

# # print_memory_status("AFTER CONFIG")

# # ===================================================================
# # PHẦN 2: LỚP DATASET (Giữ nguyên code vá lỗi)
# # ===================================================================
# class CocoDataset(Dataset):
#     """Dataset class với error handling và lọc class"""
    
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         print(f"\n📂 Loading dataset from: {annotation_file}")
#         self.img_dir = img_dir
#         self.transform = transform
#         self.allowed_classes_set = allowed_classes if allowed_classes else None
        
#         try:
#             self.coco = COCO(annotation_file)
#             self.img_ids = list(sorted(self.coco.imgs.keys()))
            
#             all_cats_dict = self.coco.loadCats(self.coco.getCatIds())
            
#             if self.allowed_classes_set:
#                 print(f"Filtering for: {self.allowed_classes_set}")
#                 cats_filtered = [
#                     cat for cat in all_cats_dict 
#                     if cat['name'] in self.allowed_classes_set
#                 ]
#             else:
#                 print("No filter, loading all classes.")
#                 cats_filtered = all_cats_dict

#             cats_filtered.sort(key=lambda x: x['name'])
            
#             self.cat_id_to_class = {
#                 cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)
#             }
#             self.class_names = ['background'] + [cat['name'] for cat in cats_filtered]
#             self.num_classes = len(self.class_names)
            
#             print(f"✅ Successfully loaded {len(self.img_ids)} images")
#             print(f"✅ Found and filtered to {len(cats_filtered)} classes: {self.class_names[1:]}")
#             print(f"Total classes (inc. background): {self.num_classes}")
            
#         except Exception as e:
#             print(f"❌ ERROR loading dataset: {e}")
#             traceback.print_exc()
#             raise

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
            
#             image = np.array(Image.open(img_path).convert("RGB"))
#             h, w = image.shape[:2]
            
#             mask = np.zeros((h, w), dtype=np.uint8)
            
#             ann_ids = self.coco.getAnnIds(imgIds=img_id)
#             anns = self.coco.loadAnns(ann_ids)
            
#             for ann in anns:
#                 cat_id = ann['category_id']
#                 class_idx = self.cat_id_to_class.get(cat_id, 0) 
                
#                 if class_idx > 0:
#                     if 'segmentation' in ann and isinstance(ann['segmentation'], list):
#                         for seg in ann['segmentation']:
#                             poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                             cv2.fillPoly(mask, [poly], color=int(class_idx))
#                     elif 'segmentation' in ann and isinstance(ann['segmentation'], dict):
#                         rle = ann['segmentation']
#                         binary_mask = coco_mask.decode(rle)
#                         mask[binary_mask > 0] = int(class_idx)
            
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image = augmented['image']
#                 mask = augmented['mask']
                
#             return image, mask.long()
            
#         except Exception as e:
#             print(f"❌ ERROR loading sample {idx} (Image: {img_info.get('file_name', 'N/A')}): {e}")
#             traceback.print_exc()
#             return torch.randn(3, IMAGE_HEIGHT, IMAGE_WIDTH), \
#                    torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # ===================================================================
# # PHẦN 3: MODEL (THAY ĐỔI: DEEPLABV3+)
# # ===================================================================
# print("\nKhởi tạo Model: DEEPLABV3+ (ResNet50 Pretrained) (Giải pháp 4)")

# def create_deeplabv3_model(num_classes, pretrained=True):
#     """
#     Tạo DeepLabV3+ model với ResNet50 backbone
#     Pretrained trên ImageNet để có feature extraction tốt hơn
#     """
#     print(f"Đang tải DeepLabV3+ (Pretrained={pretrained})...")
#     model = deeplabv3_resnet50(pretrained=pretrained, progress=True)
    
#     # Thay đổi classifier head cho số class của chúng ta (NUM_CLASSES = 4)
#     model.classifier = DeepLabHead(2048, num_classes)
    
#     print("✅ DeepLabV3+ sẵn sàng.")
#     return model

# # ===================================================================
# # PHẦN 3_BIS: LOSS FUNCTIONS (THAY ĐỔI: COMBO 3 MÓN)
# # ===================================================================
# print("\nKhởi tạo các class Loss Function (Dice, Focal)...")

# class DiceLoss(nn.Module):
#     def __init__(self, smooth=1.0):
#         super().__init__()
#         self.smooth = smooth
#     def forward(self, pred, target):
#         pred = F.softmax(pred, dim=1)
#         target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 3, 1, 2).float()
#         intersection = (pred[:, 1:] * target_one_hot[:, 1:]).sum(dim=(2, 3))
#         union = pred[:, 1:].sum(dim=(2, 3)) + target_one_hot[:, 1:].sum(dim=(2, 3))
#         dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
#         return 1.0 - dice.mean()

# class FocalLoss(nn.Module):
#     def __init__(self, alpha=0.25, gamma=2.0):
#         super().__init__()
#         self.alpha = alpha
#         self.gamma = gamma
#     def forward(self, pred, target):
#         ce_loss = F.cross_entropy(pred, target, reduction='none')
#         pt = torch.exp(-ce_loss)
#         focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
#         return focal_loss.mean()

# class CombinedLoss(nn.Module):
#     """ (MỚI) Kết hợp CE + Dice + Focal """
#     def __init__(self, ce_weight=0.3, dice_weight=0.5, focal_weight=0.2):
#         super().__init__()
#         self.ce = nn.CrossEntropyLoss()
#         self.dice = DiceLoss()
#         self.focal = FocalLoss()
#         self.ce_weight = ce_weight
#         self.dice_weight = dice_weight
#         self.focal_weight = focal_weight
#         print(f"Khởi tạo CombinedLoss (CE: {ce_weight*100}%, Dice: {dice_weight*100}%, Focal: {focal_weight*100}%)")

#     def forward(self, pred, target):
#         return (self.ce_weight * self.ce(pred, target) + 
#                 self.dice_weight * self.dice(pred, target) + 
#                 self.focal_weight * self.focal(pred, target))

# # ===================================================================
# # PHẦN 4: TRAINING FUNCTIONS (Giữ nguyên)
# # ===================================================================

# class EarlyStopping:
#     def __init__(self, patience=10, delta=0.0001, path='checkpoint.pth', trace_func=print):
#         self.patience = patience
#         self.delta = delta
#         self.path = path
#         self.trace_func = trace_func
#         self.counter = 0
#         self.best_score = None
#         self.early_stop = False
#         self.best_loss = float('inf')

#     def __call__(self, val_loss, model):
#         score = -val_loss
#         if self.best_score is None:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#         elif score < self.best_score + self.delta:
#             self.counter += 1
#             self.trace_func(f'EarlyStopping counter: {self.counter}/{self.patience}')
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#             self.counter = 0

#     def save_checkpoint(self, val_loss, model):
#         self.trace_func(f'✅ Validation loss giảm ({self.best_loss:.4f} → {val_loss:.4f}). Đang lưu model...')
#         if isinstance(model, nn.DataParallel):
#             torch.save(model.module.state_dict(), self.path)
#         else:
#             torch.save(model.state_dict(), self.path)
#         self.best_loss = val_loss

# def calculate_iou(pred, target, num_classes):
#     ious = []
#     pred = pred.view(-1)
#     target = target.view(-1)
    
#     for cls_id in range(num_classes):
#         pred_cls = (pred == cls_id)
#         target_cls = (target == cls_id)
        
#         intersection = (pred_cls & target_cls).sum().float()
#         union = (pred_cls | target_cls).sum().float()
        
#         if union == 0:
#             ious.append(float('nan'))
#         else:
#             ious.append((intersection / union).item())
#     return ious

# def train_one_epoch(model, loader, optimizer, loss_fn, scaler, device, epoch):
#     model.train()
#     running_loss = 0.0
#     loop = tqdm(loader, desc=f"Training Epoch {epoch}")
    
#     try:
#         for batch_idx, (images, masks) in enumerate(loop):
#             # if batch_idx % 20 == 0:
#             #     gpu_mem = get_gpu_memory_info()
#             #     if gpu_mem:
#             #         loop.set_postfix(
#             #             loss=running_loss/(batch_idx+1) if batch_idx > 0 else 0,
#             #             gpu_alloc=f"{gpu_mem['allocated']:.2f}GB",
#             #             gpu_free=f"{gpu_mem['free']:.2f}GB"
#             #         )
            
#             images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
#             masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
            
#             with torch.amp.autocast('cuda'):
#                 outputs = model(images)
#                 # (MỚI) DeepLab trả về dict, ta cần lấy 'out'
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
                
#             optimizer.zero_grad(set_to_none=True) 
#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()
            
#             running_loss += loss.item()
#             loop.set_postfix(loss=loss.item())
            
#             del images, masks, outputs, loss
            
#             if batch_idx % 10 == 0:
#                 torch.cuda.empty_cache()
        
#         return running_loss / len(loader)
        
#     except Exception as e:
#         print(f"\n❌ ERROR in training loop at batch {batch_idx}:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status(f"ERROR at batch {batch_idx}")
#         raise

# def validate(model, loader, loss_fn, device, num_classes):
#     model.eval()
#     running_loss = 0.0
#     all_ious = [[] for _ in range(num_classes)]
    
#     try:
#         with torch.no_grad():
#             for batch_idx, (images, masks) in enumerate(tqdm(loader, desc="Validating")):
#                 images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
#                 masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
                
#                 outputs = model(images)
#                 # (MỚI) DeepLab trả về dict
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
#                 running_loss += loss.item()
                
#                 preds = torch.argmax(outputs, dim=1)
#                 batch_ious = calculate_iou(preds, masks, num_classes)
                
#                 for i, iou in enumerate(batch_ious):
#                     if not np.isnan(iou):
#                         all_ious[i].append(iou)
                
#                 del images, masks, outputs, preds
                        
#         mean_ious = [np.nanmean(ious) if ious else 0.0 for ious in all_ious]
#         mean_iou_total = np.nanmean([iou for iou in mean_ious if not np.isnan(iou)])
#         val_loss = running_loss / len(loader)
        
#         print(f"\nValidation Loss: {val_loss:.4f}")
#         print(f"Mean IoU (mIoU): {mean_iou_total:.4f}")
        
#         for i, class_name in enumerate(CLASS_NAMES): 
#             print(f"  - {class_name}: {mean_ious[i]:.4f}")
            
#         return val_loss, mean_iou_total
        
#     except Exception as e:
#         print(f"\n❌ ERROR in validation loop:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR in validation")
#         raise

# # ===================================================================
# # PHẦN 5: MAIN TRAINING (THAY ĐỔI: MODEL, LOSS, OPTIMIZER)
# # ===================================================================
# def run_training():
#     print("\n" + "="*70)
#     print("🚀 BẮT ĐẦU HUẤN LUYỆN - GIẢI PHÁP 4: DEEPLABV3+")
#     print("="*70)
    
#     try:
#         # --- 1. Transforms ---
#         print("\n📋 Khởi tạo transforms...")
#         # (Giữ nguyên Augmentation như giải pháp 1)
#         train_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#             A.HorizontalFlip(p=0.5),
#             A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
#             A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
        
#         val_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
#         print("✅ Transforms created")
        
#         # --- 2. Datasets ---
#         print("\n📦 Loading datasets...")
#         # print_memory_status("BEFORE LOADING DATASETS")
        
#         train_dataset = CocoDataset(
#             TRAIN_IMG_DIR, TRAIN_JSON, 
#             transform=train_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # print_memory_status("AFTER LOADING TRAIN DATASET")
        
#         val_dataset = CocoDataset(
#             VALID_IMG_DIR, VALID_JSON, 
#             transform=val_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # # print_memory_status("AFTER LOADING VAL DATASET")

#         if train_dataset.num_classes != NUM_CLASSES or val_dataset.num_classes != NUM_CLASSES:
#              print(f"❌ LỖI NGHIÊM TRỌNG: Cấu hình NUM_CLASSES ({NUM_CLASSES}) không khớp với dataset ({train_dataset.num_classes})")
#              raise ValueError("Class count mismatch")

#         # --- 3. DataLoaders ---
#         print("\n🔄 Creating DataLoaders...")
#         train_loader = DataLoader(
#             train_dataset, batch_size=BATCH_SIZE, shuffle=True,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         val_loader = DataLoader(
#             val_dataset, batch_size=BATCH_SIZE, shuffle=False,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         print("✅ DataLoaders created")
#         print_memory_status("AFTER CREATING DATALOADERS")
        
#         # --- 4. Model (THAY ĐỔI) ---
#         print("\n🏗️ Creating model...")
#         model = create_deeplabv3_model(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE) # <-- ĐÃ THAY
        
#         if torch.cuda.device_count() > 1:
#             print(f"✅ Using {torch.cuda.device_count()} GPUs with DataParallel")
#             model = nn.DataParallel(model)
#         else:
#             print("✅ Using single GPU")
            
#         # print_memory_status("AFTER CREATING MODEL")
        
#         # --- 5. Loss, Optimizer, Scheduler (THAY ĐỔI) ---
#         print("\n⚙️ Setting up training components...")
#         loss_fn = CombinedLoss(ce_weight=0.3, dice_weight=0.5, focal_weight=0.2).to(DEVICE) # <-- ĐÃ THAY LOSS
        
#         print("Setting up optimizer with differential learning rates...")
#         # Tách riêng backbone (LR thấp) và classifier (LR cao)
#         model_params = [
#             {'params': (model.module.backbone.parameters() if isinstance(model, nn.DataParallel) 
#                         else model.backbone.parameters()), 'lr': LEARNING_RATE * 0.1},
            
#             {'params': (model.module.classifier.parameters() if isinstance(model, nn.DataParallel) 
#                         else model.classifier.parameters()), 'lr': LEARNING_RATE},
#         ]
#         optimizer = optim.Adam(model_params, lr=LEARNING_RATE) # <-- ĐÃ THAY OPTIMIZER
        
#         scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#             optimizer, mode='min', factor=0.1, patience=5, verbose=True
#         )
#         scaler = torch.amp.GradScaler('cuda')
#         early_stopping = EarlyStopping(patience=10, delta=0.0001, path=CHECKPOINT_PATH)
#         print("✅ Training components ready")
        
#         # --- 6. Test forward pass ---
#         print("\n🧪 Testing forward pass with dummy batch...")
#         try:
#             dummy_input = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH).to(DEVICE)
#             with torch.no_grad():
#                 dummy_output = model(dummy_input)
#                 if isinstance(dummy_output, dict): # (MỚI)
#                     dummy_output = dummy_output['out']
            
#             print(f"✅ Forward pass successful! Output shape: {dummy_output.shape}")
#             if dummy_output.shape != (2, NUM_CLASSES, IMAGE_HEIGHT, IMAGE_WIDTH):
#                  print(f"❌ LỖI SHAPE: Output shape {dummy_output.shape} không khớp mong đợi")
#                  raise ValueError("Output shape mismatch")
#             del dummy_input, dummy_output
#             torch.cuda.empty_cache()
#             # print_memory_status("AFTER DUMMY FORWARD PASS")
#         except Exception as e:
#             print(f"❌ Forward pass failed: {e}")
#             traceback.print_exc()
#             return
        
#         # --- 7. Training Loop ---
#         print("\n" + "="*70)
#         print("🎯 STARTING TRAINING LOOP")
#         print("="*70)
        
#         best_iou = -1.0
        
#         for epoch in range(NUM_EPOCHS):
#             print(f"\n{'='*70}")
#             print(f"📅 Epoch {epoch+1}/{NUM_EPOCHS}")
#             print(f"{'='*70}")
#             # print_memory_status(f"START of Epoch {epoch+1}")
            
#             epoch_start = time.time()
            
#             train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, DEVICE, epoch+1)
#             # print_memory_status(f"AFTER TRAINING Epoch {epoch+1}")
            
#             val_loss, val_iou = validate(model, val_loader, loss_fn, DEVICE, NUM_CLASSES)
#             # print_memory_status(f"AFTER VALIDATION Epoch {epoch+1}")
            
#             epoch_time = time.time() - epoch_start
            
#             print(f"\n📊 Tóm tắt Epoch {epoch+1}:")
#             print(f"   ⏱️  Time: {epoch_time:.2f}s")
#             print(f"   📉 Train Loss: {train_loss:.4f}")
#             print(f"   📉 Val Loss: {val_loss:.4f}")
#             print(f"   📈 Val mIoU: {val_iou:.4f}")
            
#             scheduler.step(val_loss)
            
#             if val_iou > best_iou:
#                 best_iou = val_iou
#                 print(f"✨ New best mIoU: {best_iou:.4f}")
                
#             early_stopping(val_loss, model)
#             if early_stopping.early_stop:
#                 print("\n⏹️ Early stopping triggered")
#                 break
                
#             torch.cuda.empty_cache()
#             gc.collect()
                    
#         print("\n" + "="*70)
#         print("🎉 TRAINING COMPLETED")
#         print("="*70)
#         print(f"✅ Best model saved at: {CHECKPOINT_PATH}")
#         print(f"🏆 Best mIoU: {best_iou:.4f}")
#         print_memory_status("FINAL")
        
#     except Exception as e:
#         print(f"\n{'='*70}")
#         print(f"❌ CRITICAL ERROR IN TRAINING")
#         print(f"{'='*70}")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR STATE")
#         raise

# # ===================================================================
# # PHẦN 6: INFERENCE (THAY ĐỔI: MODEL)
# # ===================================================================

# def preprocess_image_test(image_path, height, width):
#     image = cv2.imread(image_path)
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     original_size = image.shape[:2]
    
#     transform = A.Compose([
#         A.Resize(height, width, interpolation=cv2.INTER_LINEAR),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     transformed = transform(image=image_rgb)
#     tensor = transformed['image'].unsqueeze(0).to(DEVICE)
#     return tensor, image, original_size

# def mask_to_color_img(mask):
#     h, w = mask.shape
#     color_mask = np.zeros((h, w, 3), dtype=np.uint8)
#     for class_id, color in COLOR_MAP.items():
#         color_mask[mask == class_id] = color
#     return color_mask

# def overlay_mask_on_image(image, mask, alpha=0.5):
#     return cv2.addWeighted(image, 1 - alpha, mask, alpha, 0)

# def run_inference():
#     print(f"\n{'='*70}")
#     print("🔍 STARTING INFERENCE - GIẢI PHÁP 4")
#     print(f"{'='*70}")
    
#     try:
#         print("\n🏗️ Loading model...")
#         # (THAY ĐỔI) Khởi tạo đúng model, pretrained=False vì ta load checkpoint
#         model = create_deeplabv3_model(num_classes=NUM_CLASSES, pretrained=False)
        
#         try:
#             model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#         except RuntimeError:
#             print("⚠️ Model was trained with DataParallel, removing 'module.' prefix...")
#             from collections import OrderedDict
#             state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
#             new_state_dict = OrderedDict()
#             for k, v in state_dict.items():
#                 name = k.replace('module.', '')
#                 new_state_dict[name] = v
#             model.load_state_dict(new_state_dict)

#         model.to(DEVICE)
#         model.eval()
#         print("✅ Model loaded successfully")

#         image_files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
#         print(f"\n📁 Found {len(image_files)} test images")
        
#         if len(image_files) == 0:
#             print("⚠️ No test images found!")
#             return
        
#         for img_file in tqdm(image_files, desc="Processing"):
#             img_path = os.path.join(TEST_REAL_DIR, img_file)
            
#             tensor, original_bgr_img, original_size = preprocess_image_test(
#                 img_path, IMAGE_HEIGHT, IMAGE_WIDTH
#             )
            
#             with torch.no_grad():
#                 output = model(tensor)
#                 if isinstance(output, dict): # (MỚI)
#                     output = output['out']
                
#                 prediction_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
                
#             prediction_resized = cv2.resize(
#                 prediction_mask.astype(np.uint8),
#                 (original_size[1], original_size[0]),
#                 interpolation=cv2.INTER_NEAREST
#             )
            
#             color_mask = mask_to_color_img(prediction_resized)
#             overlay_img = overlay_mask_on_image(original_bgr_img, color_mask, alpha=0.6)
            
#             base_name = os.path.splitext(img_file)[0]
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_mask.png"), color_mask)
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_overlay.png"), overlay_img)
            
#         print(f"\n✅ Inference completed! Results saved to: {PREDICTION_OUTPUT_DIR}")
        
#     except Exception as e:
#         print(f"\n❌ ERROR in inference:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         raise

# # ===================================================================
# # MAIN
# # ===================================================================
# if __name__ == "__main__":
#     print("\n" + "="*70)
#     print("🚀 STARTING PROGRAM - GIẢI PHÁP 4: DEEPLABV3+")
#     print("="*70)
    
#     try:
#         if DEVICE == "cuda":
#             torch.cuda.empty_cache()
#             gc.collect()
            
#         # print_memory_status("INITIAL")
        
#         # Training
#         run_training()
        
#         # Inference
#         if os.path.exists(CHECKPOINT_PATH):
#             run_inference()
#         else:
#             print(f"⚠️ Checkpoint not found at {CHECKPOINT_PATH}")
            
#     except KeyboardInterrupt:
#         print("\n⚠️ Training interrupted by user")
#         # print_memory_status("INTERRUPTED")
#     except Exception as e:
#         print(f"\n❌ FATAL ERROR:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         # print_memory_status("FATAL ERROR")

**Giải pháp 5**

In [11]:
# # ===================================================================
# # GIẢI PHÁP 5: ATTENTION U-NET + LOSS XỊN + ẢNH NÉT (384x640)
# # ===================================================================
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import cv2
# import numpy as np
# import json
# from PIL import Image
# from tqdm import tqdm
# from pycocotools.coco import COCO
# from pycocotools import mask as coco_mask
# import matplotlib.pyplot as plt
# import gc
# import psutil
# import time
# import traceback

# # ===================================================================
# # PHẦN 0: HÀM MONITORING & DEBUG (Giữ nguyên)
# # ===================================================================

# def get_gpu_memory_info():
#     """Lấy thông tin GPU memory"""
#     if torch.cuda.is_available():
#         allocated = torch.cuda.memory_allocated() / 1024**3  # GB
#         reserved = torch.cuda.memory_reserved() / 1024**3   # GB
#         max_allocated = torch.cuda.max_memory_allocated() / 1024**3  # GB
#         total = torch.cuda.get_device_properties(0).total_memory / 1024**3  # GB
#         return {
#             'allocated': allocated,
#             'reserved': reserved,
#             'max_allocated': max_allocated,
#             'total': total,
#             'free': total - allocated
#         }
#     return None

# def get_cpu_memory_info():
#     """Lấy thông tin CPU memory"""
#     mem = psutil.virtual_memory()
#     return {
#         'total': mem.total / 1024**3,  # GB
#         'available': mem.available / 1024**3,  # GB
#         'used': mem.used / 1024**3,  # GB
#         'percent': mem.percent
#     }

# def print_memory_status(stage=""):
#     """In chi tiết memory status"""
#     print(f"\n{'='*70}")
#     print(f"🔍 MEMORY STATUS - {stage}")
#     print(f"{'='*70}")
    
#     # CPU Memory
#     cpu_mem = get_cpu_memory_info()
#     print(f"💻 CPU Memory:")
#     print(f"    Total: {cpu_mem['total']:.2f} GB")
#     print(f"    Used: {cpu_mem['used']:.2f} GB ({cpu_mem['percent']:.1f}%)")
#     print(f"    Available: {cpu_mem['available']:.2f} GB")
    
#     # GPU Memory
#     if torch.cuda.is_available():
#         for i in range(torch.cuda.device_count()):
#             gpu_mem = get_gpu_memory_info()
#             print(f"\n🎮 GPU {i} Memory:")
#             print(f"    Total: {gpu_mem['total']:.2f} GB")
#             print(f"    Allocated: {gpu_mem['allocated']:.2f} GB")
#             print(f"    Reserved: {gpu_mem['reserved']:.2f} GB")
#             print(f"    Free: {gpu_mem['free']:.2f} GB")
#             print(f"    Max Allocated: {gpu_mem['max_allocated']:.2f} GB")
#     print(f"{'='*70}\n")

# # ===================================================================
# # PHẦN 1: CẤU HÌNH
# # ===================================================================
# print("--- PHẦN 1: Bắt đầu cấu hình ---")

# # --- Đường dẫn ---
# DATASET_ROOT = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, 'train')
# TRAIN_JSON = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
# VALID_IMG_DIR = os.path.join(DATASET_ROOT, 'valid')
# VALID_JSON = os.path.join(VALID_IMG_DIR, '_annotations.coco.json')
# TEST_REAL_DIR = os.path.join(DATASET_ROOT, 'test_seg_lane')

# # --- Checkpoint & Outputs (THAY ĐỔI) ---
# CHECKPOINT_PATH = "/kaggle/working/gp5_attnunet_hires_best.pth"
# PREDICTION_OUTPUT_DIR = "/kaggle/working/gp5_predictions_hires/"
# LOG_FILE = "/kaggle/working/training_log_gp5.txt"
# os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# # --- Hyperparameters ---
# LEARNING_RATE = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 4     # (THAY ĐỔI) Giảm BATCH_SIZE vì ảnh to hơn
# NUM_EPOCHS = 100
# NUM_WORKERS = 0    # (Giữ = 0 để tránh crash)
# IMAGE_HEIGHT = 384 # (THAY ĐỔI) Tăng độ phân giải
# IMAGE_WIDTH = 640  # (THAY ĐỔI) Tăng độ phân giải
# PIN_MEMORY = True

# # --- Cấu hình Class (Giữ nguyên) ---
# ALLOWED_CLASSES = {'phan_cach_nguoc_chieu', 'phan_lan', 'le_duong'}
# NUM_CLASSES = len(ALLOWED_CLASSES) + 1  # 3 class + 1 background = 4
# CLASS_NAMES = ['background'] + sorted(list(ALLOWED_CLASSES))
# # ['background', 'le_duong', 'phan_cach_nguoc_chieu', 'phan_lan']

# COLOR_MAP = {
#     0: [0, 0, 0],         # background
#     1: [255, 0, 255],     # le_duong
#     2: [0, 0, 255],       # phan_cach_nguoc_chieu
#     3: [0, 255, 0],       # phan_lan
# }

# # --- Cấu hình Memory (Giữ nguyên) ---
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.deterministic = False

# print(f"Device: {DEVICE}")
# print(f"Batch Size: {BATCH_SIZE}")
# print(f"Image Size: {IMAGE_HEIGHT}x{IMAGE_WIDTH} (ĐỘ PHÂN GIẢI CAO)")
# print(f"Num Workers: {NUM_WORKERS}")
# print(f"Total Classes (đã lọc): {NUM_CLASSES}")
# print(f"Class names (đã lọc): {CLASS_NAMES}")

# # print_memory_status("AFTER CONFIG")

# # ===================================================================
# # PHẦN 2: LỚP DATASET (Giữ nguyên code vá lỗi)
# # ===================================================================
# class CocoDataset(Dataset):
#     """Dataset class với error handling và lọc class"""
    
#     def __init__(self, img_dir, annotation_file, transform=None, allowed_classes=None):
#         print(f"\n📂 Loading dataset from: {annotation_file}")
#         self.img_dir = img_dir
#         self.transform = transform
#         self.allowed_classes_set = allowed_classes if allowed_classes else None
        
#         try:
#             self.coco = COCO(annotation_file)
#             self.img_ids = list(sorted(self.coco.imgs.keys()))
            
#             all_cats_dict = self.coco.loadCats(self.coco.getCatIds())
            
#             if self.allowed_classes_set:
#                 print(f"Filtering for: {self.allowed_classes_set}")
#                 cats_filtered = [
#                     cat for cat in all_cats_dict 
#                     if cat['name'] in self.allowed_classes_set
#                 ]
#             else:
#                 print("No filter, loading all classes.")
#                 cats_filtered = all_cats_dict

#             cats_filtered.sort(key=lambda x: x['name'])
            
#             self.cat_id_to_class = {
#                 cat['id']: idx for idx, cat in enumerate(cats_filtered, start=1)
#             }
#             self.class_names = ['background'] + [cat['name'] for cat in cats_filtered]
#             self.num_classes = len(self.class_names)
            
#             print(f"✅ Successfully loaded {len(self.img_ids)} images")
#             print(f"✅ Found and filtered to {len(cats_filtered)} classes: {self.class_names[1:]}")
#             print(f"Total classes (inc. background): {self.num_classes}")
            
#         except Exception as e:
#             print(f"❌ ERROR loading dataset: {e}")
#             traceback.print_exc()
#             raise

#     def __len__(self):
#         return len(self.img_ids)

#     def __getitem__(self, idx):
#         try:
#             img_id = self.img_ids[idx]
#             img_info = self.coco.loadImgs(img_id)[0]
#             img_path = os.path.join(self.img_dir, img_info['file_name'])
            
#             image = np.array(Image.open(img_path).convert("RGB"))
#             h, w = image.shape[:2]
            
#             mask = np.zeros((h, w), dtype=np.uint8)
            
#             ann_ids = self.coco.getAnnIds(imgIds=img_id)
#             anns = self.coco.loadAnns(ann_ids)
            
#             for ann in anns:
#                 cat_id = ann['category_id']
#                 class_idx = self.cat_id_to_class.get(cat_id, 0) 
                
#                 if class_idx > 0:
#                     if 'segmentation' in ann and isinstance(ann['segmentation'], list):
#                         for seg in ann['segmentation']:
#                             poly = np.array(seg).reshape(-1, 2).astype(np.int32)
#                             cv2.fillPoly(mask, [poly], color=int(class_idx))
#                     elif 'segmentation' in ann and isinstance(ann['segmentation'], dict):
#                         rle = ann['segmentation']
#                         binary_mask = coco_mask.decode(rle)
#                         mask[binary_mask > 0] = int(class_idx)
            
#             if self.transform:
#                 augmented = self.transform(image=image, mask=mask)
#                 image = augmented['image']
#                 mask = augmented['mask']
                
#             return image, mask.long()
            
#         except Exception as e:
#             print(f"❌ ERROR loading sample {idx} (Image: {img_info.get('file_name', 'N/A')}): {e}")
#             traceback.print_exc()
#             return torch.randn(3, IMAGE_HEIGHT, IMAGE_WIDTH), \
#                    torch.zeros(IMAGE_HEIGHT, IMAGE_WIDTH).long()

# # ===================================================================
# # PHẦN 3: MODEL (THAY ĐỔI: ATTENTION U-NET)
# # ===================================================================
# print("\nKhởi tạo Model: ATTENTION U-NET (Giải pháp 5)")

# class DoubleConv(nn.Module):
#     """(Giữ nguyên như U-Net cũ)"""
#     def __init__(self, in_c, out_c):
#         super().__init__()
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False), 
#             nn.BatchNorm2d(out_c), 
#             nn.ReLU(inplace=True), 
#             nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False), 
#             nn.BatchNorm2d(out_c), 
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x): 
#         return self.conv(x)

# class AttentionBlock(nn.Module):
#     """Khối Attention Gate (MỚI)"""
#     def __init__(self, F_g, F_l, F_int):
#         super().__init__()
#         self.W_g = nn.Sequential(
#             nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(F_int)
#         )
#         self.W_x = nn.Sequential(
#             nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(F_int)
#         )
#         self.psi = nn.Sequential(
#             nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(1),
#             nn.Sigmoid()
#         )
#         self.relu = nn.ReLU(inplace=True)
        
#     def forward(self, g, x):
#         g1 = self.W_g(g)
#         x1 = self.W_x(x)
#         psi = self.relu(g1 + x1)
#         psi = self.psi(psi)
#         return x * psi

# class AttentionUNet(nn.Module):
#     """Kiến trúc U-Net với Attention Gates (MỚI)"""
#     def __init__(self, in_c=3, out_c=4, features=[64, 128, 256, 512]):
#         super().__init__()
#         self.downs = nn.ModuleList()
#         self.ups = nn.ModuleList()
#         self.attentions = nn.ModuleList() # (MỚI)
#         self.pool = nn.MaxPool2d(2, 2)
        
#         # Encoder
#         for f in features:
#             self.downs.append(DoubleConv(in_c, f))
#             in_c = f
            
#         # Bottleneck
#         self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        
#         # Decoder với Attention
#         for f in reversed(features):
#             self.ups.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
#             self.attentions.append(AttentionBlock(F_g=f, F_l=f, F_int=f // 2)) # (MỚI)
#             self.ups.append(DoubleConv(f * 2, f))
            
#         self.final_conv = nn.Conv2d(features[0], out_c, kernel_size=1)

#     def forward(self, x):
#         skip_connections = []
        
#         # Encoder
#         for down in self.downs:
#             x = down(x)
#             skip_connections.append(x)
#             x = self.pool(x)
            
#         x = self.bottleneck(x)
#         skip_connections = skip_connections[::-1]
        
#         # Decoder
#         for idx in range(0, len(self.ups), 2):
#             x = self.ups[idx](x)  # Upsample
#             skip = skip_connections[idx // 2]
            
#             attention_idx = idx // 2
#             skip = self.attentions[attention_idx](g=x, x=skip)
            
#             if x.shape != skip.shape:
#                 x = TF.resize(x, size=skip.shape[2:])
                
#             x = torch.cat([skip, x], dim=1)
#             x = self.ups[idx + 1](x)
            
#         return self.final_conv(x)

# # ===================================================================
# # PHẦN 3_BIS: LOSS FUNCTIONS (THAY ĐỔI: DICE + FOCAL)
# # ===================================================================
# print("\nKhởi tạo các class Loss Function (Dice, Focal)...")

# class DiceLoss(nn.Module):
#     def __init__(self, smooth=1.0):
#         super().__init__()
#         self.smooth = smooth
#     def forward(self, pred, target):
#         pred = F.softmax(pred, dim=1)
#         target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 3, 1, 2).float()
#         intersection = (pred[:, 1:] * target_one_hot[:, 1:]).sum(dim=(2, 3))
#         union = pred[:, 1:].sum(dim=(2, 3)) + target_one_hot[:, 1:].sum(dim=(2, 3))
#         dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
#         return 1.0 - dice.mean()

# class FocalLoss(nn.Module):
#     def __init__(self, alpha=0.25, gamma=2.0):
#         super().__init__()
#         self.alpha = alpha
#         self.gamma = gamma
#     def forward(self, pred, target):
#         ce_loss = F.cross_entropy(pred, target, reduction='none')
#         pt = torch.exp(-ce_loss)
#         focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
#         return focal_loss.mean()

# class CombinedLoss(nn.Module):
#     def __init__(self, dice_weight=0.6, focal_weight=0.4):
#         super().__init__()
#         self.dice = DiceLoss()
#         self.focal = FocalLoss()
#         self.dice_weight = dice_weight
#         self.focal_weight = focal_weight
#         print(f"Khởi tạo CombinedLoss (Dice: {dice_weight*100}%, Focal: {focal_weight*100}%)")
#     def forward(self, pred, target):
#         return self.dice_weight * self.dice(pred, target) + self.focal_weight * self.focal(pred, target)

# # ===================================================================
# # PHẦN 4: TRAINING FUNCTIONS (Giữ nguyên)
# # ===================================================================

# class EarlyStopping:
#     def __init__(self, patience=10, delta=0.0001, path='checkpoint.pth', trace_func=print):
#         self.patience = patience
#         self.delta = delta
#         self.path = path
#         self.trace_func = trace_func
#         self.counter = 0
#         self.best_score = None
#         self.early_stop = False
#         self.best_loss = float('inf')

#     def __call__(self, val_loss, model):
#         score = -val_loss
#         if self.best_score is None:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#         elif score < self.best_score + self.delta:
#             self.counter += 1
#             self.trace_func(f'EarlyStopping counter: {self.counter}/{self.patience}')
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = score
#             self.save_checkpoint(val_loss, model)
#             self.counter = 0

#     def save_checkpoint(self, val_loss, model):
#         self.trace_func(f'✅ Validation loss giảm ({self.best_loss:.4f} → {val_loss:.4f}). Đang lưu model...')
#         if isinstance(model, nn.DataParallel):
#             torch.save(model.module.state_dict(), self.path)
#         else:
#             torch.save(model.state_dict(), self.path)
#         self.best_loss = val_loss

# def calculate_iou(pred, target, num_classes):
#     ious = []
#     pred = pred.view(-1)
#     target = target.view(-1)
    
#     for cls_id in range(num_classes):
#         pred_cls = (pred == cls_id)
#         target_cls = (target == cls_id)
        
#         intersection = (pred_cls & target_cls).sum().float()
#         union = (pred_cls | target_cls).sum().float()
        
#         if union == 0:
#             ious.append(float('nan'))
#         else:
#             ious.append((intersection / union).item())
#     return ious

# def train_one_epoch(model, loader, optimizer, loss_fn, scaler, device, epoch):
#     model.train()
#     running_loss = 0.0
#     loop = tqdm(loader, desc=f"Training Epoch {epoch}")
    
#     try:
#         for batch_idx, (images, masks) in enumerate(loop):
#             # if batch_idx % 20 == 0:
#             #     gpu_mem = get_gpu_memory_info()
#             #     if gpu_mem:
#             #         loop.set_postfix(
#             #             loss=running_loss/(batch_idx+1) if batch_idx > 0 else 0,
#             #             gpu_alloc=f"{gpu_mem['allocated']:.2f}GB",
#             #             gpu_free=f"{gpu_mem['free']:.2f}GB"
#             #         )
            
#             images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
#             masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
            
#             with torch.amp.autocast('cuda'):
#                 outputs = model(images)
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
                
#             optimizer.zero_grad(set_to_none=True) 
#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()
            
#             running_loss += loss.item()
#             loop.set_postfix(loss=loss.item())
            
#             del images, masks, outputs, loss
            
#             if batch_idx % 10 == 0:
#                 torch.cuda.empty_cache()
        
#         return running_loss / len(loader)
        
#     except Exception as e:
#         print(f"\n❌ ERROR in training loop at batch {batch_idx}:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status(f"ERROR at batch {batch_idx}")
#         raise

# def validate(model, loader, loss_fn, device, num_classes):
#     model.eval()
#     running_loss = 0.0
#     all_ious = [[] for _ in range(num_classes)]
    
#     try:
#         with torch.no_grad():
#             for batch_idx, (images, masks) in enumerate(tqdm(loader, desc="Validating")):
#                 images = images.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
#                 masks = masks.to(device, non_blocking=PIN_MEMORY if PIN_MEMORY else False)
                
#                 outputs = model(images)
#                 if isinstance(outputs, dict):
#                     outputs = outputs['out']
                
#                 loss = loss_fn(outputs, masks)
#                 running_loss += loss.item()
                
#                 preds = torch.argmax(outputs, dim=1)
#                 batch_ious = calculate_iou(preds, masks, num_classes)
                
#                 for i, iou in enumerate(batch_ious):
#                     if not np.isnan(iou):
#                         all_ious[i].append(iou)
                
#                 del images, masks, outputs, preds
                        
#         mean_ious = [np.nanmean(ious) if ious else 0.0 for ious in all_ious]
#         mean_iou_total = np.nanmean([iou for iou in mean_ious if not np.isnan(iou)])
#         val_loss = running_loss / len(loader)
        
#         print(f"\nValidation Loss: {val_loss:.4f}")
#         print(f"Mean IoU (mIoU): {mean_iou_total:.4f}")
        
#         for i, class_name in enumerate(CLASS_NAMES): 
#             print(f"  - {class_name}: {mean_ious[i]:.4f}")
            
#         return val_loss, mean_iou_total
        
#     except Exception as e:
#         print(f"\n❌ ERROR in validation loop:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR in validation")
#         raise

# # ===================================================================
# # PHẦN 5: MAIN TRAINING (THAY ĐỔI: MODEL, LOSS)
# # ===================================================================
# def run_training():
#     print("\n" + "="*70)
#     print("🚀 BẮT ĐẦU HUẤN LUYỆN - GIẢI PHÁP 5: ATTN U-NET (HI-RES)")
#     print("="*70)
    
#     try:
#         # --- 1. Transforms ---
#         print("\n📋 Khởi tạo transforms...")
#         train_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST), # (MỚI) 384x640
#             A.HorizontalFlip(p=0.5),
#             A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
#             A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
        
#         val_transform = A.Compose([
#             A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST), # (MỚI) 384x640
#             A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ToTensorV2(),
#         ])
#         print("✅ Transforms created")
        
#         # --- 2. Datasets ---
#         print("\n📦 Loading datasets...")
#         # print_memory_status("BEFORE LOADING DATASETS")
        
#         train_dataset = CocoDataset(
#             TRAIN_IMG_DIR, TRAIN_JSON, 
#             transform=train_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # print_memory_status("AFTER LOADING TRAIN DATASET")
        
#         val_dataset = CocoDataset(
#             VALID_IMG_DIR, VALID_JSON, 
#             transform=val_transform,
#             allowed_classes=ALLOWED_CLASSES
#         )
#         # # print_memory_status("AFTER LOADING VAL DATASET")

#         if train_dataset.num_classes != NUM_CLASSES or val_dataset.num_classes != NUM_CLASSES:
#              print(f"❌ LỖI NGHIÊM TRỌNG: Cấu hình NUM_CLASSES ({NUM_CLASSES}) không khớp với dataset ({train_dataset.num_classes})")
#              raise ValueError("Class count mismatch")

#         # --- 3. DataLoaders ---
#         print("\n🔄 Creating DataLoaders...")
#         train_loader = DataLoader(
#             train_dataset, batch_size=BATCH_SIZE, shuffle=True,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         val_loader = DataLoader(
#             val_dataset, batch_size=BATCH_SIZE, shuffle=False,
#             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
#             persistent_workers=True if NUM_WORKERS > 0 else False
#         )
#         print("✅ DataLoaders created")
#         print_memory_status("AFTER CREATING DATALOADERS")
        
#         # --- 4. Model (THAY ĐỔI) ---
#         print("\n🏗️ Creating model...")
#         model = AttentionUNet(in_c=3, out_c=NUM_CLASSES).to(DEVICE) # <-- ĐÃ THAY
        
#         if torch.cuda.device_count() > 1:
#             print(f"✅ Using {torch.cuda.device_count()} GPUs with DataParallel")
#             model = nn.DataParallel(model)
#         else:
#             print("✅ Using single GPU")
            
#         # print_memory_status("AFTER CREATING MODEL")
        
#         # --- 5. Loss, Optimizer, Scheduler (THAY ĐỔI) ---
#         print("\n⚙️ Setting up training components...")
#         loss_fn = CombinedLoss(dice_weight=0.6, focal_weight=0.4).to(DEVICE) # <-- ĐÃ THAY
#         optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        
#         scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#             optimizer, mode='min', factor=0.1, patience=5, verbose=True
#         )
#         scaler = torch.amp.GradScaler('cuda')
#         early_stopping = EarlyStopping(patience=10, delta=0.0001, path=CHECKPOINT_PATH)
#         print("✅ Training components ready")
        
#         # --- 6. Test forward pass ---
#         print("\n🧪 Testing forward pass with dummy batch...")
#         try:
#             dummy_input = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH).to(DEVICE)
#             with torch.no_grad():
#                 dummy_output = model(dummy_input)
#                 if isinstance(dummy_output, dict):
#                     dummy_output = dummy_output['out']
            
#             print(f"✅ Forward pass successful! Output shape: {dummy_output.shape}")
#             if dummy_output.shape != (2, NUM_CLASSES, IMAGE_HEIGHT, IMAGE_WIDTH):
#                  print(f"❌ LỖI SHAPE: Output shape {dummy_output.shape} không khớp mong đợi")
#                  raise ValueError("Output shape mismatch")
#             del dummy_input, dummy_output
#             torch.cuda.empty_cache()
#             # print_memory_status("AFTER DUMMY FORWARD PASS")
#         except Exception as e:
#             print(f"❌ Forward pass failed: {e}")
#             traceback.print_exc()
#             return
        
#         # --- 7. Training Loop ---
#         print("\n" + "="*70)
#         print("🎯 STARTING TRAINING LOOP")
#         print("="*70)
        
#         best_iou = -1.0
        
#         for epoch in range(NUM_EPOCHS):
#             print(f"\n{'='*70}")
#             print(f"📅 Epoch {epoch+1}/{NUM_EPOCHS}")
#             print(f"{'='*70}")
#             # print_memory_status(f"START of Epoch {epoch+1}")
            
#             epoch_start = time.time()
            
#             train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, DEVICE, epoch+1)
#             # print_memory_status(f"AFTER TRAINING Epoch {epoch+1}")
            
#             val_loss, val_iou = validate(model, val_loader, loss_fn, DEVICE, NUM_CLASSES)
#             # print_memory_status(f"AFTER VALIDATION Epoch {epoch+1}")
            
#             epoch_time = time.time() - epoch_start
            
#             print(f"\n📊 Tóm tắt Epoch {epoch+1}:")
#             print(f"   ⏱️  Time: {epoch_time:.2f}s")
#             print(f"   📉 Train Loss: {train_loss:.4f}")
#             print(f"   📉 Val Loss: {val_loss:.4f}")
#             print(f"   📈 Val mIoU: {val_iou:.4f}")
            
#             scheduler.step(val_loss)
            
#             if val_iou > best_iou:
#                 best_iou = val_iou
#                 print(f"✨ New best mIoU: {best_iou:.4f}")
                
#             early_stopping(val_loss, model)
#             if early_stopping.early_stop:
#                 print("\n⏹️ Early stopping triggered")
#                 break
                
#             torch.cuda.empty_cache()
#             gc.collect()
                    
#         print("\n" + "="*70)
#         print("🎉 TRAINING COMPLETED")
#         print("="*70)
#         print(f"✅ Best model saved at: {CHECKPOINT_PATH}")
#         print(f"🏆 Best mIoU: {best_iou:.4f}")
#         print_memory_status("FINAL")
        
#     except Exception as e:
#         print(f"\n{'='*70}")
#         print(f"❌ CRITICAL ERROR IN TRAINING")
#         print(f"{'='*70}")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         print_memory_status("ERROR STATE")
#         raise

# # ===================================================================
# # PHẦN 6: INFERENCE (THAY ĐỔI: MODEL)
# # ===================================================================

# def preprocess_image_test(image_path, height, width):
#     image = cv2.imread(image_path)
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     original_size = image.shape[:2]
    
#     transform = A.Compose([
#         A.Resize(height, width, interpolation=cv2.INTER_LINEAR),
#         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#         ToTensorV2(),
#     ])
    
#     transformed = transform(image=image_rgb)
#     tensor = transformed['image'].unsqueeze(0).to(DEVICE)
#     return tensor, image, original_size

# def mask_to_color_img(mask):
#     h, w = mask.shape
#     color_mask = np.zeros((h, w, 3), dtype=np.uint8)
#     for class_id, color in COLOR_MAP.items():
#         color_mask[mask == class_id] = color
#     return color_mask

# def overlay_mask_on_image(image, mask, alpha=0.5):
#     return cv2.addWeighted(image, 1 - alpha, mask, alpha, 0)

# def run_inference():
#     print(f"\n{'='*70}")
#     print("🔍 STARTING INFERENCE - GIẢI PHÁP 5")
#     print(f"{'='*70}")
    
#     try:
#         print("\n🏗️ Loading model...")
#         # (THAY ĐỔI) Khởi tạo đúng model
#         model = AttentionUNet(in_c=3, out_c=NUM_CLASSES)
        
#         try:
#             model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#         except RuntimeError:
#             print("⚠️ Model was trained with DataParallel, removing 'module.' prefix...")
#             from collections import OrderedDict
#             state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
#             new_state_dict = OrderedDict()
#             for k, v in state_dict.items():
#                 name = k.replace('module.', '')
#                 new_state_dict[name] = v
#             model.load_state_dict(new_state_dict)

#         model.to(DEVICE)
#         model.eval()
#         print("✅ Model loaded successfully")

#         image_files = [f for f in os.listdir(TEST_REAL_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
#         print(f"\n📁 Found {len(image_files)} test images")
        
#         if len(image_files) == 0:
#             print("⚠️ No test images found!")
#             return
        
#         for img_file in tqdm(image_files, desc="Processing"):
#             img_path = os.path.join(TEST_REAL_DIR, img_file)
            
#             tensor, original_bgr_img, original_size = preprocess_image_test(
#                 img_path, IMAGE_HEIGHT, IMAGE_WIDTH
#             )
            
#             with torch.no_grad():
#                 output = model(tensor)
#                 if isinstance(output, dict):
#                     output = output['out']
                
#                 prediction_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
                
#             prediction_resized = cv2.resize(
#                 prediction_mask.astype(np.uint8),
#                 (original_size[1], original_size[0]),
#                 interpolation=cv2.INTER_NEAREST
#             )
            
#             color_mask = mask_to_color_img(prediction_resized)
#             overlay_img = overlay_mask_on_image(original_bgr_img, color_mask, alpha=0.6)
            
#             base_name = os.path.splitext(img_file)[0]
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_mask.png"), color_mask)
#             cv2.imwrite(os.path.join(PREDICTION_OUTPUT_DIR, f"{base_name}_overlay.png"), overlay_img)
            
#         print(f"\n✅ Inference completed! Results saved to: {PREDICTION_OUTPUT_DIR}")
        
#     except Exception as e:
#         print(f"\n❌ ERROR in inference:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         raise

# # ===================================================================
# # MAIN
# # ===================================================================
# if __name__ == "__main__":
#     print("\n" + "="*70)
#     print("🚀 STARTING PROGRAM - GIẢI PHÁP 5: ATTN U-NET (HI-RES)")
#     print("="*70)
    
#     try:
#         if DEVICE == "cuda":
#             torch.cuda.empty_cache()
#             gc.collect()
            
#         # print_memory_status("INITIAL")
        
#         # Training
#         run_training()
        
#         # Inference
#         if os.path.exists(CHECKPOINT_PATH):
#             run_inference()
#         else:
#             print(f"⚠️ Checkpoint not found at {CHECKPOINT_PATH}")
            
#     except KeyboardInterrupt:
#         print("\n⚠️ Training interrupted by user")
#         # print_memory_status("INTERRUPTED")
#     except Exception as e:
#         print(f"\n❌ FATAL ERROR:")
#         print(f"Error: {e}")
#         traceback.print_exc()
#         # print_memory_status("FATAL ERROR")

In [12]:
# # ===================================================================
# # CELL DUY NHẤT: QUY TRÌNH HOÀN CHỈNH TỪ A ĐẾN Z (CÓ EARLY STOPPING)
# # ===================================================================

# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF
# from tqdm import tqdm
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import shutil
# import cv2
# import numpy as np
# import glob
# import json
# from PIL import Image
# from sklearn.model_selection import train_test_split

# # ===================================================================
# # PHẦN 0: CẤU HÌNH TOÀN BỘ QUY TRÌNH VÀ LỚP EARLY STOPPING
# # ===================================================================

# # --- Cấu hình đường dẫn ---
# KAGGLE_INPUT_DIR = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TEMP_DATA_DIR = '/kaggle/working/temp_processed_data'
# FINAL_DATASET_DIR = '/kaggle/working/final_lane_dataset'
# PREDICTION_OUTPUT_DIR = "/kaggle/working/saved_images/"
# CHECKPOINT_PATH = "/kaggle/working/unet_best_model.pth"

# # --- Cấu hình mô hình & huấn luyện ---
# LEARNING_RATE = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 16
# NUM_EPOCHS = 100 # Đặt số epoch lớn, Early Stopping sẽ tự dừng
# NUM_WORKERS = 0
# IMAGE_HEIGHT = 256
# IMAGE_WIDTH = 512
# PIN_MEMORY = True
# CLASS_MAP = {'phan_cach_nguoc_chieu': 1, 'phan_lan': 2, 'le_duong': 3}

# # LỚP EARLY STOPPING
# class EarlyStopping:
#     def __init__(self, patience=7, verbose=True, delta=0, path='checkpoint.pth', trace_func=print):
#         self.patience, self.verbose, self.delta, self.path = patience, verbose, delta, path
#         self.trace_func = trace_func
#         self.best_score, self.early_stop, self.counter = None, False, 0
#         self.best_dice_score = 0
#     def __call__(self, val_dice_score, model):
#         score = val_dice_score
#         if self.best_score is None:
#             self.best_score = score
#             self.save_checkpoint(val_dice_score, model)
#         elif score < self.best_score + self.delta:
#             self.counter += 1
#             self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = score
#             self.save_checkpoint(val_dice_score, model)
#             self.counter = 0
#     def save_checkpoint(self, val_dice_score, model):
#         if self.verbose:
#             self.trace_func(f'Dice score improved ({self.best_dice_score:.4f} --> {val_dice_score:.4f}). Saving model ...')
#         # Lưu state_dict của mô hình gốc, không phải của DataParallel wrapper
#         torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), self.path)
#         self.best_dice_score = val_dice_score

# # ===================================================================
# # PHẦN 1: TẠO DỮ LIỆU TỪ JSON (Lưu mask dưới dạng .png)
# # ===================================================================
# print("--- PHẦN 1: Bắt đầu tạo dữ liệu từ JSON ---")
# if os.path.exists(TEMP_DATA_DIR): shutil.rmtree(TEMP_DATA_DIR)
# TEMP_IMAGE_DIR = os.path.join(TEMP_DATA_DIR, 'images')
# TEMP_MASK_DIR = os.path.join(TEMP_DATA_DIR, 'masks')
# os.makedirs(TEMP_IMAGE_DIR, exist_ok=True); os.makedirs(TEMP_MASK_DIR, exist_ok=True)
# subfolders = [f.name for f in os.scandir(KAGGLE_INPUT_DIR) if f.is_dir()]
# processed_files_count = 0
# for folder_name in tqdm(subfolders, desc="Đang xử lý các thư mục"):
#     image_folder_path = os.path.join(KAGGLE_INPUT_DIR, folder_name)
#     json_path = os.path.join(KAGGLE_INPUT_DIR, f"labels_{folder_name}.json")
#     if not os.path.exists(json_path): continue
#     with open(json_path, 'r') as f: current_annotations = json.load(f)
#     for img_path in glob.glob(os.path.join(image_folder_path, '*.jpg')):
#         filename = os.path.basename(img_path)
#         if filename in current_annotations:
#             annotation = current_annotations[filename]
#             image = cv2.imread(img_path)
#             if image is None: continue
#             height, width, _ = image.shape
#             mask = np.zeros((height, width), dtype=np.uint8)
#             if 'regions' in annotation and annotation['regions']:
#                  for region in annotation['regions'].values():
#                     if 'region_attributes' in region and 'label' in region['region_attributes']:
#                         class_name = region['region_attributes']['label']
#                         if class_name in CLASS_MAP:
#                             points = np.array(list(zip(region['shape_attributes']['all_points_x'], region['shape_attributes']['all_points_y'])), dtype=np.int32)
#                             cv2.fillPoly(mask, [points], color=CLASS_MAP[class_name])
#             base_filename = os.path.splitext(filename)[0]
#             unique_name = f"{folder_name}_{base_filename}"
#             cv2.imwrite(os.path.join(TEMP_IMAGE_DIR, f"{unique_name}.jpg"), image)
#             cv2.imwrite(os.path.join(TEMP_MASK_DIR, f"{unique_name}.png"), mask)
#             processed_files_count += 1
# print(f"--- Hoàn tất PHẦN 1! Đã tạo {processed_files_count} cặp ảnh/mask ---\n")

# # ===================================================================
# # PHẦN 2: PHÂN CHIA VÀ TẠO DATASET CUỐI CÙNG
# # ===================================================================
# print("--- PHẦN 2: Bắt đầu phân chia dữ liệu ---")
# if os.path.exists(FINAL_DATASET_DIR): shutil.rmtree(FINAL_DATASET_DIR)
# image_paths = sorted(glob.glob(os.path.join(TEMP_IMAGE_DIR, '*.jpg')))
# mask_paths = sorted(glob.glob(os.path.join(TEMP_MASK_DIR, '*.png')))
# train_val_imgs, test_imgs, train_val_masks, test_masks = train_test_split(image_paths, mask_paths, test_size=0.15, random_state=32)
# train_imgs, val_imgs, train_masks, val_masks = train_test_split(train_val_imgs, train_val_masks, test_size=0.176, random_state=32)
# split_data = {'train': (train_imgs, train_masks), 'val': (val_imgs, val_masks), 'test': (test_imgs, test_masks)}
# print(f"Phân chia hoàn tất: Train={len(train_imgs)}, Val={len(val_imgs)}, Test={len(test_imgs)}")
# for split_name, (img_list, mask_list) in split_data.items():
#     os.makedirs(os.path.join(FINAL_DATASET_DIR, 'images', split_name), exist_ok=True)
#     os.makedirs(os.path.join(FINAL_DATASET_DIR, 'masks', split_name), exist_ok=True)
#     for i in tqdm(range(len(img_list)), desc=f'Sao chép tập {split_name}'):
#         shutil.copy(img_list[i], os.path.join(FINAL_DATASET_DIR, 'images', split_name))
#         shutil.copy(mask_list[i], os.path.join(FINAL_DATASET_DIR, 'masks', split_name))
# print(f"--- Hoàn tất PHẦN 2! Dữ liệu đã sẵn sàng tại: {FINAL_DATASET_DIR} ---\n")

# # ===================================================================
# # PHẦN 3: HUẤN LUYỆN MÔ HÌNH U-NET
# # ===================================================================
# print("--- PHẦN 3: Bắt đầu huấn luyện mô hình U-Net ---")

# # --- Lớp Dataset ---
# class LaneDataset(Dataset):
#     def __init__(self, image_dir, mask_dir, transform=None):
#         self.image_dir, self.mask_dir, self.transform = image_dir, mask_dir, transform
#         self.images = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
#     def __len__(self): return len(self.images)
#     def __getitem__(self, index):
#         img_filename = self.images[index]
#         img_path = os.path.join(self.image_dir, img_filename)
#         mask_path = os.path.join(self.mask_dir, os.path.splitext(img_filename)[0] + ".png")
#         image = np.array(Image.open(img_path).convert("RGB"))
#         mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
#         if self.transform:
#             augmentations = self.transform(image=image, mask=mask)
#             image, mask = augmentations["image"], augmentations["mask"]
#         return image, mask

# # --- Kiến trúc U-Net ---
# class DoubleConv(nn.Module):
#     def __init__(self, in_c, out_c):
#         super().__init__(); self.conv = nn.Sequential(nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
#     def forward(self, x): return self.conv(x)
# class UNET(nn.Module):
#     def __init__(self, in_c=3, out_c=4, features=[64, 128, 256, 512]):
#         super().__init__(); self.ups, self.downs, self.pool = nn.ModuleList(), nn.ModuleList(), nn.MaxPool2d(2, 2)
#         for f in features: self.downs.append(DoubleConv(in_c, f)); in_c = f
#         for f in reversed(features): self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2)); self.ups.append(DoubleConv(f*2, f))
#         self.bottleneck, self.final_conv = DoubleConv(features[-1], features[-1]*2), nn.Conv2d(features[0], out_c, 1)
#     def forward(self, x):
#         skips = [];
#         for down in self.downs: x = down(x); skips.append(x); x = self.pool(x)
#         x = self.bottleneck(x); skips = skips[::-1]
#         for i in range(0, len(self.ups), 2):
#             x = self.ups[i](x); skip = skips[i//2]
#             if x.shape != skip.shape: x = TF.resize(x, size=skip.shape[2:])
#             x = self.ups[i+1](torch.cat((skip, x), 1))
#         return self.final_conv(x)

# # --- Các hàm tiện ích & Huấn luyện ---
# def get_loaders(train_dir, train_maskdir, val_dir, val_maskdir, batch_size, train_transform, val_transform, num_workers, pin_memory):
#     train_ds = LaneDataset(train_dir, train_maskdir, train_transform); val_ds = LaneDataset(val_dir, val_maskdir, val_transform)
#     train_loader = DataLoader(train_ds, batch_size, True, num_workers=num_workers, pin_memory=pin_memory)
#     val_loader = DataLoader(val_ds, batch_size, False, num_workers=num_workers, pin_memory=pin_memory)
#     return train_loader, val_loader
# def check_accuracy(loader, model, device):
#     num_correct, num_pixels, dice_score = 0, 0, 0; model.eval()
#     with torch.no_grad():
#         for x, y in loader:
#             x, y = x.to(device), y.to(device)
#             preds = torch.argmax(torch.softmax(model(x), 1), 1)
#             num_correct += (preds == y).sum(); num_pixels += torch.numel(preds)
#             dice_score += (2*(preds*y).sum())/((preds+y).sum()+1e-8)
#     acc = num_correct/num_pixels*100; dice = dice_score/len(loader)
#     print(f"Accuracy: {acc:.2f} | Dice score: {dice:.4f}"); model.train(); return dice
# def train_fn(loader, model, optimizer, loss_fn, scaler, device):
#     loop = tqdm(loader, desc="Training");
#     for data, targets in loop:
#         data, targets = data.to(device), targets.long().to(device)
#         with torch.autocast(device_type=device):
#             preds = model(data); loss = loss_fn(preds, targets)
#         optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
#         loop.set_postfix(loss=loss.item())

# # --- Khối thực thi chính ---
# train_transform = A.Compose([
#     A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, cv2.INTER_NEAREST),
#     A.Rotate(limit=35, p=0.8, interpolation=cv2.INTER_NEAREST, border_mode=cv2.BORDER_CONSTANT, mask_value=0),
#     A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.1),
#     A.Normalize(mean=[0,0,0], std=[1,1,1], max_pixel_value=255.0), ToTensorV2(),
# ])
# val_transform = A.Compose([
#     A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH, cv2.INTER_NEAREST),
#     A.Normalize(mean=[0,0,0], std=[1,1,1], max_pixel_value=255.0), ToTensorV2(),
# ])
# model = UNET(out_c=len(CLASS_MAP)+1).to(DEVICE)
# loss_fn = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
# if torch.cuda.device_count() > 1: print(f"Sử dụng {torch.cuda.device_count()} GPUs!"); model = nn.DataParallel(model)
# train_loader, val_loader = get_loaders(os.path.join(FINAL_DATASET_DIR, "images/train"), os.path.join(FINAL_DATASET_DIR, "masks/train"), os.path.join(FINAL_DATASET_DIR, "images/val"), os.path.join(FINAL_DATASET_DIR, "masks/val"), BATCH_SIZE, train_transform, val_transform, NUM_WORKERS, PIN_MEMORY)
# scaler = torch.amp.GradScaler(device=DEVICE);

# # Khởi tạo Early Stopping
# early_stopping = EarlyStopping(patience=7, verbose=True, path=CHECKPOINT_PATH)

# for epoch in range(NUM_EPOCHS):
#     print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
#     train_fn(train_loader, model, optimizer, loss_fn, scaler, DEVICE)
#     print("Kiểm tra trên tập validation...")
#     current_dice = check_accuracy(val_loader, model, DEVICE)
    
#     # Gọi Early Stopping
#     early_stopping(current_dice, model)
#     if early_stopping.early_stop:
#         print("Dừng sớm do không cải thiện trên tập validation.")
#         break
        
# print("\n--- HOÀN TẤT HUẤN LUYỆN ---")
# print(f"Model tốt nhất đã được lưu tại: {CHECKPOINT_PATH}")

In [13]:
# # ===================================================================
# # CELL 4: KIỂM TRA TRỰC QUAN TRÊN DỮ LIỆU TEST
# # ===================================================================
# # Nhiệm vụ:
# # 1. Tải lại kiến trúc mô hình.
# # 2. Load trọng số đã được huấn luyện tốt nhất.
# # 3. Chọn ngẫu nhiên 5 ảnh từ tập test.
# # 4. Hiển thị so sánh (Ảnh Gốc - Nhãn Thật - Dự Đoán).
# # ===================================================================

# import torch
# import torch.nn as nn
# import torchvision.transforms.functional as TF
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import glob
# import random
# import numpy as np
# import cv2
# from PIL import Image
# import matplotlib.pyplot as plt

# # --- 1. CẤU HÌNH ---
# # Đảm bảo các biến này khớp với cell huấn luyện
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# IMAGE_HEIGHT = 256
# IMAGE_WIDTH = 512
# NUM_CLASSES = 4 # 3 lớp + 1 nền
# MODEL_PATH = "/kaggle/working/unet_best_model.pth"
# FINAL_DATASET_DIR = '/kaggle/working/final_lane_dataset'
# TEST_IMG_DIR = os.path.join(FINAL_DATASET_DIR, "images/test/")
# TEST_MASK_DIR = os.path.join(FINAL_DATASET_DIR, "masks/test/")

# # --- 2. TẢI LẠI KIẾN TRÚC MÔ HÌNH ---
# # (Bạn phải định nghĩa lại kiến trúc để có thể load trọng số vào)
# class DoubleConv(nn.Module):
#     def __init__(self, in_c, out_c):
#         super().__init__(); self.conv = nn.Sequential(nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
#     def forward(self, x): return self.conv(x)

# class UNET(nn.Module):
#     def __init__(self, in_c=3, out_c=4, features=[64, 128, 256, 512]):
#         super().__init__(); self.ups, self.downs, self.pool = nn.ModuleList(), nn.ModuleList(), nn.MaxPool2d(2, 2)
#         for f in features: self.downs.append(DoubleConv(in_c, f)); in_c = f
#         for f in reversed(features): self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2)); self.ups.append(DoubleConv(f*2, f))
#         self.bottleneck, self.final_conv = DoubleConv(features[-1], features[-1]*2), nn.Conv2d(features[0], out_c, 1)
#     def forward(self, x):
#         skips = [];
#         for down in self.downs: x = down(x); skips.append(x); x = self.pool(x)
#         x = self.bottleneck(x); skips = skips[::-1]
#         for i in range(0, len(self.ups), 2):
#             x = self.ups[i](x); skip = skips[i//2]
#             if x.shape != skip.shape: x = TF.resize(x, size=skip.shape[2:])
#             x = self.ups[i+1](torch.cat((skip, x), 1))
#         return self.final_conv(x)

# # --- 3. LOAD MODEL VÀ TRỌNG SỐ ---
# print(f"--- Tải model từ: {MODEL_PATH} ---")
# model = UNET(out_c=NUM_CLASSES).to(DEVICE)
# model.load_state_dict(torch.load(MODEL_PATH))
# model.eval() # Chuyển model sang chế độ đánh giá

# # --- 4. CHUẨN BỊ DỮ LIỆU TEST VÀ HIỂN THỊ ---
# # Phép biến đổi phải giống hệt như tập validation/test khi huấn luyện
# test_transform = A.Compose([
#     A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#     A.Normalize(mean=[0,0,0], std=[1,1,1], max_pixel_value=255.0),
#     ToTensorV2(),
# ])

# # Lấy 5 file ngẫu nhiên từ tập test
# test_image_files = glob.glob(os.path.join(TEST_IMG_DIR, '*.jpg'))
# random_test_files = random.sample(test_image_files, 10)

# print("\n--- Hiển thị 10 kết quả dự đoán ngẫu nhiên ---")
# fig, axes = plt.subplots(10, 3, figsize=(15, 20))
# fig.tight_layout()

# for i, img_path in enumerate(random_test_files):
#     # --- Chuẩn bị dữ liệu ---
#     # Tải ảnh gốc
#     image = np.array(Image.open(img_path).convert("RGB"))
    
#     # Tải nhãn thật (ground truth)
#     mask_filename = os.path.splitext(os.path.basename(img_path))[0] + ".png"
#     mask_path = os.path.join(TEST_MASK_DIR, mask_filename)
#     gt_mask = np.array(Image.open(mask_path).convert("L"))
    
#     # Áp dụng transform cho ảnh đầu vào
#     input_tensor = test_transform(image=image)["image"].unsqueeze(0).to(DEVICE)
    
#     # --- Dự đoán ---
#     with torch.no_grad():
#         preds_logits = model(input_tensor)
#         preds_mask = torch.argmax(torch.softmax(preds_logits, dim=1), dim=1).squeeze(0).cpu().numpy()

#     # --- Hiển thị ---
#     axes[i, 0].imshow(image)
#     axes[i, 0].set_title(f"Ảnh Gốc\n{os.path.basename(img_path)}")
#     axes[i, 0].axis('off')

#     axes[i, 1].imshow(gt_mask, cmap='jet')
#     axes[i, 1].set_title("Nhãn Thật (Ground Truth)")
#     axes[i, 1].axis('off')

#     axes[i, 2].imshow(preds_mask, cmap='jet')
#     axes[i, 2].set_title("Dự Đoán của Model")
#     axes[i, 2].axis('off')

# plt.show()

In [14]:
# # ===================================================================
# # CELL 1: KIỂM TRA ĐƯỜNG DẪN, GỠ LỖI VÀ TẠO MASK
# # ===================================================================
# import os
# import json
# import cv2
# import numpy as np
# import glob
# import random
# import shutil
# import matplotlib.pyplot as plt
# from tqdm import tqdm

# # --- 1. KIỂM TRA CẤU TRÚC THƯ MỤC ---
# # !!! VUI LÒNG KIỂM TRA VÀ CHỈNH SỬA ĐƯỜNG DẪN NÀY CHO ĐÚNG !!!
# # Đây là đường dẫn mà code sẽ quét tìm các thư mục con (Dataset_seg_lane_1.1, v.v.)
# KAGGLE_INPUT_DIR = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane/Dataset_seg_lane' 

# print("--- Bắt đầu kiểm tra cấu trúc thư mục ---")
# print(f"Code đang tìm kiếm các thư mục con bên trong: '{KAGGLE_INPUT_DIR}'")

# # In ra cấu trúc thư mục thực tế để bạn đối chiếu
# print("\n--- Cấu trúc thư mục thực tế (lệnh ls -R): ---")
# !ls -R {KAGGLE_INPUT_DIR}
# print("--------------------------------------------")

# # Lấy danh sách các thư mục con mà code tìm thấy
# subfolders = []
# if os.path.exists(KAGGLE_INPUT_DIR):
#     subfolders = [f.name for f in os.scandir(KAGGLE_INPUT_DIR) if f.is_dir()]

# print(f"\nCode đã tìm thấy {len(subfolders)} thư mục con để xử lý:")
# print(subfolders)

# if not subfolders:
#     print("\n\nLỖI NGHIÊM TRỌNG: Không tìm thấy thư mục con nào.")
#     print("Vui lòng kiểm tra lại đường dẫn `KAGGLE_INPUT_DIR` ở đầu cell này.")
#     print("Đường dẫn này phải trỏ đến thư mục chứa các thư mục con như 'Dataset_seg_lane_1.1', 'Dataset_seg_lane_2', v.v.")
#     # Dừng chương trình ở đây nếu không tìm thấy gì
#     raise FileNotFoundError("Không tìm thấy thư mục con để xử lý. Vui lòng kiểm tra lại đường dẫn.")

# # --- 2. BẮT ĐẦU XỬ LÝ DỮ LIỆU ---
# TEMP_OUTPUT_DIR = '/kaggle/working/temp_processed_data'
# CLASS_MAP = {'phan_cach_ngoc_chieu': 1, 'phan_lan': 2, 'le_duong': 3}
# if os.path.exists(TEMP_OUTPUT_DIR): shutil.rmtree(TEMP_OUTPUT_DIR)
# TEMP_IMAGE_DIR = os.path.join(TEMP_OUTPUT_DIR, 'original_images')
# TEMP_MASK_DIR = os.path.join(TEMP_OUTPUT_DIR, 'masks')
# os.makedirs(TEMP_IMAGE_DIR, exist_ok=True); os.makedirs(TEMP_MASK_DIR, exist_ok=True)

# print("\n--- Bắt đầu quét và xử lý dữ liệu theo cấu trúc thư mục ---")
# processed_files_count = 0

# for folder_name in tqdm(subfolders, desc="Đang xử lý các thư mục"):
#     image_folder_path = os.path.join(KAGGLE_INPUT_DIR, folder_name)
#     json_path = os.path.join(KAGGLE_INPUT_DIR, f"labels_{folder_name}.json")
#     if not os.path.exists(json_path): continue
#     with open(json_path, 'r') as f: current_annotations = json.load(f)
    
#     for img_path in glob.glob(os.path.join(image_folder_path, '*.jpg')):
#         filename = os.path.basename(img_path)
#         if filename in current_annotations:
#             annotation = current_annotations[filename]
#             image = cv2.imread(img_path)
#             if image is None: continue
#             height, width, _ = image.shape
#             mask = np.zeros((height, width), dtype=np.uint8)
#             if 'regions' in annotation and annotation['regions']:
#                  for region in annotation['regions'].values():
#                     if 'region_attributes' in region and 'label' in region['region_attributes']:
#                         class_name = region['region_attributes']['label']
#                         if class_name in CLASS_MAP:
#                             points = np.array(list(zip(region['shape_attributes']['all_points_x'], region['shape_attributes']['all_points_y'])), dtype=np.int32)
#                             cv2.fillPoly(mask, [points], color=CLASS_MAP[class_name])
            
#             base_filename = os.path.splitext(filename)[0]
#             unique_img_filename = f"{folder_name}_{base_filename}.jpg"
#             unique_mask_filename = f"{folder_name}_{base_filename}.png"

#             cv2.imwrite(os.path.join(TEMP_IMAGE_DIR, unique_img_filename), image)
#             cv2.imwrite(os.path.join(TEMP_MASK_DIR, unique_mask_filename), mask)
#             processed_files_count += 1

# print(f"\n--- Hoàn tất xử lý! Đã tạo {processed_files_count} cặp ảnh/mask ---")

In [15]:
# # ===================================================================
# # CELL 2: TIỀN XỬ LÝ ẢNH, PHÂN CHIA VÀ TẠO DATASET HOÀN CHỈNH
# # >> CHẠY CELL NÀY SAU KHI CELL 1 ĐÃ HOÀN TẤT <<
# # ===================================================================
# # Nhiệm vụ:
# # 1. Sử dụng dữ liệu đã tạo ở Cell 1.
# # 2. Định nghĩa hàm tiền xử lý ảnh (CLAHE + Sharpening).
# # 3. Phân chia danh sách các file thành 3 tập train/val/test.
# # 4. Áp dụng tiền xử lý cho ảnh và sao chép mask vào cấu trúc thư mục cuối cùng.
# # 5. Hiển thị 5 cặp (ảnh đã tiền xử lý, mask) ngẫu nhiên để kiểm tra.
# # ===================================================================

# import os
# import cv2
# import numpy as np
# import shutil
# import glob
# import random
# import matplotlib.pyplot as plt
# from sklearn.model_selection import train_test_split
# from tqdm import tqdm

# # --- 1. CẤU HÌNH VÀ HÀM TIỀN XỬ LÝ ---
# TEMP_OUTPUT_DIR = '/kaggle/working/temp_processed_data'
# FINAL_DATASET_DIR = '/kaggle/working/final_lane_dataset'

# # Đường dẫn từ Cell 1
# TEMP_IMAGE_DIR = os.path.join(TEMP_OUTPUT_DIR, 'original_images')
# TEMP_MASK_DIR = os.path.join(TEMP_OUTPUT_DIR, 'masks')

# def preprocess_image(image):
#     """Áp dụng CLAHE và Sharpening để cải thiện chất lượng ảnh."""
#     lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
#     l_channel, a_channel, b_channel = cv2.split(lab_image)
#     clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
#     clahe_l_channel = clahe.apply(l_channel)
#     merged_lab_image = cv2.merge([clahe_l_channel, a_channel, b_channel])
#     enhanced_image = cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2BGR)
#     sharpen_kernel = np.array([[-1, -1, -1], [-1,  9, -1], [-1, -1, -1]])
#     sharpened_image = cv2.filter2D(enhanced_image, -1, sharpen_kernel)
#     return sharpened_image

# # Xóa thư mục cũ nếu tồn tại
# if os.path.exists(FINAL_DATASET_DIR):
#     shutil.rmtree(FINAL_DATASET_DIR)

# # --- 2. PHÂN CHIA DỮ LIỆU ---
# print("--- Bắt đầu phân chia dữ liệu thành các tập train, val, test ---")
# image_paths = sorted(glob.glob(os.path.join(TEMP_IMAGE_DIR, '*.jpg')))
# mask_paths = sorted(glob.glob(os.path.join(TEMP_MASK_DIR, '*.jpg')))

# # Phân chia lần 1: tách tập test ra (ví dụ 15%)
# train_val_imgs, test_imgs, train_val_masks, test_masks = train_test_split(
#     image_paths, mask_paths, test_size=0.15, random_state=42
# )

# # Phân chia lần 2: tách tập train và val từ phần còn lại (ví dụ 15%/85% ~ 17.6%)
# train_imgs, val_imgs, train_masks, val_masks = train_test_split(
#     train_val_imgs, train_val_masks, test_size=0.176, random_state=42
# )

# print(f"Phân chia hoàn tất:")
# print(f"- Tập Train: {len(train_imgs)} ảnh")
# print(f"- Tập Val  : {len(val_imgs)} ảnh")
# print(f"- Tập Test : {len(test_imgs)} ảnh")

# # Gộp lại thành một cấu trúc để dễ xử lý
# split_data = {
#     'train': (train_imgs, train_masks),
#     'val': (val_imgs, val_masks),
#     'test': (test_imgs, test_masks),
# }

# # --- 3. TIỀN XỬ LÝ VÀ LƯU VÀO CẤU TRÚC THƯ MỤC CUỐI CÙNG ---
# print("\n--- Bắt đầu tiền xử lý ảnh và tạo bộ dữ liệu cuối cùng ---")
# for split_name, (img_list, mask_list) in split_data.items():
    
#     # Tạo thư mục con
#     final_img_dir = os.path.join(FINAL_DATASET_DIR, 'images', split_name)
#     final_mask_dir = os.path.join(FINAL_DATASET_DIR, 'masks', split_name)
#     os.makedirs(final_img_dir, exist_ok=True)
#     os.makedirs(final_mask_dir, exist_ok=True)
    
#     # Xử lý và sao chép file
#     for i in tqdm(range(len(img_list)), desc=f'Xử lý tập {split_name}'):
#         img_path = img_list[i]
#         mask_path = mask_list[i]
        
#         # Đọc ảnh gốc
#         original_image = cv2.imread(img_path)
        
#         # Tiền xử lý ảnh
#         preprocessed_img = preprocess_image(original_image)
        
#         # Lưu ảnh đã xử lý
#         filename = os.path.basename(img_path)
#         cv2.imwrite(os.path.join(final_img_dir, filename), preprocessed_img)
        
#         # Sao chép file mask tương ứng
#         shutil.copy(mask_path, os.path.join(final_mask_dir, filename))

# print(f"\n--- Đã tạo thành công bộ dữ liệu tại: {FINAL_DATASET_DIR} ---")

# # --- 4. KIỂM TRA TRỰC QUAN TRÊN DỮ LIỆU ĐÃ XỬ LÝ ---
# print("\n--- Hiển thị 5 cặp (ảnh ĐÃ TIỀN XỬ LÝ, mask) ngẫu nhiên từ tập train ---")
# train_img_final_paths = glob.glob(os.path.join(FINAL_DATASET_DIR, 'images/train', '*.jpg'))
# train_mask_final_paths = glob.glob(os.path.join(FINAL_DATASET_DIR, 'masks/train', '*.jpg'))

# random_indices = random.sample(range(len(train_img_final_paths)), min(5, len(train_img_final_paths)))

# fig, axes = plt.subplots(len(random_indices), 2, figsize=(10, len(random_indices) * 4))
# if len(random_indices) == 1:
#     axes = [axes]

# for i, idx in enumerate(random_indices):
#     img_path = train_img_final_paths[idx]
#     mask_path = os.path.join(FINAL_DATASET_DIR, 'masks/train', os.path.basename(img_path))
    
#     img = cv2.imread(img_path)
#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
#     mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
#     axes[i][0].imshow(img)
#     axes[i][0].set_title(f"Ảnh đã xử lý: {os.path.basename(img_path)}")
#     axes[i][0].axis('off')
    
#     axes[i][1].imshow(mask, cmap='jet')
#     axes[i][1].set_title(f"Mask tương ứng")
#     axes[i][1].axis('off')

# plt.tight_layout()
# plt.show()

In [16]:
# import shutil
# import os

# # Tên file zip bạn muốn tạo
# output_filename = '/kaggle/working/image_labels_dataset'

# # Thư mục nguồn bạn muốn nén
# source_dir = '/kaggle/working/temp_processed_data'

# # Kiểm tra xem thư mục nguồn có tồn tại không
# if os.path.exists(source_dir):
#     print(f"Bắt đầu nén thư mục '{source_dir}'...")
#     # Thực hiện nén
#     shutil.make_archive(output_filename, 'zip', source_dir)
#     print(f"Đã nén thành công! File được lưu tại: {output_filename}.zip")
# else:
#     print(f"LỖI: Không tìm thấy thư mục nguồn '{source_dir}' để nén.")

In [17]:
# # ===================================================================
# # CELL A: SAO CHÉP DỮ LIỆU SANG THƯ MỤC LÀM VIỆC VÀ LÀM SẠCH
# # ===================================================================
# # Nhiệm vụ:
# # 1. Sao chép bộ dữ liệu từ /kaggle/input (chỉ đọc) sang /kaggle/working (có thể ghi).
# # 2. Quét và xóa các file lỗi trên bản sao này.
# # ===================================================================

# import os
# import shutil
# import cv2
# import numpy as np
# import glob
# from tqdm import tqdm

# INPUT_DATA_DIR = '/kaggle/input/dataset-seg-lane/final_dataset' # Dữ liệu gốc
# WORKING_DATA_DIR = '/kaggle/working/final_lane_dataset' # Nơi làm việc, có thể ghi/xóa

# # --- 1. SAO CHÉP DỮ LIỆU ---
# print(f"--- Bắt đầu sao chép dữ liệu từ '{INPUT_DATA_DIR}' sang '{WORKING_DATA_DIR}' ---")

# # Xóa thư mục cũ nếu tồn tại để tránh lỗi
# if os.path.exists(WORKING_DATA_DIR):
#     shutil.rmtree(WORKING_DATA_DIR)

# shutil.copytree(INPUT_DATA_DIR, WORKING_DATA_DIR)
# print("--- Sao chép hoàn tất! ---")


# # --- 2. LÀM SẠCH DỮ LIỆU TRÊN BẢN SAO ---
# print("\n--- Bắt đầu quá trình quét và lọc dữ liệu lỗi ---")

# bad_files = set()
# mask_dirs_to_check = glob.glob(os.path.join(WORKING_DATA_DIR, "masks", "*"))

# for mask_dir in mask_dirs_to_check:
#     print(f"\nĐang quét thư mục: {mask_dir}")
#     if not os.path.isdir(mask_dir): continue

#     for mask_path in tqdm(glob.glob(os.path.join(mask_dir, '*.jpg'))):
#         mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
#         if mask is not None and mask.max() >= 4:
#             bad_files.add(os.path.basename(mask_path))

# print(f"\n\nPhát hiện tổng cộng {len(bad_files)} file mask có giá trị không hợp lệ.")

# if bad_files:
#     print("--- Bắt đầu xóa các file bị lỗi ---")
    
#     dirs_to_clean = glob.glob(os.path.join(WORKING_DATA_DIR, "*", "*"))
#     files_deleted_count = 0
    
#     for filename in tqdm(list(bad_files), desc="Đang xóa file"):
#         for directory in dirs_to_clean:
#              if not os.path.isdir(directory): continue
#              file_to_delete = os.path.join(directory, filename)
#              if os.path.exists(file_to_delete):
#                  os.remove(file_to_delete)
#                  files_deleted_count += 1
    
#     print(f"\n--- Hoàn tất! Đã xóa {files_deleted_count // 2} cặp ảnh/mask bị lỗi. ---")
# else:
#     print("--- Không tìm thấy file lỗi nào. Bộ dữ liệu của bạn đã sạch! ---")

# print("\nBộ dữ liệu đã sẵn sàng để huấn luyện.")

In [18]:
# # ===================================================================
# # CELL B: HUẤN LUYỆN U-NET TRÊN BỘ DỮ LIỆU ĐÃ LÀM SẠCH
# # ===================================================================

# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF
# from tqdm import tqdm
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import numpy as np
# import cv2
# from PIL import Image
# import glob

# # --- 1. CẤU HÌNH VÀ SIÊU THAM SỐ ---
# LEARNING_RATE = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 16
# NUM_EPOCHS = 25
# NUM_WORKERS = 0 # GIỮ NGUYÊN BẰNG 0
# IMAGE_HEIGHT = 256
# IMAGE_WIDTH = 512
# PIN_MEMORY = True
# # !!! QUAN TRỌNG: Trỏ đến thư mục đã được làm sạch trong /kaggle/working/
# DATA_DIR = '/kaggle/working/final_lane_dataset' 
# TRAIN_IMG_DIR = os.path.join(DATA_DIR, "images/train/")
# TRAIN_MASK_DIR = os.path.join(DATA_DIR, "masks/train/")
# VAL_IMG_DIR = os.path.join(DATA_DIR, "images/val/")
# VAL_MASK_DIR = os.path.join(DATA_DIR, "masks/val/")
# PREDICTION_OUTPUT_DIR = "/kaggle/working/saved_images/"
# CHECKPOINT_PATH = "/kaggle/working/unet_checkpoint.pth"

# # --- 2. LỚP DATASET (Không thay đổi) ---
# class LaneDataset(Dataset):
#     def __init__(self, image_dir, mask_dir, transform=None):
#         self.image_dir = image_dir
#         self.mask_dir = mask_dir
#         self.transform = transform
#         self.images = os.listdir(image_dir)
#     def __len__(self): return len(self.images)
#     def __getitem__(self, index):
#         img_path = os.path.join(self.image_dir, self.images[index])
#         mask_path = os.path.join(self.mask_dir, self.images[index])
#         image = np.array(Image.open(img_path).convert("RGB"))
#         mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
#         if self.transform is not None:
#             augmentations = self.transform(image=image, mask=mask)
#             image = augmentations["image"]
#             mask = augmentations["mask"]
#         return image, mask

# # --- 3. KIẾN TRÚC MÔ HÌNH U-NET (Không thay đổi) ---
# class DoubleConv(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super(DoubleConv, self).__init__()
#         self.conv = nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
#     def forward(self, x): return self.conv(x)
# class UNET(nn.Module):
#     def __init__(self, in_channels=3, out_channels=4, features=[64, 128, 256, 512]):
#         super(UNET, self).__init__()
#         self.ups, self.downs, self.pool = nn.ModuleList(), nn.ModuleList(), nn.MaxPool2d(kernel_size=2, stride=2)
#         for feature in features: self.downs.append(DoubleConv(in_channels, feature)); in_channels = feature
#         for feature in reversed(features): self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)); self.ups.append(DoubleConv(feature*2, feature))
#         self.bottleneck = DoubleConv(features[-1], features[-1]*2); self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
#     def forward(self, x):
#         skip_connections = []
#         for down in self.downs: x = down(x); skip_connections.append(x); x = self.pool(x)
#         x = self.bottleneck(x); skip_connections = skip_connections[::-1]
#         for idx in range(0, len(self.ups), 2):
#             x = self.ups[idx](x); skip_connection = skip_connections[idx//2]
#             if x.shape != skip_connection.shape: x = TF.resize(x, size=skip_connection.shape[2:])
#             x = self.ups[idx+1](torch.cat((skip_connection, x), dim=1))
#         return self.final_conv(x)

# # --- 4. CÁC HÀM TIỆN ÍCH (Không thay đổi) ---
# def get_loaders(train_dir, train_maskdir, val_dir, val_maskdir, batch_size, train_transform, val_transform, num_workers, pin_memory):
#     train_ds = LaneDataset(image_dir=train_dir, mask_dir=train_maskdir, transform=train_transform)
#     train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory, shuffle=True)
#     val_ds = LaneDataset(image_dir=val_dir, mask_dir=val_maskdir, transform=val_transform)
#     val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory, shuffle=False)
#     return train_loader, val_loader
# def check_accuracy(loader, model, device="cuda"):
#     num_correct, num_pixels, dice_score = 0, 0, 0
#     model.eval()
#     with torch.no_grad():
#         for x, y in loader:
#             x, y = x.to(device), y.to(device)
#             preds = torch.softmax(model(x), dim=1); preds = torch.argmax(preds, dim=1)
#             num_correct += (preds == y).sum(); num_pixels += torch.numel(preds)
#             dice_score += (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)
#     print(f"Got {num_correct}/{num_pixels} with acc {num_correct/num_pixels*100:.2f}")
#     print(f"Dice score: {dice_score/len(loader):.4f}"); model.train()
#     return dice_score/len(loader)
# def save_predictions_as_imgs(loader, model, folder, device="cuda"):
#     if not os.path.exists(folder): os.makedirs(folder)
#     model.eval()
#     for idx, (x, y) in enumerate(loader):
#         x = x.to(device=device)
#         with torch.no_grad():
#             preds = torch.softmax(model(x), dim=1); preds = torch.argmax(preds, dim=1).unsqueeze(1)
#         for i in range(min(3, x.shape[0])):
#             TF.to_pil_image(preds[i].float()/3.0).save(f"{folder}/pred_{idx*BATCH_SIZE + i}.png")
#             TF.to_pil_image(y[i].float()/3.0).save(f"{folder}/gt_{idx*BATCH_SIZE + i}.png")
#         break
#     model.train()

# # --- 5. HÀM HUẤN LUYỆN (Không thay đổi) ---
# def train_fn(loader, model, optimizer, loss_fn, scaler):
#     loop = tqdm(loader, desc="Training")
#     for batch_idx, (data, targets) in enumerate(loop):
#         data = data.to(device=DEVICE); targets = targets.long().to(device=DEVICE)
#         with torch.autocast(device_type=DEVICE):
#             predictions = model(data); loss = loss_fn(predictions, targets)
#         optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
#         loop.set_postfix(loss=loss.item())

# # ===================================================================
# # KHỐI THỰC THI CHÍNH
# # ===================================================================
# train_transform = A.Compose([
#     A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#     A.Rotate(limit=35, p=0.8, interpolation=cv2.INTER_NEAREST, border_mode=cv2.BORDER_CONSTANT, value=0),
#     A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.1),
#     A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0),
#     ToTensorV2(),
# ])
# val_transform = A.Compose([
#     A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#     A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0),
#     ToTensorV2(),
# ])

# model = UNET(in_channels=3, out_channels=4)
# loss_fn = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# if torch.cuda.device_count() > 1:
#     print(f"Let's use {torch.cuda.device_count()} GPUs!")
#     model = nn.DataParallel(model)
# model.to(DEVICE)

# train_loader, val_loader = get_loaders(TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR, BATCH_SIZE, train_transform, val_transform, NUM_WORKERS, PIN_MEMORY)

# scaler = torch.amp.GradScaler(device=DEVICE)
# best_dice_score = -1.0

# for epoch in range(NUM_EPOCHS):
#     print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
#     train_fn(train_loader, model, optimizer, loss_fn, scaler)
#     print("Checking accuracy on validation set...")
#     current_dice_score = check_accuracy(val_loader, model, device=DEVICE)
#     if current_dice_score > best_dice_score:
#         best_dice_score = current_dice_score
#         torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), CHECKPOINT_PATH)
#         print(f"==> New best model saved to {CHECKPOINT_PATH} with Dice Score: {best_dice_score:.4f}")
#     print("Saving some prediction examples...")
#     epoch_save_folder = os.path.join(PREDICTION_OUTPUT_DIR, f"epoch_{epoch+1}")
#     save_predictions_as_imgs(val_loader, model, folder=epoch_save_folder, device=DEVICE)

# print("\n--- HOÀN TẤT HUẤN LUYỆN ---")

In [19]:
# # ===================================================================
# # CELL DUY NHẤT: TỪ DỮ LIỆU THÔ ĐẾN HUẤN LUYỆN MODEL (ĐÃ LỌC FILE LỖI)
# # ===================================================================

# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF
# from tqdm import tqdm
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import os
# import numpy as np
# import cv2
# from PIL import Image
# import glob
# import json
# import shutil
# from sklearn.model_selection import train_test_split
# import collections

# print("--- BẮT ĐẦU TOÀN BỘ QUY TRÌNH ---")

# # ===================================================================
# # PHẦN 1: TẠO MASK TỪ DỮ LIỆU GỐC
# # ===================================================================
# print("\n[PHẦN 1] Đang đọc file JSON và tạo ảnh mask...")

# # --- Cấu hình ---
# KAGGLE_INPUT_DIR = '/kaggle/input/dataset-seg-lane/Dataset_seg_lane'
# TEMP_OUTPUT_DIR = '/kaggle/working/temp_processed_data'

# # =======================================================
# # SỬA LỖI TYPO LẦN CUỐI TẠI ĐÂY: "chiu" -> "chieu"
# # =======================================================
# CLASS_MAP = {'phan_cach_nguoc_chieu': 1, 'phan_lan': 2, 'le_duong': 3}

# if os.path.exists(TEMP_OUTPUT_DIR): shutil.rmtree(TEMP_OUTPUT_DIR)
# TEMP_IMAGE_DIR = os.path.join(TEMP_OUTPUT_DIR, 'original_images')
# TEMP_MASK_DIR = os.path.join(TEMP_OUTPUT_DIR, 'masks')
# os.makedirs(TEMP_IMAGE_DIR, exist_ok=True)
# os.makedirs(TEMP_MASK_DIR, exist_ok=True)

# # --- Quét và xử lý ---
# subfolders = [f.name for f in os.scandir(KAGGLE_INPUT_DIR) if f.is_dir()]
# all_processed_files = 0
# unmapped_labels = collections.defaultdict(int)

# for folder_name in tqdm(subfolders, desc="  Đang xử lý thư mục"):
#     image_folder_path = os.path.join(KAGGLE_INPUT_DIR, folder_name)
#     json_path = os.path.join(KAGGLE_INPUT_DIR, f"labels_{folder_name}.json")
#     if not os.path.exists(json_path): continue
#     with open(json_path, 'r') as f: current_annotations = json.load(f)
#     image_files_in_folder = glob.glob(os.path.join(image_folder_path, '*.jpg'))
#     for img_path in image_files_in_folder:
#         filename = os.path.basename(img_path)
#         if filename in current_annotations:
#             annotation = current_annotations[filename]
#             image = cv2.imread(img_path)
#             if image is None: continue
#             height, width, _ = image.shape
#             mask = np.zeros((height, width), dtype=np.uint8)
#             has_valid_region = False
#             if 'regions' in annotation and annotation['regions']:
#                 for region in annotation['regions'].values():
#                     if 'region_attributes' in region and region['region_attributes'] and 'label' in region['region_attributes']:
#                         class_name = region['region_attributes']['label']
#                         if class_name in CLASS_MAP:
#                             has_valid_region = True
#                             pixel_value = CLASS_MAP[class_name]
#                             points = np.array(list(zip(region['shape_attributes']['all_points_x'], region['shape_attributes']['all_points_y'])), dtype=np.int32)
#                             cv2.fillPoly(mask, [points], color=pixel_value)
#                         else:
#                             # Thêm gỡ lỗi: Ghi nhận các nhãn không khớp
#                             unmapped_labels[class_name] += 1
            
#             # Chỉ lưu file nếu có ít nhất một vùng hợp lệ được vẽ
#             if has_valid_region:
#                 cv2.imwrite(os.path.join(TEMP_IMAGE_DIR, filename), image)
#                 cv2.imwrite(os.path.join(TEMP_MASK_DIR, filename), mask)
#                 all_processed_files += 1

# print(f"--- [PHẦN 1] Hoàn tất. Đã tạo {all_processed_files} cặp ảnh/mask. ---")
# if unmapped_labels:
#     print("\nCẢNH BÁO: Đã tìm thấy các nhãn trong file JSON không có trong CLASS_MAP:")
#     for label, count in unmapped_labels.items():
#         print(f"  - '{label}' (xuất hiện {count} lần)")

# # ===================================================================
# # PHẦN 2: TIỀN XỬ LÝ, LỌC FILE LỖI VÀ CHIA DATASET
# # ===================================================================
# print("\n[PHẦN 2] Đang tiền xử lý, lọc file lỗi và chia dữ liệu...")

# # --- DANH SÁCH CÁC FILE GÂY LỖI CẦN LOẠI BỎ (nếu có) ---
# BLOCKLIST = set() # Tạm thời để trống, vì lỗi gốc đã được sửa

# # --- Cấu hình ---
# FINAL_DATASET_DIR = '/kaggle/working/final_lane_dataset'
# if os.path.exists(FINAL_DATASET_DIR): shutil.rmtree(FINAL_DATASET_DIR)

# # --- Lọc file lỗi ---
# all_image_paths = sorted(glob.glob(os.path.join(TEMP_IMAGE_DIR, '*.jpg')))
# print(f"  Số lượng ảnh ban đầu: {len(all_image_paths)}")
# image_paths = [p for p in all_image_paths if os.path.basename(p) not in BLOCKLIST]
# mask_paths = [os.path.join(TEMP_MASK_DIR, os.path.basename(p)) for p in image_paths]
# print(f"  Số lượng ảnh sau khi lọc: {len(image_paths)}")

# # --- Chia dữ liệu ---
# train_val_imgs, test_imgs, train_val_masks, test_masks = train_test_split(image_paths, mask_paths, test_size=0.15, random_state=42)
# train_imgs, val_imgs, train_masks, val_masks = train_test_split(train_val_imgs, train_val_masks, test_size=0.176, random_state=42)
# split_data = {'train': (train_imgs, train_masks), 'val': (val_imgs, val_masks), 'test': (test_imgs, test_masks)}

# # --- Tiền xử lý và lưu ---
# def preprocess_image(image):
#     lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
#     l, a, b = cv2.split(lab)
#     clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
#     cl = clahe.apply(l)
#     limg = cv2.merge((cl, a, b))
#     final = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
#     kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
#     return cv2.filter2D(final, -1, kernel)

# for split_name, (img_list, mask_list) in split_data.items():
#     final_img_dir = os.path.join(FINAL_DATASET_DIR, 'images', split_name)
#     final_mask_dir = os.path.join(FINAL_DATASET_DIR, 'masks', split_name)
#     os.makedirs(final_img_dir, exist_ok=True); os.makedirs(final_mask_dir, exist_ok=True)
#     for i in tqdm(range(len(img_list)), desc=f"  Xử lý tập {split_name}"):
#         preprocessed_img = preprocess_image(cv2.imread(img_list[i]))
#         cv2.imwrite(os.path.join(final_img_dir, os.path.basename(img_list[i])), preprocessed_img)
#         shutil.copy(mask_list[i], os.path.join(final_mask_dir, os.path.basename(mask_list[i])))
# print("--- [PHẦN 2] Hoàn tất. ---")

# # ===================================================================
# # PHẦN 3: HUẤN LUYỆN MODEL
# # ===================================================================
# print("\n[PHẦN 3] Bắt đầu huấn luyện mô hình U-Net...")

# # --- Cấu hình ---
# LEARNING_RATE = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BATCH_SIZE = 16
# NUM_EPOCHS = 25
# NUM_WORKERS = 0
# IMAGE_HEIGHT, IMAGE_WIDTH = 256, 512
# PIN_MEMORY = True
# TRAIN_IMG_DIR = os.path.join(FINAL_DATASET_DIR, "images/train/")
# TRAIN_MASK_DIR = os.path.join(FINAL_DATASET_DIR, "masks/train/")
# VAL_IMG_DIR = os.path.join(FINAL_DATASET_DIR, "images/val/")
# VAL_MASK_DIR = os.path.join(FINAL_DATASET_DIR, "masks/val/")
# PREDICTION_OUTPUT_DIR = "/kaggle/working/saved_images/"
# CHECKPOINT_PATH = "/kaggle/working/unet_checkpoint.pth"

# # --- Dataset, Model, và các hàm tiện ích ---
# class LaneDataset(Dataset):
#     def __init__(self, image_dir, mask_dir, transform=None):
#         self.image_dir, self.mask_dir, self.transform = image_dir, mask_dir, transform
#         self.images = os.listdir(image_dir)
#     def __len__(self): return len(self.images)
#     def __getitem__(self, index):
#         img_path = os.path.join(self.image_dir, self.images[index])
#         mask_path = os.path.join(self.mask_dir, self.images[index])
#         image = np.array(Image.open(img_path).convert("RGB"))
#         mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
#         if self.transform is not None:
#             augmentations = self.transform(image=image, mask=mask)
#             image, mask = augmentations["image"], augmentations["mask"]
#         return image, mask
# class DoubleConv(nn.Module):
#     def __init__(self, in_c, out_c):
#         super(DoubleConv, self).__init__()
#         self.conv = nn.Sequential(nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
#     def forward(self, x): return self.conv(x)
# class UNET(nn.Module):
#     def __init__(self, in_c=3, out_c=4, features=[64, 128, 256, 512]):
#         super(UNET, self).__init__()
#         self.ups, self.downs, self.pool = nn.ModuleList(), nn.ModuleList(), nn.MaxPool2d(kernel_size=2, stride=2)
#         for feature in features: self.downs.append(DoubleConv(in_c, feature)); in_c = feature
#         for feature in reversed(features): self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)); self.ups.append(DoubleConv(feature*2, feature))
#         self.bottleneck = DoubleConv(features[-1], features[-1]*2); self.final_conv = nn.Conv2d(features[0], out_c, kernel_size=1)
#     def forward(self, x):
#         skips = []
#         for down in self.downs: x = down(x); skips.append(x); x = self.pool(x)
#         x = self.bottleneck(x); skips = skips[::-1]
#         for idx in range(0, len(self.ups), 2):
#             x = self.ups[idx](x); skip = skips[idx//2]
#             if x.shape != skip.shape: x = TF.resize(x, size=skip.shape[2:])
#             x = self.ups[idx+1](torch.cat((skip, x), dim=1))
#         return self.final_conv(x)
# def get_loaders(train_dir, train_maskdir, val_dir, val_maskdir, batch_size, train_transform, val_transform, num_workers, pin_memory):
#     train_ds = LaneDataset(image_dir=train_dir, mask_dir=train_maskdir, transform=train_transform)
#     train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory, shuffle=True)
#     val_ds = LaneDataset(image_dir=val_dir, mask_dir=val_maskdir, transform=val_transform)
#     val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory, shuffle=False)
#     return train_loader, val_loader
# def check_accuracy(loader, model, device="cuda"):
#     num_correct, num_pixels, dice_score = 0, 0, 0
#     model.eval()
#     with torch.no_grad():
#         for x, y in loader:
#             x, y = x.to(device), y.to(device)
#             preds = torch.softmax(model(x), dim=1); preds = torch.argmax(preds, dim=1)
#             num_correct += (preds == y).sum(); num_pixels += torch.numel(preds)
#             dice_score += (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)
#     print(f"  Got {num_correct}/{num_pixels} with acc {num_correct/num_pixels*100:.2f}")
#     print(f"  Dice score: {dice_score/len(loader):.4f}"); model.train()
#     return dice_score/len(loader)
# def save_predictions(loader, model, folder, device="cuda"):
#     if not os.path.exists(folder): os.makedirs(folder)
#     model.eval()
#     for idx, (x, y) in enumerate(loader):
#         x = x.to(device=device)
#         with torch.no_grad():
#             preds = torch.softmax(model(x), dim=1); preds = torch.argmax(preds, dim=1).unsqueeze(1)
#         for i in range(min(3, x.shape[0])):
#             TF.to_pil_image(preds[i].float()/3.0).save(f"{folder}/pred_{idx*BATCH_SIZE + i}.png")
#             TF.to_pil_image(y[i].float()/3.0).save(f"{folder}/gt_{idx*BATCH_SIZE + i}.png")
#         break
#     model.train()
# def train_fn(loader, model, optimizer, loss_fn, scaler):
#     loop = tqdm(loader, desc="  Training")
#     for batch_idx, (data, targets) in enumerate(loop):
#         data = data.to(device=DEVICE); targets = targets.long().to(device=DEVICE)
#         with torch.autocast(device_type=DEVICE):
#             predictions = model(data); loss = loss_fn(predictions, targets)
#         optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
#         loop.set_postfix(loss=loss.item())

# # --- Augmentation và vòng lặp huấn luyện ---
# train_transform = A.Compose([
#     A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#     A.Rotate(limit=35, p=0.8, interpolation=cv2.INTER_NEAREST, border_mode=cv2.BORDER_CONSTANT),
#     A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.1),
#     A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0),
#     ToTensorV2(),
# ])
# val_transform = A.Compose([
#     A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH, interpolation=cv2.INTER_NEAREST),
#     A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0),
#     ToTensorV2(),
# ])

# model = UNET(in_channels=3, out_channels=4)
# loss_fn = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
# if torch.cuda.device_count() > 1:
#     print(f"Let's use {torch.cuda.device_count()} GPUs!")
#     model = nn.DataParallel(model)
# model.to(DEVICE)
# train_loader, val_loader = get_loaders(TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR, BATCH_SIZE, train_transform, val_transform, NUM_WORKERS, PIN_MEMORY)
# scaler = torch.amp.GradScaler(device=DEVICE)
# best_dice_score = -1.0

# for epoch in range(NUM_EPOCHS):
#     print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
#     train_fn(train_loader, model, optimizer, loss_fn, scaler)
#     print("  Checking accuracy on validation set...")
#     current_dice_score = check_accuracy(val_loader, model, device=DEVICE)
#     if current_dice_score > best_dice_score:
#         best_dice_score = current_dice_score
#         torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), CHECKPOINT_PATH)
#         print(f"  ==> New best model saved to {CHECKPOINT_PATH} with Dice Score: {best_dice_score:.4f}")
#     print("  Saving some prediction examples...")
#     save_predictions(val_loader, model, folder=os.path.join(PREDICTION_OUTPUT_DIR, f"epoch_{epoch+1}"), device=DEVICE)

# print("\n--- [PHẦN 3] HOÀN TẤT HUẤN LUYỆN ---")